In [ ]:
###this code generates figure 3: Wind and wave forcing for one-dimensional cases. 
#Wind stress magnitude $|\boldsymbol{\tau}|$(left), 
#magnitude of surface-layer averaged Stokes drift $<|\boldsymbol{u}^S|>_{SL}$(center), 
#and turbulent Langmuir number $La_t$ (right) for a idealized TC moving 
#at 5 $\rm ms^{-1}$ and 10 $\rm ms^{-1}$. $(x_c,y_c)$ is the TC center position, 
#$RMW$ refers to radius of maximum wind. The horizontal and vertical axes 
#are the horizontal and vertical distance to the TC center, normalized 
#by the radius of maximum wind (50 km). The black dashed line indicates 1 $RMW$ 
#and the black contour line indicates isolines. Symbols mark the LES sampling locations: 
#black open circles denote right-of-track points, white open circles indicate left-of-track points, 
#and the black open prismatic is the center $y_t=0$.
#

#!/usr/bin/env python3
import os
import numpy as np
import scipy.io as sio
import xarray as xr
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

# ============================================================
# COMBINED FIGURE: 2 rows × 3 cols  (each panel has its OWN colorbar)
# Row 0: 5 m/s  (tau, LES <|Us|>_SL, MOM6 La_t)
# Row 1: 10 m/s (tau, LES <|Us|>_SL, MOM6 La_t)
#
# Requirements:
# - Alignment: fixed GridSpec with [ax,cbar] repeated across 3 columns, for BOTH rows
# - Colorbar ranges: do NOT change (each panel uses its own fixed clim/levels)
# - Labels: Row tags on each row; Column tags ONLY on first row
#
# Layout fixes (RIGHT-side ticks, no overlap):
# - Make cbar axes wider (CBAR_W) and keep moderate wspace
# - Keep all colorbar ticks on the RIGHT (including 2nd column)
# - Raise cbar zorder so it never gets covered
# - (Optional) slightly widen ONLY the 2nd-column cbar axes
#
# Text fixes:
# - Replace vector arrows with bold vector notation for tau and Us in titles
# ============================================================

# =========================
# 0) COMMON SETTINGS
# =========================
RMW  = 50e3
xlim = (-4.0, 4.0)
ylim = (-4.0, 4.0)

rho_a = 1.25
p_n   = 101200
p_c   = 96800
Umax  = 65
Rmax  = 50e3
transdir = np.pi
fcor  = 5.5659e-5

# Y list (same for both rows)
Y_KM_ALL      = [300, 200, 150, 130, 110, 90, 70, 50, 30, 10, 0, -20, -40, -60, -80, -100, -120, -140, -200, -300]
Y_FOLDER_ALL  = ["300","200","150","130","110","90","70","50","30","10","0","S20","S40","S60","S80","S100","S120","S140","S200","S300"]
LES_LT_CASES  = ["021","031","036","038","040","042","044","046","048","050","051","053","055","057","059","061","063","065","071","081"]
LES_NLT_CASES = ["021","031","036","038","040","042","044","046","048","050","051","053","055","057","059","061","063","065","071","081"]

LES_MODE_FOR_STOKES = "LT"   # "LT" or "ST"
X_ref = np.linspace(xlim[0], xlim[1], 320)

# offline tau grid
nx_off, ny_off = 401, 401
Xn_off = np.linspace(xlim[0], xlim[1], nx_off)
Yn_off = np.linspace(ylim[0], ylim[1], ny_off)

# FIXED color ranges (do not change)
TAU_LEVELS = np.linspace(0.0, 10.0, 21)   # 0..10
USL_CLIM   = (0.0, 0.3)
LAT_CLIM   = (0.0, 0.8)

# tags
ROW_TAGS = ["(a)", "(b)"]
COL_TAGS = ["(i)", "(ii)", "(iii)"]

# output
FIGSIZE = (13.2, 9.2)
DPI = 250
SAVE_EPS = "/archive/Qian.Xiao/Qian.Xiao/eps/forcingSCM"

# =========================
# 1) HELPERS (your style)
# =========================
def cmap_turbo():
    return plt.get_cmap("turbo") if "turbo" in plt.colormaps() else plt.get_cmap("jet")

def draw_rmw_circle(ax, r=1.0):
    th = np.linspace(0, 2*np.pi, 361)
    ax.plot(r*np.cos(th), r*np.sin(th), "k--", lw=1.2, zorder=10)

def interp_to_X(X_src, v_src, X_tgt):
    f = interp1d(X_src, v_src, kind="linear", bounds_error=False, fill_value=np.nan)
    return f(X_tgt)

def isel_point(ds):
    idx = {d: 0 for d in ["xh","yh","xq","yq"] if d in ds.dims}
    return ds.isel(**idx) if idx else ds

def Hbl_from_max_Tgrad(theta, z):
    dTdz = np.gradient(theta, z, axis=1)
    k = np.nanargmax(np.abs(dTdz), axis=1)
    return z[k]

def surface_layer_avg(U_mag, z, Hbl, frac=0.2):
    ntime, nz = U_mag.shape
    out = np.full(ntime, np.nan)
    for t in range(ntime):
        H = Hbl[t]
        if (not np.isfinite(H)) or H <= 0:
            continue
        zmax = frac * H
        m = (z >= 0) & (z <= zmax)
        if np.count_nonzero(m) < 2:
            continue
        zz = z[m]
        out[t] = np.trapz(U_mag[t, m], zz) / (zz[-1] - zz[0])
    return out

def mark_locations_right_edge(ax, y_norm, x_edge=3.92):
    y_norm = np.asarray(y_norm)
    pos = y_norm > 0
    neg = y_norm < 0
    zer = y_norm == 0

    ax.scatter(np.full(pos.sum(), x_edge), y_norm[pos],
               s=65, marker="o", facecolors="white", edgecolors="k",
               linewidths=1.3, zorder=30)
    ax.scatter(np.full(neg.sum(), x_edge), y_norm[neg],
               s=65, marker="o", facecolors="none", edgecolors="white",
               linewidths=1.8, zorder=30)
    ax.scatter(np.full(zer.sum(), x_edge), y_norm[zer],
               s=70, marker="D", facecolors="white", edgecolors="k",
               linewidths=1.3, zorder=30)

# ---- offline tau model ----
def Cd_model_vec(U10):
    U10 = np.asarray(U10)
    return np.where(
        U10 < 11.0,
        1.2e-3,
        np.where(U10 <= 20.0, (0.49 + 0.065*U10)*1e-3, 0.0018)
    )

def wind_model_xy(xx, yy,
                  rho_a=1.25, p_n=101200, p_c=96800,
                  Umax=65, Rmax=50e3, transpeed=5.0, transdir=np.pi,
                  f=5.5659e-5):
    dp = p_n - p_c
    C  = Umax / np.sqrt(dp)
    B  = C**2 * rho_a * np.exp(1.0)

    # keep historical km-bug exactly
    A  = (Rmax/1000.0)**B

    r  = np.sqrt(xx**2 + yy**2)
    rB = (r/1000.0)**B

    U10 = np.sqrt(A*B*dp*np.exp(-A/rB)/(rho_a*rB) + 0.25*(r*f)**2) - 0.5*r*f
    Adir = np.arctan2(yy, xx)

    # BR Wind angle model
    RSTR = r / Rmax
    A0 = -0.9*RSTR + (-0.09*Umax) + (-14.33)
    A1 = -A0 * (0.04*RSTR + 0.05*transpeed + 0.14)
    P1 = (6.88*RSTR + (-9.60*transpeed) + 85.31) * np.pi/180.0

    Alph = A0 + A1*np.cos(np.pi - transdir - Adir - P1)
    Alph = Alph*np.pi/180.0

    u10x = U10*np.sin(Adir - np.pi - Alph)
    u10y = U10*np.cos(Adir - Alph)
    return u10x, u10y

# =========================
# 2) BUILD ONE ROW (tau, USL, LAT)
# =========================
def build_row(VTRAN, X0, transpeed, les_lt_root, les_st_root, run_template):
    # X(t)
    def x_from_t_days(t_days):
        return -(X0 - (t_days * 86400.0 * VTRAN)) / RMW

    # offline tau
    X2_off, Y2_off = np.meshgrid(Xn_off, Yn_off)
    xx_off = X2_off * RMW
    yy_off = Y2_off * RMW
    u10x_off, u10y_off = wind_model_xy(
        xx_off, yy_off,
        rho_a=rho_a, p_n=p_n, p_c=p_c, Umax=Umax, Rmax=Rmax,
        transpeed=transpeed, transdir=transdir, f=fcor
    )
    W_off   = np.hypot(u10x_off, u10y_off)
    Cd_off  = Cd_model_vec(W_off)
    TAU_off = rho_a * Cd_off * W_off**2

    # LES root / cases
    if LES_MODE_FOR_STOKES.upper() == "LT":
        les_root = les_lt_root
        les_cases = LES_LT_CASES
    else:
        les_root = les_st_root
        les_cases = LES_NLT_CASES

    # Y
    Y_raw = np.array(Y_KM_ALL, dtype=float) * 1e3 / RMW

    # LES USL
    usl_rows = []
    for y_tag, case_id in zip(Y_FOLDER_ALL, les_cases):
        fmat = os.path.join(les_root, case_id, "LESout.mat")
        if not os.path.exists(fmat):
            raise FileNotFoundError(f"Missing LES file: {fmat}")

        mat = sio.loadmat(fmat)

        t_days = np.asarray(mat["t"]).squeeze() / 86400.0
        X_les  = x_from_t_days(t_days)

        z = np.asarray(mat["z"]).T.squeeze()

        # temp field
        if "T" in mat:
            T = np.asarray(mat["T"]).astype(float) - 273.15
        elif "temp" in mat:
            T = np.asarray(mat["temp"]).astype(float)
        else:
            raise KeyError(f"No T/temp in {fmat}. Keys: {list(mat.keys())}")

        # ensure (ntime,nz)
        if T.shape[0] != t_days.size and T.shape[1] == t_days.size:
            T = T.T

        Hbl = Hbl_from_max_Tgrad(T, z)

        Us = np.asarray(mat["Us"]).astype(float)
        Vs = np.asarray(mat["Vs"]).astype(float)
        if Us.shape[0] != t_days.size and Us.shape[1] == t_days.size:
            Us = Us.T
            Vs = Vs.T

        U_mag = np.sqrt(Us**2 + Vs**2)
        Us_sl = surface_layer_avg(U_mag, z, Hbl, frac=0.2)

        usl_rows.append(interp_to_X(X_les, Us_sl, X_ref))

    USL = np.vstack(usl_rows)

    # MOM6 LAT
    la_rows = []
    for y_tag in Y_FOLDER_ALL:
        f_vis = os.path.join(run_template.format(y=y_tag), "visc.nc")
        if not os.path.exists(f_vis):
            raise FileNotFoundError(f"Missing MOM6 visc.nc: {f_vis}")

        with xr.open_dataset(f_vis) as dsv:
            dsv = isel_point(dsv)
            la  = dsv["La_turbulent"].values.squeeze()

            nT = la.shape[0]
            t_mom_days = 300.0/86400.0 * (np.arange(nT) + 0.5)
            X_mom = x_from_t_days(t_mom_days)

        la_rows.append(interp_to_X(X_mom, la, X_ref))

    LAT = np.vstack(la_rows)

    # sort Y ascending (NO mirroring)
    order = np.argsort(Y_raw)
    Y   = Y_raw[order]
    USL = USL[order, :]
    LAT = LAT[order, :]

    return dict(
        X2_off=X2_off, Y2_off=Y2_off, TAU_off=TAU_off,
        Y=Y, USL=USL, LAT=LAT
    )

# =========================
# 3) PLOTTING FUNCTIONS (your original style)
# =========================
cmap = cmap_turbo()

def panel_tau_like_original(ax, X2_off, Y2_off, TAU_off):
    vmax = np.nanmax(TAU_off)
    levels_c = [0.35*vmax, 0.65*vmax, 0.9*vmax]

    cf = ax.contourf(X2_off, Y2_off, TAU_off, levels=TAU_LEVELS, cmap=cmap, extend="max")
    ax.contour(X2_off, Y2_off, TAU_off, levels=levels_c, colors="k", linewidths=1.0)
    draw_rmw_circle(ax, r=1.0)

    # arrow -> bold vector
    ax.set_title(r"$|\boldsymbol{\tau}|\,(\mathrm{N\,m^{-2}})$", fontsize=13)

    ax.set_xlabel(r"$(x-x_c)/RMW$")
    ax.set_ylabel(r"$(y-y_c)/RMW$")
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")
    return cf

def panel_like_your_LES(ax, X, Y, F, title, clim):
    X2, Y2 = np.meshgrid(X, Y)
    vmin, vmax = clim
    levels_f = np.linspace(vmin, vmax, 21)
    if vmin == 0:
        levels_c = [0.3*vmax, 0.6*vmax, 0.9*vmax]
    else:
        levels_c = np.linspace(vmin, vmax, 5)[1:-1]

    cf = ax.contourf(X2, Y2, F, levels=levels_f, cmap=cmap, extend="both")
    ax.contour(X2, Y2, F, levels=levels_c, colors="k", linewidths=0.9)
    draw_rmw_circle(ax, r=1.0)

    ax.set_title(title, fontsize=13)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel(r"$(x-x_c)/RMW$")
    ax.set_aspect("equal", adjustable="box")
    return cf

# =========================
# 4) BUILD BOTH ROWS
# =========================
row0 = build_row(
    VTRAN=5.0, X0=6.48e5, transpeed=5.0,
    les_lt_root="/archive/bgr/Datasets/LES/MoreHurr/TS05_ML10/LT/",
    les_st_root="/archive/bgr/Datasets/LES/MoreHurr/TS05_ML10/ST/",
    run_template="/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/waveLT_res_10m/{y}_results"
)

row1 = build_row(
    VTRAN=10.0, X0=1.296e6, transpeed=10.0,
    les_lt_root="/archive/bgr/Datasets/LES/MoreHurr/TS10_ML10/LT/",
    les_st_root="/archive/bgr/Datasets/LES/MoreHurr/TS10_ML10/ST/",
    run_template="/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/waveLT_res_10m_10mps/{y}_results"
)

# =========================
# 5) PLOT: 2 rows × 3 cols (each with its own colorbar axis)
# =========================
fig = plt.figure(figsize=FIGSIZE)

# ---- layout tuning (RIGHT-side ticks, no overlap) ----
CBAR_W = 0.06   # wider than before so tick labels fit without spilling into next panel
WSPACE = 0.45    # moderate spacing; still close but prevents overlap
HSPACE = 0.035

gs = fig.add_gridspec(
    nrows=2, ncols=6,
    width_ratios=[1.0, CBAR_W, 1.0, CBAR_W, 1.0, CBAR_W],
    height_ratios=[1.0, 1.0],
    wspace=WSPACE, hspace=HSPACE
)

# axes: (row, colpanel) -> (ax, cax)
axs = [[None]*3 for _ in range(2)]
caxs = [[None]*3 for _ in range(2)]
for r in range(2):
    axs[r][0]  = fig.add_subplot(gs[r, 0])
    caxs[r][0] = fig.add_subplot(gs[r, 1])

    axs[r][1]  = fig.add_subplot(gs[r, 2], sharey=axs[r][0])
    caxs[r][1] = fig.add_subplot(gs[r, 3])

    axs[r][2]  = fig.add_subplot(gs[r, 4], sharey=axs[r][0])
    caxs[r][2] = fig.add_subplot(gs[r, 5])

def _shrink_cax(cax, shrink=0.80):
    pos = cax.get_position()
    new_h = pos.height * shrink
    new_y0 = pos.y0 + 0.5 * (pos.height - new_h)
    cax.set_position([pos.x0, new_y0, pos.width, new_h])

    # keep cbar always on top
    cax.set_zorder(50)
    cax.patch.set_alpha(1.0)

def widen_cax(cax, widen=1.18):
    """Slightly widen a specific cbar axis (useful for the middle column)."""
    pos = cax.get_position()
    new_w = pos.width * widen
    cax.set_position([pos.x0, pos.y0, new_w, pos.height])
    cax.set_zorder(50)
    cax.patch.set_alpha(1.0)

# shrink all cbars a bit
for r in range(2):
    for j in range(3):
        _shrink_cax(caxs[r][j], shrink=0.80)

# OPTIONAL but recommended: widen ONLY the 2nd-column colorbars a touch
for r in range(2):
    widen_cax(caxs[r][1], widen=1.18)

# keep main panels below cbar (visual layering safety)
for r in range(2):
    for c in range(3):
        axs[r][c].set_zorder(1)

# ---- Row 0 (5 m/s) ----
cf00 = panel_tau_like_original(axs[0][0], row0["X2_off"], row0["Y2_off"], row0["TAU_off"])
cb00 = fig.colorbar(cf00, cax=caxs[0][0])
cb00.ax.yaxis.set_ticks_position("right")
cb00.ax.yaxis.set_label_position("right")
cb00.ax.tick_params(pad=1)
cb00.ax.set_zorder(60)

cf01 = panel_like_your_LES(
    axs[0][1], X_ref, row0["Y"], row0["USL"],
    r"$\langle|\boldsymbol{U}_s|\rangle_{SL}\,(\mathrm{m\,s^{-1}})$",
    USL_CLIM
)
cb01 = fig.colorbar(cf01, cax=caxs[0][1])
cb01.ax.yaxis.set_ticks_position("right")
cb01.ax.yaxis.set_label_position("right")
cb01.ax.tick_params(pad=1)
cb01.ax.set_zorder(60)

cf02 = panel_like_your_LES(
    axs[0][2], X_ref, row0["Y"], row0["LAT"],
    r"$La_{\mathrm{t}}$",
    LAT_CLIM
)
mark_locations_right_edge(axs[0][2], row0["Y"], x_edge=xlim[1]-0.08)
cb02 = fig.colorbar(cf02, cax=caxs[0][2])
cb02.ax.yaxis.set_ticks_position("right")
cb02.ax.yaxis.set_label_position("right")
cb02.ax.tick_params(pad=1)
cb02.ax.set_zorder(60)

# ---- Row 1 (10 m/s) ----
cf10 = panel_tau_like_original(axs[1][0], row1["X2_off"], row1["Y2_off"], row1["TAU_off"])
cb10 = fig.colorbar(cf10, cax=caxs[1][0])
cb10.ax.yaxis.set_ticks_position("right")
cb10.ax.yaxis.set_label_position("right")
cb10.ax.tick_params(pad=1)
cb10.ax.set_zorder(60)

cf11 = panel_like_your_LES(
    axs[1][1], X_ref, row1["Y"], row1["USL"],
    r"$\langle|\boldsymbol{U}_s|\rangle_{SL}\,(\mathrm{m\,s^{-1}})$",
    USL_CLIM
)
cb11 = fig.colorbar(cf11, cax=caxs[1][1])
cb11.ax.yaxis.set_ticks_position("right")
cb11.ax.yaxis.set_label_position("right")
cb11.ax.tick_params(pad=1)
cb11.ax.set_zorder(60)

cf12 = panel_like_your_LES(
    axs[1][2], X_ref, row1["Y"], row1["LAT"],
    r"$La_{\mathrm{t}}$",
    LAT_CLIM
)
mark_locations_right_edge(axs[1][2], row1["Y"], x_edge=xlim[1]-0.08)
cb12 = fig.colorbar(cf12, cax=caxs[1][2])
cb12.ax.yaxis.set_ticks_position("right")
cb12.ax.yaxis.set_label_position("right")
cb12.ax.tick_params(pad=1)
cb12.ax.set_zorder(60)

# ---- y-label cleanup: only first column keeps y tick labels per row ----
for r in range(2):
    for c in (1, 2):
        axs[r][c].set_ylabel("")
        axs[r][c].tick_params(labelleft=False)

# ---- consistent ticks ----
for r in range(2):
    for c in range(3):
        axs[r][c].set_xticks([-4, -2, 0, 2, 4])
        axs[r][c].set_yticks([-4, -2, 0, 2, 4])

# ---- tags ----
for r in range(2):
    axs[r][0].text(-0.22, 1.03, ROW_TAGS[r],
                   transform=axs[r][0].transAxes,
                   ha="left", va="bottom", fontsize=14)

for j in range(3):
    axs[0][j].text(0.02, 1.03, COL_TAGS[j],
                   transform=axs[0][j].transAxes,
                   ha="left", va="bottom", fontsize=14)

plt.show()

fig.savefig(SAVE_EPS, format="eps", bbox_inches="tight", pad_inches=0.03)
print("Saved:", SAVE_EPS)


In [1]:
#this code generates figure 4:Wind and wave forcing for the three-dimensional idealized cases. 
#Wind stress magnitude $|\boldsymbol{\tau}|$(left),
#significant wave height $H_s$ from wave model (center), 
#and turbulent Langmuir number $La_t$ (right) for a idealized TC moving 
#at 5 $\rm ms^{-1}$ and 2 $\rm ms^{-1}$. $(x_0,y_0)$ is the TC center position, 
#$RMW$ refers to radius of maximum wind. The horizontal and vertical axes 
#are the horizontal and vertical distance to the TC center, 
#normalized by the radius of maximum wind (50 km). The black dashed circle indicates 1$RMW$ 
#and the black contours indicates isolines.

#!/usr/bin/env python3
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from netCDF4 import Dataset

# =========================
# 0) constants / domain
# =========================
RMW   = 50e3          # m
DX    = 5e3           # m
DY    = 5e3           # m
Nx    = 600
Ny    = 360
Lx_m  = 3.0e6
Ly_m  = 1.8e6
tsec  = 3*86400       # 3 days final moment

lon_min, lon_max = -13.5, 13.5
lat_min, lat_max = -8.1, 8.1

x_m_1d = (np.arange(Nx) + 0.5) * DX
y_m_1d = (np.arange(Ny) + 0.5) * DY
X_m, Y_m = np.meshgrid(x_m_1d, y_m_1d, indexing="xy")

cmap = plt.get_cmap("turbo") if "turbo" in plt.colormaps() else plt.get_cmap("jet")
cmap = cmap.copy()
cmap.set_bad("gray")

def draw_rmw_circle(ax, r=1.0):
    th = np.linspace(0, 2*np.pi, 361)
    ax.plot(r*np.cos(th), r*np.sin(th), "k--", lw=1.2, zorder=10)

def ww3_deg_to_m(lon2d, lat2d):
    x = (lon2d - lon_min) * (Lx_m/(lon_max-lon_min))
    y = (lat2d - lat_min) * (Ly_m/(lat_max-lat_min))
    return x, y

# =========================
# 1) theoretical stress model (unchanged)
# =========================
rho_a = 1.2
p_n   = 1.012e5
p_c   = 9.68e4
Rmax  = 5.0e4
rad_edge    = 10.0
rad_ambient = 15.0
Umax  = 50.0
transdir = np.deg2rad(180.0)
fcor  = 5.5659e-5
t0_sec = 0.0

A0_0 = -14.33; A0_Rnorm = -0.9;  A0_speed = -0.09
A1_0 =  0.14;  A1_Rnorm =  0.04; A1_speed =  0.05
P1_0 = 85.31;  P1_Rnorm =  6.88; P1_speed = -9.60
Deg2Rad = np.pi/180.0

def Cd_from_du10(du10):
    return np.where(du10 < 11.0, 1.2e-3,
           np.where(du10 <= 20.0, (0.49 + 0.065*du10)*1e-3, 1.8e-3))

dp = p_n - p_c
B  = (Umax**2) * rho_a * np.e / dp

rrB_e  = rad_edge**(-B)
tmpA_e = (rrB_e*B) * dp
r_e    = rad_edge * Rmax
tmpB_e = (0.5*r_e*fcor) * rho_a
num_e  = tmpA_e * np.exp(-rrB_e)
den_e  = tmpB_e + np.sqrt((tmpA_e*rho_a)*np.exp(-rrB_e) + tmpB_e**2)
U10_edge = float(num_e / max(den_e, 1e-20))

def tau_mag_theory(tsec, transpeed, X0, Y0):
    XC = X0 + (tsec - t0_sec) * transpeed * np.cos(transdir)
    YC = Y0 + (tsec - t0_sec) * transpeed * np.sin(transdir)

    dX = X_m - XC
    dY = Y_m - YC
    r  = np.hypot(dX, dY)
    rr = r / Rmax

    U10 = np.zeros_like(r, dtype=np.float64)
    core = (rr > 1e-3) & (rr <= rad_edge)
    if np.any(core):
        rrB  = np.power(rr[core], -B)
        tmpA = (rrB*B) * dp
        tmpB = (0.5*r[core]*fcor) * rho_a
        num  = tmpA * np.exp(-rrB)
        den  = tmpB + np.sqrt((tmpA*rho_a)*np.exp(-rrB) + tmpB**2)
        U10[core] = num / np.maximum(den, 1e-20)

    shell = (rr > rad_edge) & (rr < rad_ambient)
    if np.any(shell):
        fac = (rad_ambient - rr[shell])/(rad_ambient - rad_edge)
        U10[shell] = fac * U10_edge

    Adir = np.arctan2(dY, dX)
    RSTR = np.minimum(rad_edge, rr)
    A0 = (A0_Rnorm*RSTR + A0_speed*Umax) + A0_0
    A1 = -A0*((A1_Rnorm*RSTR + A1_speed*transpeed) + A1_0)
    P1 = ((P1_Rnorm*RSTR + P1_speed*transpeed) + P1_0) * Deg2Rad
    Alph = (A0 - A1*np.cos((transdir - Adir) - P1)) * Deg2Rad
    if np.any(rr > rad_edge):
        taper = np.clip((rad_ambient - rr)/(rad_ambient - rad_edge), 0.0, 1.0)
        Alph = np.where(rr > rad_edge, Alph*taper, Alph)

    U_TS = 0.5*transpeed*np.cos(transdir)
    V_TS = 0.5*transpeed*np.sin(transdir)
    u10x = U10*np.sin(Adir - np.pi - Alph) + U_TS
    u10y = U10*np.cos(Adir - Alph)        + V_TS
    du10 = np.hypot(u10x, u10y)

    Cd   = Cd_from_du10(du10)
    Taux = rho_a * Cd * du10 * u10x
    Tauy = rho_a * Cd * du10 * u10y
    return np.hypot(Taux, Tauy).astype(np.float32), float(XC), float(YC)

# =========================
# 2) cases
# =========================
cases = [
    dict(label="5 m/s", transpeed=5.0, X0=3.432e6, Y0=9.0e5,
         visc="/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp/3D_hurricane/3D_kappa_ePBL_waveLT_5km/5_rescale_ustar0p01_results/20110401.visc.nc",
         ww3 ="/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp/3D_hurricane/3D_kappa_ePBL_waveLT_5km/5_results/ww3.20110401.nc"),
    dict(label="2 m/s", transpeed=2.0, X0=3.1728e6, Y0=9.0e5,
         visc="/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp/3D_hurricane/3D_kappa_ePBL_waveLT_5km_2mps/5_rescale_results/20110401.visc.nc",
         ww3 ="/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp/3D_hurricane/3D_kappa_ePBL_waveLT_5km_2mps/5_all_results/ww3.20110401.nc"),
]

# =========================
# 3) load fields
# =========================
out = []
hs_max = 0.0

for c in cases:
    tau_mag, XC, YC = tau_mag_theory(tsec, c["transpeed"], c["X0"], c["Y0"])
    Xn_tau = (X_m - XC)/RMW
    Yn_tau = (Y_m - YC)/RMW

    visc = xr.open_dataset(c["visc"], decode_timedelta=False)
    La = visc["La_turbulent"].isel(Time=-1).values  # (Ny,Nx)

    Xn = (X_m - XC)/RMW
    Yn = (Y_m - YC)/RMW

    f = Dataset(c["ww3"])
    hs = f.variables["hs"][-1, :, :]
    lon = f.variables["longitude"][:]
    lat = f.variables["latitude"][:]
    Xw_m, Yw_m = ww3_deg_to_m(lon, lat)
    Xn_hs = (Xw_m - XC)/RMW
    Yn_hs = (Yw_m - YC)/RMW
    hs_max = max(hs_max, np.nanmax(hs))

    out.append(dict(
        Xn_tau=Xn_tau, Yn_tau=Yn_tau, tau=tau_mag,
        Xn_hs=Xn_hs,   Yn_hs=Yn_hs,   hs=hs,
        Xn=Xn, Yn=Yn, La=La
    ))

# =========================
# 4) plotting style (keep your original)
# =========================
tau_levels = np.linspace(0.0, 6.0, 21)
tau_lines  = [2.0, 4.0, 6.0]

hs_levels  = np.linspace(0.0, hs_max, 21)

la_levels  = np.linspace(0.0, 0.5, 21)
la_lines   = [0.1, 0.3, 0.5]

xlim = (-4, 4)
ylim = (-4, 4)

# =========================
# 5) PLOT: match your "second code" layout
#    2 rows × 3 cols, EACH panel has its OWN right-side colorbar
# =========================
FIGSIZE = (13.2, 9.2)
fig = plt.figure(figsize=FIGSIZE)

CBAR_W = 0.06
WSPACE = 0.45
HSPACE = 0.06

gs = fig.add_gridspec(
    nrows=2, ncols=6,
    width_ratios=[1.0, CBAR_W, 1.0, CBAR_W, 1.0, CBAR_W],
    height_ratios=[1.0, 1.0],
    wspace=WSPACE, hspace=HSPACE
)

axs  = [[None]*3 for _ in range(2)]
caxs = [[None]*3 for _ in range(2)]
for r in range(2):
    axs[r][0]  = fig.add_subplot(gs[r, 0])
    caxs[r][0] = fig.add_subplot(gs[r, 1])

    axs[r][1]  = fig.add_subplot(gs[r, 2], sharey=axs[r][0])
    caxs[r][1] = fig.add_subplot(gs[r, 3])

    axs[r][2]  = fig.add_subplot(gs[r, 4], sharey=axs[r][0])
    caxs[r][2] = fig.add_subplot(gs[r, 5])

def _shrink_cax(cax, shrink=0.80):
    pos = cax.get_position()
    new_h = pos.height * shrink
    new_y0 = pos.y0 + 0.5 * (pos.height - new_h)
    cax.set_position([pos.x0, new_y0, pos.width, new_h])
    cax.set_zorder(50)
    cax.patch.set_alpha(1.0)

def widen_cax(cax, widen=1.18):
    pos = cax.get_position()
    new_w = pos.width * widen
    cax.set_position([pos.x0, pos.y0, new_w, pos.height])
    cax.set_zorder(50)
    cax.patch.set_alpha(1.0)

for r in range(2):
    for j in range(3):
        _shrink_cax(caxs[r][j], shrink=0.80)

for r in range(2):
    widen_cax(caxs[r][1], widen=1.18)

for r in range(2):
    for j in range(3):
        axs[r][j].set_zorder(1)

# ---- titles you requested ----
TITLE_TAU = r"$|\boldsymbol{\tau}|\,(\mathrm{N\,m^{-2}})$"
TITLE_HS  = r"$H_s\,(\mathrm{m})$"
TITLE_LAT = r"$La_{\mathrm{t}}$"

# ---- draw panels ----
m = [[None]*3 for _ in range(2)]
for i in range(2):
    # tau
    ax = axs[i][0]
    m[i][0] = ax.contourf(out[i]["Xn_tau"], out[i]["Yn_tau"],
                          np.ma.masked_invalid(out[i]["tau"]),
                          levels=tau_levels, cmap=cmap, extend="max")
    ax.contour(out[i]["Xn_tau"], out[i]["Yn_tau"], out[i]["tau"],
               levels=tau_lines, colors="k", linewidths=1.0)
    draw_rmw_circle(ax, 1.0)
    ax.set_title(TITLE_TAU, fontsize=13)

    # Hs
    ax = axs[i][1]
    m[i][1] = ax.contourf(out[i]["Xn_hs"], out[i]["Yn_hs"],
                          np.ma.masked_invalid(out[i]["hs"]),
                          levels=hs_levels, cmap=cmap, extend="max")
    ax.contour(out[i]["Xn_hs"], out[i]["Yn_hs"], out[i]["hs"],
               levels=np.linspace(0.3*hs_max, 0.9*hs_max, 3),
               colors="k", linewidths=0.9)
    draw_rmw_circle(ax, 1.0)
    ax.set_title(TITLE_HS, fontsize=13)

    # La_t
    ax = axs[i][2]
    m[i][2] = ax.contourf(out[i]["Xn"], out[i]["Yn"],
                          np.ma.masked_invalid(out[i]["La"]),
                          levels=la_levels, cmap=cmap, extend="max")
    ax.contour(out[i]["Xn"], out[i]["Yn"], out[i]["La"],
               levels=la_lines, colors="k", linewidths=0.9)
    draw_rmw_circle(ax, 1.0)
    ax.set_title(TITLE_LAT, fontsize=13)

# ---- per-panel colorbars (RIGHT) ----
for r in range(2):
    for j in range(3):
        cb = fig.colorbar(m[r][j], cax=caxs[r][j])
        cb.ax.yaxis.set_ticks_position("right")
        cb.ax.yaxis.set_label_position("right")
        cb.ax.tick_params(pad=1)
        cb.ax.set_zorder(60)

# ---- cosmetics: axes limits/aspect/labels ----
for r in range(2):
    for j in range(3):
        ax = axs[r][j]
        ax.set_xlim(*xlim); ax.set_ylim(*ylim)
        ax.set_aspect("equal", adjustable="box")
        ax.tick_params(labelsize=10, length=3)
        ax.set_xticks([-4,-2,0,2,4])
        ax.set_yticks([-4,-2,0,2,4])

        ax.set_xlabel(r"$(x-x_c)/RMW$")
        if j == 0:
            ax.set_ylabel(r"$(y-y_c)/RMW$")
        else:
            ax.set_ylabel("")
            ax.tick_params(labelleft=False)

# ---- remove your old row labels: (no 5m/s or 2m/s text) ----
# (nothing to do; we simply do not draw them)

# ---- tags ----
ROW_TAGS = ["(a)", "(b)"]
COL_TAGS = ["(i)", "(ii)", "(iii)"]

for r in range(2):
    axs[r][0].text(-0.22, 1.03, ROW_TAGS[r],
                   transform=axs[r][0].transAxes,
                   ha="left", va="bottom", fontsize=14)

for j in range(3):
    axs[0][j].text(0.02, 1.03, COL_TAGS[j],
                   transform=axs[0][j].transAxes,
                   ha="left", va="bottom", fontsize=14)

plt.show()
# fig.savefig("Fig_tau_hs_lat_like_panel_cbar.eps", format="eps", bbox_inches="tight", pad_inches=0.03)


#

In [ ]:
#this code is for figure 5:Wind and wave forcing for the Hurricane Ivan (2004) 
#in the Northeast Atlantic ocean. Wind stress magnitude $|\boldsymbol{\tau}|$(top), 
#significant wave height $H_s$ from wave model (center), 
#and turbulent Langmuir number $La_t$ (bottom) for Hurricane Ivan during 12-16 September, 2004. 
#The red line indicates the track of Ivan obtained by the best track and the red dot mark 
#above the line indicates the center of the location at 00:00 UTC of corresponding dates 
#labeled aside.

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# ============================================================
# 0) PATHS (edit)
# ============================================================
MODEL_TRACK_CSV = "/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_ZC/c6/postProcessing/buoydata/2004/ivan_Model_track_hourly.csv"
OCEAN_WAVEALL_FILE = "/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp_C6/NWA12_wave_Ivan/20040907_results_all/20040907.ocean_hourly.nc"
WW3_FILE = "/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp_C6/NWA12_wave_Ivan/20040907_results_5036_waveall/ww3.20040907.nc"

# ============================================================
# 1) FIG SETTINGS
# ============================================================
LON_BOUNDS = (-100, -75)
LAT_BOUNDS = (5, 33)

# --- show maps at 00:00 UTC each day ---
TIMES_SHOW = [
    "2004-09-12T00:00",
    "2004-09-13T00:00",
    "2004-09-14T00:00",
    "2004-09-15T00:00",
    "2004-09-16T00:00",
]

# -------- Track cutoff (NO 09/17, 09/18) --------
TRACK_END_EXCLUSIVE = "2004-09-17T00:00"   # keep times < this

# Track style (red)
TRACK_COLOR = "red"
TRACK_LW = 1.8
TRACK_MARKER = "o"
TRACK_MARKER_SIZE = 7

# Label style: DAILY 00:00 UTC
LABEL_AT_HOUR = 0
LABEL_TEXT_FMT = "%m/%d"
LABEL_DX = 0.15
LABEL_DY = 0.15
LABEL_BBOX = dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.65)

LAND_COLOR = "lightgray"
ROBUST_PCTS = (2, 98)
LA_FIXED_RANGE = (0.0, 1.0)  # set None if you prefer robust range

# ============================================================
# 2) helpers
# ============================================================
def lon360_to_180(lon):
    lon = np.asarray(lon)
    return ((lon + 180.0) % 360.0) - 180.0

def ensure_lon180_and_sort(ds, lon_names):
    ds = ds.copy()
    for nm in lon_names:
        if nm in ds.coords:
            v = ds[nm].values
            if np.nanmax(v) > 180.0:
                ds = ds.assign_coords({nm: lon360_to_180(v)}).sortby(nm)
    return ds

def tau_mag_cellcenter(ds):
    taux_c = ds["taux"].interp(xq=ds["xh"])
    tauy_c = ds["tauy"].interp(yq=ds["yh"])
    return np.hypot(taux_c, tauy_c).rename("tau_mag")

def make_land_mask_from_sst(ds, lon_bounds, lat_bounds, sst_name="SST_hm"):
    if sst_name not in ds:
        return None
    sst0 = ds[sst_name].isel(time=0).sel(
        xh=slice(lon_bounds[0], lon_bounds[1]),
        yh=slice(lat_bounds[0], lat_bounds[1])
    )
    land = np.isnan(sst0.values)
    return land, sst0["xh"].values, sst0["yh"].values

def plot_land(ax, lon1d, lat1d, land_mask, land_color="lightgray"):
    if land_mask is None:
        return
    cmap_land = ListedColormap(["none", land_color])
    land_int = land_mask.astype(int)
    Lon2d, Lat2d = np.meshgrid(lon1d, lat1d)
    ax.pcolormesh(Lon2d, Lat2d, land_int, cmap=cmap_land, vmin=0, vmax=1,
                  shading="auto", zorder=0)

def robust_vmin_vmax(arr_list, pcts=(2,98)):
    vals = np.concatenate([np.ravel(a[np.isfinite(a)]) for a in arr_list if a is not None])
    if vals.size == 0:
        return None, None
    return np.percentile(vals, pcts[0]), np.percentile(vals, pcts[1])

def subset_ww3_if_2d(lon2d, lat2d, field2d, lon_bounds, lat_bounds):
    mask = (
        (lon2d >= lon_bounds[0]) & (lon2d <= lon_bounds[1]) &
        (lat2d >= lat_bounds[0]) & (lat2d <= lat_bounds[1])
    )
    out = np.array(field2d, copy=True)
    out[~mask] = np.nan
    return out

def prep_model_track(df, track_end_exclusive):
    df = df.copy()
    df["time"] = pd.to_datetime(df["time"])
    if df["lon_smooth"].max() > 180:
        df["lon_smooth"] = lon360_to_180(df["lon_smooth"].values)
    df = df.sort_values("time").reset_index(drop=True)
    t_end = pd.to_datetime(track_end_exclusive)
    df = df[df["time"] < t_end].copy().reset_index(drop=True)
    return df

def pick_track_labels(model_track, label_hour=0, fmt="%m/%d"):
    df = model_track.copy()
    df = df[(df["time"].dt.hour == label_hour) & (df["time"].dt.minute == 0)]
    df["day"] = df["time"].dt.floor("D")
    df = df.drop_duplicates(subset=["day"])
    df["label"] = df["time"].dt.strftime(fmt)
    return df

def draw_track(ax, model_track, labels_df=None,
               line_color="red", lw=1.8,
               text_color="k", dx=0.15, dy=0.15,
               marker="o", ms=5):
    ax.plot(model_track["lon_smooth"].values,
            model_track["lat_smooth"].values,
            color=line_color, lw=lw, zorder=6)

    if labels_df is not None and not labels_df.empty:
        ax.plot(labels_df["lon_smooth"].values,
                labels_df["lat_smooth"].values,
                linestyle="none",
                marker=marker,
                markersize=ms,
                markerfacecolor=line_color,
                markeredgecolor=line_color,
                zorder=7)

        for _, r in labels_df.iterrows():
            ax.text(r["lon_smooth"] + dx, r["lat_smooth"] + dy, r["label"],
                    color=text_color, fontsize=9, zorder=8, bbox=LABEL_BBOX)

# ============================================================
# 3) track + cutoff + daily 00 labels
# ============================================================
model_track = pd.read_csv(MODEL_TRACK_CSV, parse_dates=["time"])
model_track = prep_model_track(model_track, TRACK_END_EXCLUSIVE)
labels_df = pick_track_labels(model_track, label_hour=LABEL_AT_HOUR, fmt=LABEL_TEXT_FMT)

# ============================================================
# 4) MOM6 (tau + La_t)
# ============================================================
ds_o = xr.open_dataset(OCEAN_WAVEALL_FILE, decode_times=True)
ds_o = ensure_lon180_and_sort(ds_o, ["xh","xq"])
ds_o = ds_o.sel(xh=slice(LON_BOUNDS[0], LON_BOUNDS[1]),
                yh=slice(LAT_BOUNDS[0], LAT_BOUNDS[1]))

tau_mag = tau_mag_cellcenter(ds_o)

if "La_turbulent_hm" not in ds_o:
    raise KeyError("La_turbulent_hm not found in ocean_hourly.nc")
la_t = ds_o["La_turbulent_hm"]

land_info = make_land_mask_from_sst(ds_o, LON_BOUNDS, LAT_BOUNDS)
land_mask, land_lon, land_lat = (None, None, None)
if land_info is not None:
    land_mask, land_lon, land_lat = land_info

xh = ds_o["xh"].values
yh = ds_o["yh"].values

# ============================================================
# 5) WW3 Hs
# ============================================================
ds_w = xr.open_dataset(WW3_FILE, decode_times=True)
lon_name = "longitude" if "longitude" in ds_w.coords else "lon"
lat_name = "latitude"  if "latitude"  in ds_w.coords else "lat"

if ds_w[lon_name].ndim == 1 and np.nanmax(ds_w[lon_name].values) > 180:
    ds_w = ds_w.assign_coords({lon_name: lon360_to_180(ds_w[lon_name].values)}).sortby(lon_name)

hs = ds_w["hs"]

if ds_w[lon_name].ndim == 1 and ds_w[lat_name].ndim == 1:
    hs = hs.sel(**{lon_name: slice(LON_BOUNDS[0], LON_BOUNDS[1]),
                   lat_name: slice(LAT_BOUNDS[0], LAT_BOUNDS[1])})

# ============================================================
# 6) Compute consistent color limits across selected times
# ============================================================
times_actual = [pd.to_datetime(t) for t in TIMES_SHOW]

tau_maps, hs_maps, la_maps = [], [], []
for t in times_actual:
    tau2 = tau_mag.sel(time=t, method="nearest").values
    la2  = la_t.sel(time=t, method="nearest").values

    hs2_da = hs.sel(time=t, method="nearest")
    hs2 = hs2_da.values
    if ds_w[lon_name].ndim == 2 or ds_w[lat_name].ndim == 2:
        hs2 = subset_ww3_if_2d(ds_w[lon_name].values, ds_w[lat_name].values, hs2, LON_BOUNDS, LAT_BOUNDS)

    tau_maps.append(tau2)
    la_maps.append(la2)
    hs_maps.append(hs2)

# keep your fixed ranges (as in your script)
tau_vmin, tau_vmax = (0, 1)
hs_vmin,  hs_vmax  = (0, 10)

if LA_FIXED_RANGE is not None:
    la_vmin, la_vmax = LA_FIXED_RANGE
else:
    la_vmin, la_vmax = robust_vmin_vmax(la_maps, ROBUST_PCTS)
la_vmin, la_vmax = (0, 0.5)

# ============================================================
# 7) Plot mosaic + tags + label rules
# ============================================================
N = len(times_actual)
fig, axes = plt.subplots(3, N, figsize=(4.5*N, 10), constrained_layout=True)

ROW_TAGS = ["(a)", "(b)", "(c)"]
COL_TAGS = ["(i)", "(ii)", "(iii)", "(iv)", "(v)"]

for j, t in enumerate(times_actual):
    tstamp = pd.to_datetime(t).strftime("%m-%d %H UTC")  # now should be 00 UTC

    # Row 1: |tau|
    ax = axes[0, j]
    plot_land(ax, land_lon if land_lon is not None else xh,
                 land_lat if land_lat is not None else yh,
                 land_mask, land_color=LAND_COLOR)

    im_tau = ax.pcolormesh(xh, yh, tau_mag.sel(time=t, method="nearest").values,
                           vmin=tau_vmin, vmax=tau_vmax, shading="auto", zorder=1)
    draw_track(ax, model_track, labels_df,
               line_color=TRACK_COLOR, lw=TRACK_LW,
               text_color="k", dx=LABEL_DX, dy=LABEL_DY,
               marker=TRACK_MARKER, ms=TRACK_MARKER_SIZE)

    # column title (time)
    ax.set_title(tstamp)
    ax.set_xlim(LON_BOUNDS); ax.set_ylim(LAT_BOUNDS)

    # x labels: row0/row1 none; row2 yes
    ax.set_xlabel("")
    if j == 0:
        ax.set_ylabel("Latitude")
    else:
        ax.set_ylabel("")
        ax.tick_params(labelleft=False)

    ax.grid(True, alpha=0.2)

    # Row 2: Hs
    ax = axes[1, j]
    plot_land(ax, land_lon if land_lon is not None else xh,
                 land_lat if land_lat is not None else yh,
                 land_mask, land_color=LAND_COLOR)

    hs2_da = hs.sel(time=t, method="nearest")
    if ds_w[lon_name].ndim == 1 and ds_w[lat_name].ndim == 1:
        im_hs = ax.pcolormesh(hs2_da[lon_name].values, hs2_da[lat_name].values, hs2_da.values,
                              vmin=hs_vmin, vmax=hs_vmax, shading="auto", zorder=1)
    else:
        lon2d = ds_w[lon_name].values
        lat2d = ds_w[lat_name].values
        hs2 = subset_ww3_if_2d(lon2d, lat2d, hs2_da.values, LON_BOUNDS, LAT_BOUNDS)
        im_hs = ax.pcolormesh(lon2d, lat2d, hs2,
                              vmin=hs_vmin, vmax=hs_vmax, shading="auto", zorder=1)

    draw_track(ax, model_track, labels_df,
               line_color=TRACK_COLOR, lw=TRACK_LW,
               text_color="k", dx=LABEL_DX, dy=LABEL_DY,
               marker=TRACK_MARKER, ms=TRACK_MARKER_SIZE)

    ax.set_xlim(LON_BOUNDS); ax.set_ylim(LAT_BOUNDS)
    ax.set_xlabel("")
    if j == 0:
        ax.set_ylabel("Latitude")
    else:
        ax.set_ylabel("")
        ax.tick_params(labelleft=False)
    ax.grid(True, alpha=0.2)

    # Row 3: La_t
    ax = axes[2, j]
    plot_land(ax, land_lon if land_lon is not None else xh,
                 land_lat if land_lat is not None else yh,
                 land_mask, land_color=LAND_COLOR)

    im_la = ax.pcolormesh(xh, yh, la_t.sel(time=t, method="nearest").values,
                          vmin=la_vmin, vmax=la_vmax, shading="auto", zorder=1)

    draw_track(ax, model_track, labels_df,
               line_color=TRACK_COLOR, lw=TRACK_LW,
               text_color="k", dx=LABEL_DX, dy=LABEL_DY,
               marker=TRACK_MARKER, ms=TRACK_MARKER_SIZE)

    ax.set_xlim(LON_BOUNDS); ax.set_ylim(LAT_BOUNDS)
    ax.set_xlabel("Longitude")
    if j == 0:
        ax.set_ylabel("Latitude")
    else:
        ax.set_ylabel("")
        ax.tick_params(labelleft=False)
    ax.grid(True, alpha=0.2)

# ---- Row labels (left side) ----
axes[0,0].text(-0.17, 0.5, r"$|\boldsymbol{\tau}|\,(\mathrm{N\,m^{-2}})$",
               transform=axes[0,0].transAxes, rotation=90,
               va="center", ha="center", fontsize=12)
axes[1,0].text(-0.17, 0.5, r"$H_s\,(\mathrm{m})$",
               transform=axes[1,0].transAxes, rotation=90,
               va="center", ha="center", fontsize=12)
axes[2,0].text(-0.17, 0.5, r"$La_{\mathrm{t}}$",
               transform=axes[2,0].transAxes, rotation=90,
               va="center", ha="center", fontsize=12)

# ---- tags: row tags (a)(b)(c) ----
for r in range(3):
    axes[r,0].text(-0.25, 1.03, ROW_TAGS[r],
                   transform=axes[r,0].transAxes,
                   ha="left", va="bottom", fontsize=14)

# ---- tags: column tags (i..v) ONLY on first row ----
for j in range(N):
    axes[0,j].text(0.02, 1.03, COL_TAGS[j],
                   transform=axes[0,j].transAxes,
                   ha="left", va="bottom", fontsize=14)

# ---- Shared colorbars per row (as your original) ----
cbar0 = fig.colorbar(im_tau, ax=axes[0,:].ravel().tolist(), fraction=0.015, pad=0.01)
cbar0.set_label(r"$\mathrm{N\,m^{-2}}$")

cbar1 = fig.colorbar(im_hs, ax=axes[1,:].ravel().tolist(), fraction=0.015, pad=0.01)
cbar1.set_label(r"$\mathrm{m}$")

cbar2 = fig.colorbar(im_la, ax=axes[2,:].ravel().tolist(), fraction=0.015, pad=0.01)
cbar2.set_label("")  # dimensionless; leave blank

plt.show()





In [ ]:
##this code is for figure 6:Sea surface temperature (SST) comparisons 
#between LES and MOM6-SCM for one-dimensional idealized hurricanes translating 
#at 5 and 10 $\rm ms^{-1}$. Shown are three representative locations: 
#a right-of-track point ($y_t$=30 km), a left-of-track point ($y_t$=-40 km) 
#and centerline $y_t$=0 km. "shear" results are obtained by using $\kappa$-shear alone 
#and "ePBL" represents results simulated by RH18 alone without combination 
#with $\kappa$-shear scheme. Hybrid ST+LT (res) refers to the results 
#with LT enhancement calculated by Eq. \ref{eq:rescaling2} with the rescaling factor, 
#whereas Hybrid ST+LT (nores) represents RH18 with RL19 LT enhancement calculated by 
#Eq. \ref{eq:RL19}. $\Delta$ refers to the SST difference between ST+LT results 
#and corresponding ST results.

#!/usr/bin/env python3
import os
import numpy as np
import scipy.io as sio
import xarray as xr
import matplotlib.pyplot as plt

# ============================================================
# COMBINED FIGURE: 6 rows × 3 cols
#   Rows 0-2: 5 mps   (baseline, ST+LT, Δ)
#   Rows 3-5: 10 mps  (baseline, ST+LT, Δ)
#
# Labels:
#   Row tags: (a)(b)(c)(d)(e)(f)
#   Col tags: ONLY on first row: (i)(ii)(iii)
#
# Text changes requested:
#   - "Hybrid ST (kappa-ePBL)" -> "Hybrid ST"
#   - "SST ST+LT" -> "SST_ST+LT"
#   - "y/Rmax" -> "(y-y_c)/RMW"
#
# NOTE: keep all physics/data/ylims logic same as your originals.
# ============================================================

# -------------------------------
# COMMON settings
# -------------------------------
RMW = 50e3          # m
RMAX_KM = 50.0      # km (for choosing points)
Y_POINTS_R = (0.6, 0.0, -0.8)  # choose 3 points

# X range (days)
XLIM = (1, 3)

# Y lists (same for both speeds)
Y_KM_ALL      = [300, 200, 150, 130, 110, 90, 70, 50, 30, 10, 0, -20, -40, -60, -80, -100, -120, -140, -200, -300]
Y_FOLDER_ALL  = ["300","200","150","130","110","90","70","50","30","10","0","S20","S40","S60","S80","S100","S120","S140","S200","S300"]
LES_LT_CASES  = ["021","031","036","038","040","042","044","046","048","050","051","053","055","057","059","061","063","065","071","081"]
LES_NLT_CASES = ["021","031","036","038","040","042","044","046","048","050","051","053","055","057","059","061","063","065","071","081"]

# Row tags + Col tags
ROW_TAGS_6 = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]
COL_TAGS_3 = ["(i)", "(ii)", "(iii)"]

# -------------------------------
# Speed configs (ONLY paths/ylims differ)
# -------------------------------
cfg_5 = dict(
    name="5mps",
    les_lt_root="/archive/bgr/Datasets/LES/MoreHurr/TS05_ML10/LT/",
    les_st_root="/archive/bgr/Datasets/LES/MoreHurr/TS05_ML10/ST/",
    model_dirs={
        "energy_coupling_noLT"            : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/noLT_10m/{y}_results",
        "energy_coupling_waveLT_all"      : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/waveLT_res_10m/{y}_results",
        "energy_coupling_waveLT_norescale": "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/waveLT_nores_10m/{y}_results",
        "shear"                           : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/shear_10m/{y}_results",
        "ePBL"                            : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/ePBL_noshear_10m/{y}_results",
    },
    # keep your original intent: baseline/LT auto, delta fixed
    YLIM_BASELINE=None,
    YLIM_LT=None,
    YLIM_DELTA=(-0.5, 0.5),
)

cfg_10 = dict(
    name="10mps",
    les_lt_root="/archive/bgr/Datasets/LES/MoreHurr/TS10_ML10/LT/",
    les_st_root="/archive/bgr/Datasets/LES/MoreHurr/TS10_ML10/ST/",
    model_dirs={
        "energy_coupling_noLT"            : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/noLT_10m_10mps/{y}_results",
        "energy_coupling_waveLT_all"      : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/waveLT_res_10m_10mps/{y}_results",
        "energy_coupling_waveLT_norescale": "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/waveLT_nores_10m_10mps/{y}_results",
        "shear"                           : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/shear_10m_10mps/{y}_results",
        "ePBL"                            : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/ePBL_noshear_10m_10mps/{y}_results",
    },
    # keep your original intent: all auto
    YLIM_BASELINE=None,
    YLIM_LT=None,
    YLIM_DELTA=None,
)

# output (optional)

SAVE_EPS = "/archive/Qian.Xiao/Qian.Xiao/eps/SSTSCM"
DO_SAVE  = False


# ============================================================
# Helpers (same logic as your originals)
# ============================================================
def interp_to(t_src, y_src, t_tgt):
    t_src = np.asarray(t_src)
    y_src = np.asarray(y_src)
    t_tgt = np.asarray(t_tgt)
    if t_src.size == 0:
        return np.full_like(t_tgt, np.nan, dtype=float)
    if np.any(np.diff(t_src) < 0):
        order = np.argsort(t_src)
        t_src = t_src[order]
        y_src = y_src[order]
    return np.interp(t_tgt, t_src, y_src, left=y_src[0], right=y_src[-1])


def load_les_sst_from_LESout(mat_path):
    """
    Read LESout.mat:
      t : seconds
      T : (Nt, Nz) Kelvin   (sometimes transposed)
    return t_days, sst_C
    """
    m = sio.loadmat(mat_path)

    t = np.asarray(m["t"]).squeeze().astype("f8")
    t_days = t / 86400.0

    if "T" not in m:
        raise KeyError(f"No T in {mat_path}. Keys: {list(m.keys())}")

    T = np.asarray(m["T"]).astype("f8")
    # ensure (Nt, Nz)
    if T.ndim == 2 and T.shape[0] != t_days.size and T.shape[1] == t_days.size:
        T = T.T

    sst = T[:, 0].squeeze()
    # Kelvin -> Celsius
    if np.nanmedian(sst) > 100.0:
        sst = sst - 273.15

    del m
    return t_days, sst


def load_les_pair_sst(case_id, les_lt_root, les_st_root):
    """
    LT:  <les_lt_root>/<case_id>/LESout.mat
    ST:  <les_st_root>/<case_id>/LESout.mat
    """
    f_lt = os.path.join(les_lt_root, case_id, "LESout.mat")
    f_st = os.path.join(les_st_root, case_id, "LESout.mat")

    if not os.path.exists(f_lt):
        raise FileNotFoundError(f"Missing LT LES file: {f_lt}")
    if not os.path.exists(f_st):
        raise FileNotFoundError(f"Missing ST(noLT) LES file: {f_st}")

    t_lt, sst_lt = load_les_sst_from_LESout(f_lt)
    t_st, sst_st = load_les_sst_from_LESout(f_st)

    return (t_lt, sst_lt), (t_st, sst_st)


def load_mom_sst_from_prog(results_dir):
    prog = os.path.join(results_dir, "prog.nc")
    if not os.path.exists(prog):
        raise FileNotFoundError(f"Missing file: {prog}")

    with xr.open_dataset(prog) as ds:
        t_days = 300.0/86400.0 * (np.arange(ds.Time.size) + 0.5)

        if "temp" in ds.variables:
            da = ds["temp"]
        elif "thetao" in ds.variables:
            da = ds["thetao"]
        elif "Temp" in ds.variables:
            da = ds["Temp"]
        else:
            raise KeyError(f"No temperature var found in {prog}. Vars: {list(ds.variables)}")

        indexers = {}
        if "xh" in da.dims: indexers["xh"] = 0
        if "yh" in da.dims: indexers["yh"] = 0
        if "zl" in da.dims: indexers["zl"] = 0

        sst = da.isel(**indexers).values

    return t_days, sst


def _nearest_available_y_km(y_km, tol_km=1e-6):
    y_arr = np.array(Y_KM_ALL, dtype=float)
    j = int(np.argmin(np.abs(y_arr - y_km)))
    if abs(y_arr[j] - y_km) > tol_km:
        raise ValueError(f"Requested y={y_km} km not found in available {Y_KM_ALL}.")
    return float(y_arr[j])


def _auto_ylim(values_list, pad=0.03):
    vv = np.concatenate([np.ravel(v) for v in values_list if v is not None and np.size(v) > 0])
    vv = vv[np.isfinite(vv)]
    if vv.size == 0:
        return None
    vmin, vmax = float(vv.min()), float(vv.max())
    if vmin == vmax:
        return (vmin - 0.1, vmax + 0.1)
    dv = vmax - vmin
    return (vmin - pad*dv, vmax + pad*dv)


def build_sst_data(cfg, y_points_r=None, y_points_km=None):
    """
    Returns list of 3 records (L/C/R):
      rec = {
        'y_km': float,
        'y_norm': float (= (y-yc)/RMW),
        'LES': (t_lt, sst_lt),
        'LES_noLT': (t_st, sst_st),
        'MOM': { key: (t_mom, sst_mom), ... }
      }
    """
    if (y_points_r is None) and (y_points_km is None):
        raise ValueError("Provide y_points_r or y_points_km")

    if y_points_km is None:
        y_points_km = tuple(_nearest_available_y_km(r * RMAX_KM) for r in y_points_r)
    else:
        y_points_km = tuple(_nearest_available_y_km(y) for y in y_points_km)

    y_to_idx = {float(y): i for i, y in enumerate(Y_KM_ALL)}
    need_keys = [
        "energy_coupling_noLT",
        "energy_coupling_waveLT_all",
        "energy_coupling_waveLT_norescale",
        "shear",
        "ePBL",
    ]

    out = []
    for y in y_points_km:
        i = y_to_idx[float(y)]
        y_tag = Y_FOLDER_ALL[i]

        case_id = LES_LT_CASES[i]  # same list order
        (t_lt, sst_lt), (t_st, sst_st) = load_les_pair_sst(case_id, cfg["les_lt_root"], cfg["les_st_root"])

        mom = {}
        for k in need_keys:
            base = cfg["model_dirs"][k].format(y=y_tag)
            mom[k] = load_mom_sst_from_prog(base)

        out.append({
            "y_km": float(y),
            "y_norm": float(y) * 1e3 / RMW,   # (y - y_c)/RMW  (y_c=0)
            "LES": (t_lt, sst_lt),
            "LES_noLT": (t_st, sst_st),
            "MOM": mom
        })

    return out


# ============================================================
# Plot combined 6x3
# ============================================================
def plot_SCM_Fig2_SST_6x3(sst5, sst10, cfg5, cfg10, xlim=(1, 3),
                          save_png=None, save_eps=None, do_save=False):

    KEY_BASE  = "energy_coupling_noLT"
    KEY_ALL   = "energy_coupling_waveLT_all"
    KEY_NRS   = "energy_coupling_waveLT_norescale"
    KEY_SHEAR = "shear"
    KEY_EPBL  = "ePBL"

    # 6 rows x 3 cols
    fig, ax = plt.subplots(6, 3, figsize=(15, 15.8), sharex=True, sharey='row')

    # pack
    blocks = [
        (sst5,  cfg5,  0),  # rows 0-2
        (sst10, cfg10, 3),  # rows 3-5
    ]

    # store per-row values to set common y-lim within each row
    row_vals = {r: [] for r in range(6)}

    for sst_data, cfg, r0 in blocks:
        for j, rec in enumerate(sst_data):
            y_km = rec["y_km"]
            y_n  = rec["y_norm"]
            t_lt, sst_lt = rec["LES"]
            t_st, sst_st = rec["LES_noLT"]
            mom = rec["MOM"]

            # column titles ONLY on first row (per your request)
            title_txt = rf"$(y-y_c)/RMW$={y_n:+.2f}  (y={y_km:.0f} km)"
            if r0 == 0:
                ax[0, j].set_title(title_txt)

            # ---- Row baseline
            a = ax[r0 + 0, j]
            a.plot(t_st, sst_st, "b--", lw=2.2, label="LES ST")

            t_base, s_base = mom[KEY_BASE]
            a.plot(t_base, s_base, "k-", lw=2.2, label="Hybrid ST")  # <-- changed label

            t_sh, s_sh = mom[KEY_SHEAR]
            a.plot(t_sh, s_sh, ls="--", lw=2.0, label="Shear")

            t_ep, s_ep = mom[KEY_EPBL]
            a.plot(t_ep, s_ep, ls="-", lw=2.0, label="RH18")

            a.set_xlim(*xlim)
            a.grid(True, alpha=0.25)

            if j == 0:
                a.set_ylabel("SST_ST (°C)")
                a.legend(loc="best", frameon=False)
            else:
                a.tick_params(labelleft=False)

            row_vals[r0 + 0].extend([sst_st, s_base, s_sh, s_ep])

            # ---- Row absolute ST+LT
            b = ax[r0 + 1, j]
            b.plot(t_lt, sst_lt, "b-", lw=2.4, label="LES ST+LT")

            t_all, s_all = mom[KEY_ALL]
            b.plot(t_all, s_all, "m-", lw=2.4, label="Hybrid ST+LT (res)")

            t_nrs, s_nrs = mom[KEY_NRS]
            b.plot(t_nrs, s_nrs, "m--", lw=2.4, label="Hybrid ST+LT (nores)")

            b.set_xlim(*xlim)
            b.grid(True, alpha=0.25)

            if j == 0:
                b.set_ylabel("SST_ST+LT (°C)")  # <-- changed
                b.legend(loc="best", frameon=False)
            else:
                b.tick_params(labelleft=False)

            row_vals[r0 + 1].extend([sst_lt, s_all, s_nrs])

            # ---- Row increment Δ
            c = ax[r0 + 2, j]

            sst_st_on_lt = interp_to(t_st, sst_st, t_lt)
            d_les = sst_lt - sst_st_on_lt
            c.plot(t_lt, d_les, "b-", lw=2.4, label=r"LES $\Delta$ (ST+LT - ST)")

            t_common = t_base
            base_on_common = s_base

            all_on_common = interp_to(t_all, s_all, t_common)
            d_all = all_on_common - base_on_common
            c.plot(t_common, d_all, "m-", lw=2.4, label=r"Hybrid $\Delta$ (res)")

            nrs_on_common = interp_to(t_nrs, s_nrs, t_common)
            d_nrs = nrs_on_common - base_on_common
            c.plot(t_common, d_nrs, "m--", lw=2.4, label=r"Hybrid $\Delta$ (nores)")

            c.axhline(0.0, color="0.35", lw=1.0)
            c.set_xlim(*xlim)
            c.grid(True, alpha=0.25)

            # keep your style: x-label only on the last row of the WHOLE figure
            if r0 == 3:
                c.set_xlabel("day")

            if j == 0:
                c.set_ylabel(r"$\Delta$SST (°C)")
                c.legend(loc="best", frameon=False)
            else:
                c.tick_params(labelleft=False)

            row_vals[r0 + 2].extend([d_les, d_all, d_nrs])

    # ---- apply y-lims row-by-row (keep your original per-speed rules)
    # rows 0-2 -> cfg5, rows 3-5 -> cfg10
    cfg_by_row = {0: cfg5, 1: cfg5, 2: cfg5, 3: cfg10, 4: cfg10, 5: cfg10}
    which = {0: "YLIM_BASELINE", 1: "YLIM_LT", 2: "YLIM_DELTA", 3: "YLIM_BASELINE", 4: "YLIM_LT", 5: "YLIM_DELTA"}

    for r in range(6):
        cfg = cfg_by_row[r]
        key = which[r]
        ylim = cfg[key]
        if ylim is None:
            pad = 0.05 if ("DELTA" in key) else 0.03
            ylim = _auto_ylim(row_vals[r], pad=pad)
        if ylim is not None:
            for j in range(3):
                ax[r, j].set_ylim(*ylim)

    # ---- row tags (a)-(f): left panel each row
    for r in range(6):
        ax[r, 0].text(-0.22, 1.03, ROW_TAGS_6[r],
                      transform=ax[r, 0].transAxes,
                      ha="left", va="bottom", fontsize=14)

    # ---- col tags ONLY first row
    for j in range(3):
        ax[0, j].text(0.02, 1.03, COL_TAGS_3[j],
                      transform=ax[0, j].transAxes,
                      ha="left", va="bottom", fontsize=14)

    # layout
    fig.tight_layout(rect=[0, 0, 1, 0.985])

    if do_save:
        if save_png:
            fig.savefig(save_png, dpi=200, bbox_inches="tight")
            print("saved:", save_png)
        if save_eps:
            fig.savefig(save_eps, format="eps", bbox_inches="tight")
            print("saved:", save_eps)

    plt.show()


# ============================================================
# RUN
# ============================================================
sst_data_5 = build_sst_data(cfg_5,  y_points_r=Y_POINTS_R)
sst_data_10 = build_sst_data(cfg_10, y_points_r=Y_POINTS_R)

plot_SCM_Fig2_SST_6x3(
    sst_data_5, sst_data_10,
    cfg_5, cfg_10,
    xlim=XLIM,
    save_png=SAVE_PNG,
    save_eps=SAVE_EPS,
    do_save=DO_SAVE
)








In [ ]:
#this code is for figure 7:Time-depth evolution of temperature anomaly at the centerline location 
#$y_t$=0 km for one-dimensional idealized hurricane translating at 5 $\rm ms^{-1}$. 
#Black curves indicate mixed-layer depth (MLD) diagnosed 
#from the maximum vertical temperature gradient; solid and dash lines 
#correspond to the MLD of the subtractor and of the minuend, respectively. 
#The top row shows the temperature difference of different vertical schemes, 
#the middle row gives  "shear" results are obtained by using $\kappa$-shear; 
#Hybrid_res refers to the hybrid scheme with rescaling technique 
#while Hybrid_nores represents hybrid scheme without rescaling technique. 
#$\Delta$ refers to the temperature difference between ST+LT results and corresponding ST results.
#The bottom row shows differences of those LT increments to highlight the effect of rescaling.
#Panel (c) is obtained by subtracting two figures in panel (b): c(i)= b(ii)-b(i); c(ii)=b(iii)-b(i); 
#c(iii)=b(iii)-b(ii).


#!/usr/bin/env python3
# ============================================================
# Fig3 (single point): 9-panel time–depth temperature DIFF (fixed colorbar)
#  Row1:
#   (1) Hybrid_noLT        - LES_noLT
#   (2) shear             - Hybrid_noLT
#   (3) Hybrid_rescale_LT - LES_LT
#  Row2 (LT increments):
#   (4) ΔLES   = LES_LT   - LES_noLT
#   (5) Δnores = Hybrid_nores_LT - Hybrid_noLT
#   (6) Δres   = Hybrid_res_LT   - Hybrid_noLT
#  Row3 (increment errors / rescaling effect)  <-- NO MLD LINES
#   (7) Δnores - ΔLES
#   (8) Δres   - ΔLES
#   (9) Δres   - Δnores
#
# Updates requested:
#  - Panel tags: row tags (a)(b)(c); column tags (i)(ii)(iii) only on first row
#  - y label "Depth" -> "z (m)"
#  - Remove global MLD legend; instead add per-panel legend ONLY on panels with MLD lines
#  - Colorbars: NOT next to first column; put a colorbar axis on the FAR RIGHT of each row
#  - Title text updates (Row1 all 3; Row2 col2/col3 wording)
# ============================================================

import os
import numpy as np
import scipy.io as sio
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D

# optional colormap
try:
    import cmocean
    CMAP_DIFF = cmocean.cm.balance
except Exception:
    CMAP_DIFF = plt.get_cmap("RdBu_r")

# -------------------------------
# User settings
# -------------------------------
les_lt_root = "/archive/bgr/Datasets/LES/MoreHurr/TS05_ML10/LT/"
les_st_root = "/archive/bgr/Datasets/LES/MoreHurr/TS05_ML10/ST/"

Y_KM_ALL      = [300, 200, 150, 130, 110, 90, 70, 50, 30, 10, 0, -20, -40, -60, -80, -100, -120, -140, -200, -300]
Y_FOLDER_ALL  = ["300","200","150","130","110","90","70","50","30","10","0","S20","S40","S60","S80","S100","S120","S140","S200","S300"]

LES_LT_CASES  = ["021","031","036","038","040","042","044","046","048","050","051","053","055","057","059","061","063","065","071","081"]
LES_NLT_CASES = ["021","031","036","038","040","042","044","046","048","050","051","053","055","057","059","061","063","065","071","081"]

# pick ONE point for Fig3
Y_TARGET_KM = 30
RMAX_KM     = 50.0

# plot limits
TMAX_DAYS = 3.0
ZMAX_M    = 200.0

# common grid
NT_COMMON = 241
NZ_COMMON = 161
t_common  = np.linspace(0.0, TMAX_DAYS, NT_COMMON)
z_common  = np.linspace(0.0, ZMAX_M,    NZ_COMMON)

# fixed colorbar range
VMIN, VMAX = -0.5, 0.5
DLEV = 0.025
levels_fixed = np.arange(VMIN, VMAX + 0.5*DLEV, DLEV)

# MOM dirs (5 mps)
model_dirs = {
    "Hybrid_noLT"      : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/noLT_10m/{y}_results",
    "Hybrid_res_LT"    : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/waveLT_res_10m/{y}_results",
    "Hybrid_nores_LT"  : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/waveLT_nores_10m/{y}_results",
    "shear"            : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/shear_10m/{y}_results",
}


SAVE_EPS = f"/archive/Qian.Xiao/Qian.Xiao/eps/tempoSCM"

# -------------------------------
# Helpers
# -------------------------------
def _ensure_increasing_z(z, A):
    z = np.asarray(z).astype(float)
    if z.size >= 2 and z[1] < z[0]:
        z = z[::-1]
        A = A[..., ::-1]
    return z, A

def load_les_T(case_id, mode="LT"):
    if mode.upper() == "LT":
        root = les_lt_root
    elif mode.upper() in ["ST", "NLT", "NOLT", "NO_LT", "NO-LT"]:
        root = les_st_root
    else:
        raise ValueError(f"mode must be 'LT' or 'ST', got {mode}")

    f = os.path.join(root, case_id, "LESout.mat")
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing LES file: {f}")

    mat = sio.loadmat(f)
    t = np.asarray(mat["t"]).squeeze().astype("f8") / 86400.0
    z = np.asarray(mat["z"]).T.squeeze().astype("f8")
    T = np.asarray(mat["T"]).astype("f8") - 273.15
    del mat

    z, T = _ensure_increasing_z(z, T)
    m = (z >= 0) & (z <= ZMAX_M)
    return t, z[m], T[:, m]

def load_mom_T(results_dir):
    prog = os.path.join(results_dir, "prog.nc")
    if not os.path.exists(prog):
        raise FileNotFoundError(f"Missing: {prog}")

    with xr.open_dataset(prog) as ds:
        t = 300.0/86400.0 * (np.arange(ds.Time.size) + 0.5)

        if "temp" in ds.variables:
            da = ds["temp"]
        elif "thetao" in ds.variables:
            da = ds["thetao"]
        elif "Temp" in ds.variables:
            da = ds["Temp"]
        else:
            raise KeyError(f"No temperature var found in {prog}. Vars: {list(ds.variables)}")

        idx = {}
        if "xh" in da.dims: idx["xh"] = 0
        if "yh" in da.dims: idx["yh"] = 0
        T = da.isel(**idx).values  # (Time, zl)

        z = ds["zl"].values if "zl" in ds.variables else da["zl"].values

    z, T = _ensure_increasing_z(z, T)
    m = (z >= 0) & (z <= ZMAX_M)
    return t, z[m], T[:, m]

def mld_maxgrad(T, z):
    z = np.asarray(z).astype(float)
    T = np.asarray(T).astype(float)
    dTdz = np.gradient(T, z, axis=1)
    dTdz = np.nan_to_num(dTdz, nan=0.0, posinf=0.0, neginf=0.0)
    idx = np.argmax(np.abs(dTdz), axis=1)
    return z[idx]

def interp_T_to_common(t, z, T, t_new, z_new):
    t = np.asarray(t); z = np.asarray(z); T = np.asarray(T)
    Tz = np.full((len(t), len(z_new)), np.nan)
    for it in range(len(t)):
        Tz[it, :] = np.interp(z_new, z, T[it, :], left=np.nan, right=np.nan)

    out = np.full((len(t_new), len(z_new)), np.nan)
    for kz in range(len(z_new)):
        col = Tz[:, kz]
        ok = np.isfinite(col)
        if ok.sum() >= 2:
            out[:, kz] = np.interp(t_new, t[ok], col[ok], left=np.nan, right=np.nan)
    return out

def interp_line_to_common(t, line, t_new):
    t = np.asarray(t); line = np.asarray(line)
    ok = np.isfinite(line)
    if ok.sum() < 2:
        return np.full_like(t_new, np.nan, dtype=float)
    return np.interp(t_new, t[ok], line[ok], left=np.nan, right=np.nan)

def smooth1d_nan(x, win=9):
    x = np.asarray(x, dtype=float)
    if win <= 1:
        return x.copy()
    if win % 2 == 0:
        win += 1

    w = np.ones(win, dtype=float)
    ok = np.isfinite(x).astype(float)
    x0 = np.where(np.isfinite(x), x, 0.0)

    num = np.convolve(x0, w, mode="same")
    den = np.convolve(ok, w, mode="same")
    y = np.full_like(x, np.nan)
    m = den > 0
    y[m] = num[m] / den[m]
    return y

def truncate_cmap(cmap, minval=0.08, maxval=0.92, n=256):
    return mpl.colors.LinearSegmentedColormap.from_list(
        f"trunc_{getattr(cmap, 'name', 'cmap')}",
        cmap(np.linspace(minval, maxval, n))
    )

# -------------------------------
# Load single-point data
# -------------------------------
y_to_idx = {y: i for i, y in enumerate(Y_KM_ALL)}
if Y_TARGET_KM not in y_to_idx:
    raise ValueError(f"Y_TARGET_KM={Y_TARGET_KM} not in {Y_KM_ALL}")

i = y_to_idx[Y_TARGET_KM]
y_tag = Y_FOLDER_ALL[i]
yhat  = Y_TARGET_KM / RMAX_KM

# LES
t_les, z_les, T_les = load_les_T(LES_LT_CASES[i], mode="LT")
t_nlt, z_nlt, T_nlt = load_les_T(LES_NLT_CASES[i], mode="ST")

# MOM
mom = {k: load_mom_T(tpl.format(y=y_tag)) for k, tpl in model_dirs.items()}

# -------------------------------
# MLD (native -> t_common) + smoothing for Hybrid nores/res
# -------------------------------
mld = {}
mld["LES_noLT"] = interp_line_to_common(t_nlt, mld_maxgrad(T_nlt, z_nlt), t_common)
mld["LES_LT"]   = interp_line_to_common(t_les, mld_maxgrad(T_les, z_les), t_common)

for k in ["Hybrid_noLT", "Hybrid_nores_LT", "Hybrid_res_LT", "shear"]:
    tt, zz, TT = mom[k]
    mld[k] = interp_line_to_common(tt, mld_maxgrad(TT, zz), t_common)

mld["Hybrid_nores_LT"] = smooth1d_nan(mld["Hybrid_nores_LT"], win=9)
mld["Hybrid_res_LT"]   = smooth1d_nan(mld["Hybrid_res_LT"],   win=9)

# -------------------------------
# Interpolate temperature fields to common grid
# -------------------------------
Tc = {}
Tc["LES_noLT"] = interp_T_to_common(t_nlt, z_nlt, T_nlt, t_common, z_common)
Tc["LES_LT"]   = interp_T_to_common(t_les, z_les, T_les, t_common, z_common)

for k in ["Hybrid_noLT", "Hybrid_nores_LT", "Hybrid_res_LT", "shear"]:
    tt, zz, TT = mom[k]
    Tc[k] = interp_T_to_common(tt, zz, TT, t_common, z_common)

# Deltas
dLES   = Tc["LES_LT"] - Tc["LES_noLT"]
dNoRes = Tc["Hybrid_nores_LT"] - Tc["Hybrid_noLT"]
dRes   = Tc["Hybrid_res_LT"]   - Tc["Hybrid_noLT"]

# -------------------------------
# Panels (9) + TITLES UPDATED
# -------------------------------
panels = [
    # Row1
    dict(title="Hybrid_ST − LES_ST",
         field=Tc["Hybrid_noLT"] - Tc["LES_noLT"],
         mld_solid=mld["LES_noLT"], mld_dash=mld["Hybrid_noLT"], draw_mld=True,
         mld_label_solid="MLD LES_ST",
         mld_label_dash ="MLD Hybrid_ST"),

    dict(title="Shear − Hybrid_ST",
         field=Tc["shear"] - Tc["Hybrid_noLT"],
         mld_solid=mld["Hybrid_noLT"], mld_dash=mld["shear"], draw_mld=True,
         mld_label_solid="MLD Hybrid_ST",
         mld_label_dash ="MLD Shear"),

    dict(title="Hybrid_ST+LT (res) − LES_ST+LT",
         field=Tc["Hybrid_res_LT"] - Tc["LES_LT"],
         mld_solid=mld["LES_LT"], mld_dash=mld["Hybrid_res_LT"], draw_mld=True,
         mld_label_solid="MLD LES_ST+LT",
         mld_label_dash ="MLD Hybrid_ST+LT (res)"),

    # Row2
    dict(title="ΔLES = LES_ST+LT − LES_ST",
         field=dLES,
         mld_solid=mld["LES_noLT"], mld_dash=mld["LES_LT"], draw_mld=True,
         mld_label_solid="MLD LES_ST",
         mld_label_dash ="MLD LES_ST+LT"),

    dict(title="Hybrid_ST+LT (nores) − Hybrid_ST",
         field=dNoRes,
         mld_solid=mld["Hybrid_noLT"], mld_dash=mld["Hybrid_nores_LT"], draw_mld=True,
         mld_label_solid="MLD Hybrid_ST",
         mld_label_dash ="MLD Hybrid_ST+LT (nores)"),

    dict(title="Hybrid_ST+LT (res) − Hybrid_ST",
         field=dRes,
         mld_solid=mld["Hybrid_noLT"], mld_dash=mld["Hybrid_res_LT"], draw_mld=True,
         mld_label_solid="MLD Hybrid_ST",
         mld_label_dash ="MLD Hybrid_ST+LT (res)"),

    # Row3  (NO LINES)
    dict(title="Δnores − ΔLES", field=dNoRes - dLES, draw_mld=False),
    dict(title="Δres − ΔLES",   field=dRes - dLES,   draw_mld=False),
    dict(title="Δres − Δnores", field=dRes - dNoRes, draw_mld=False),
]


# ============================================================
# PLOT: 3×3 + colorbar column on the far right (per row)
# ============================================================

# fixed range for ALL panels
levels = np.linspace(VMIN, VMAX, 21)

# less saturated colormap + NaN->white
CMAP_PAPER = truncate_cmap(CMAP_DIFF, 0.08, 0.92)
CMAP_PAPER.set_bad("white")

# fixed norm
norm_fixed = mpl.colors.BoundaryNorm(levels, ncolors=CMAP_PAPER.N, clip=True)

fig = plt.figure(figsize=(15.6, 9.4))
fig.patch.set_facecolor("white")

gs = fig.add_gridspec(
    nrows=3, ncols=4,
    width_ratios=[1.0, 1.0, 1.0, 0.055],  # last col reserved for row colorbar
    wspace=0.18, hspace=0.3
)

# axes 3x3
ax = np.empty((3, 3), dtype=object)
for r in range(3):
    for c in range(3):
        if r == 0 and c == 0:
            ax[r, c] = fig.add_subplot(gs[r, c])
        else:
            ax[r, c] = fig.add_subplot(gs[r, c], sharex=ax[0, 0], sharey=ax[0, 0])
        ax[r, c].set_facecolor("white")

# colorbar axes (one per row, far right)
cax = [fig.add_subplot(gs[r, 3]) for r in range(3)]

# MLD legend handles (used per-axes)
handles_mld = [
    Line2D([0],[0], color="k", lw=2.2, ls="-",  label="MLD (solid)"),
    Line2D([0],[0], color="k", lw=2.2, ls="--", label="MLD (dashed)"),
]

mappables_row = [None, None, None]

for p, meta in enumerate(panels):
    r = p // 3
    c = p % 3
    a = ax[r, c]

    M = a.pcolormesh(
        t_common, z_common, meta["field"].T,
        shading="nearest",
        cmap=CMAP_PAPER,
        norm=norm_fixed,
        rasterized=True,
    )
    if mappables_row[r] is None:
        mappables_row[r] = M

    # MLD lines + PER-PANEL legend (only where lines exist)
    if meta.get("draw_mld", False):
        # halo (white)
        a.plot(t_common, meta["mld_solid"], color="w", lw=4.6, zorder=20)
        a.plot(t_common, meta["mld_dash"],  color="w", lw=4.6, ls="--", zorder=20)
        # black on top
        l1, = a.plot(t_common, meta["mld_solid"], color="k", lw=2.4, zorder=21)
        l2, = a.plot(t_common, meta["mld_dash"],  color="k", lw=2.4, ls="--", zorder=21)
    
        # per-panel legend with OWNERS
        a.legend(
            [l1, l2],
            [meta["mld_label_solid"], meta["mld_label_dash"]],
            loc="upper right",
            frameon=False,
            fontsize=9,
            handlelength=2.6
        )


    a.set_title(meta["title"], fontsize=11)
    a.set_xlim(1, TMAX_DAYS)
    a.set_ylim(ZMAX_M, 0)
    a.grid(True, alpha=0.18)

    # y-label only first column: "z (m)"
    if c == 0:
        a.set_ylabel(r"$z\ \mathrm{(m)}$")
    # x-label only bottom row
    if r == 2:
        a.set_xlabel("Time (days)")

# colorbars: one per row, on far right
for r in range(3):
    cb = fig.colorbar(mappables_row[r], cax=cax[r])
    cb.set_label(r"$\Delta \theta\ (^{\circ}\mathrm{C})$")
    cb.set_ticks([-0.5, -0.25, 0.0, 0.25, 0.5])

# Panel tags: row tags (a)(b)(c) on first column; col tags (i)(ii)(iii) only on first row
ROW_TAGS = ["(a)", "(b)", "(c)"]
COL_TAGS = ["(i)", "(ii)", "(iii)"]

for r in range(3):
    ax[r, 0].text(-0.22, 1.03, ROW_TAGS[r],
                  transform=ax[r, 0].transAxes,
                  ha="left", va="bottom", fontsize=14)

for c in range(3):
    ax[0, c].text(0.02, 1.03, COL_TAGS[c],
                  transform=ax[0, c].transAxes,
                  ha="left", va="bottom", fontsize=14)

# fig.suptitle(
#     f"Fig3 5mps MLD=10m: Time–depth temperature differences at y={Y_TARGET_KM:+.0f} km (y/Rmax={yhat:+.2f}); fixed colorbar [{VMIN},{VMAX}] °C",
#     y=1.02, fontsize=13
# )

# no global legend
fig.tight_layout()

# Save

fig.savefig(SAVE_EPS, format="eps", bbox_inches="tight")

print("saved:", SAVE_EPS)

plt.show()





In [ ]:
#this code is for figure 8:Mechanical energy $Me$ for different vertical mixing schemes 
#as a function of time at three different test locations $y_t$=[30,0,-40] km. 
#Left column compares $Me$ for the ST baseline, the middle column provides the ST+LT case
#(with a zoomed view near the peak) and the right column shows the LT-induced increment. 
#"Bias" is the normalized mean error defined by
#$\mathrm{Bias} =100*( \frac{1}{N}\sum_{i=1}^{N}\varepsilon(t_i))/S$, 
#where $N$ is the number of sample points, 
#$\varepsilon(t_i)=\Delta^{Hybrid}(t_i)-\Delta^{LES}(t_i)$ is the error in the LT-induced increment, 
#and $S = \max_{i=1,\dots,N}\left|\left(\Delta^{LES}(t_i)\right)\right|$ is 
#the maximum absolute value of $\Delta \mathrm{LES}(t_i)$.
#"RMSE" is the normalized root-mean-square error defined by
#$\mathrm{RMSE}= 100*\left(\sqrt{\frac{1}{N}\sum_{i=1}^{N}\varepsilon(t_i)^2}\right)/S$.


# ============================================================
# SCM (5 mps): Me(t) at 3 points, 3x3 panels + CLEAN inset zoom in Col2
# Col1: baseline (noLT)   -> LES_noLT vs Hybrid_noLT (+ shear + E_ePBL from hybrid_noLT)  [NO ePBLShear]
# Col2: absolute LT       -> LES_LT  vs Hybrid LT (rescale_all, norescale)  [inset only, NO lines/boxes]
# Col3: LT increment (Δ)  -> (LT-noLT) for LES and Hybrid (rescale_all, norescale)
# IMPORTANT: For ALL MOM cases (incl. shear), Hbl is from max |dT/dz| (prog.nc)
#
# Metrics (Col3 legend):
#   Bias and RMSE are NORMALIZED (%) by |min(ΔLES)| over METRIC_XLIM=(0,3) days
#   using intersection mask (LES + res + nores all valid) within that window.
#   Plot window can be different (XLIM) without affecting metrics.
# ============================================================

import os
import numpy as np
import scipy.io as sio
import xarray as xr
import matplotlib.pyplot as plt

# -------------------------------
# -------------------------------
# >>> CHANGED: LES roots (LT/ST) <<<
les_lt_root = "/archive/bgr/Datasets/LES/MoreHurr/TS05_ML10/LT/"
les_st_root = "/archive/bgr/Datasets/LES/MoreHurr/TS05_ML10/ST/" 

Y_KM_ALL      = [300, 200, 150, 130, 110, 90, 70, 50, 30, 10,0, -20, -40, -60, -80, -100, -120, -140, -200, -300]
Y_FOLDER_ALL  = ["300","200","150","130","110","90","70","50","30","10","0","S20","S40","S60","S80","S100","S120","S140","S200","S300"]

# Your case list (same ordering as Y_FOLDER_ALL)
LES_LT_CASES  = ["021","031","036","038","040","042","044","046","048","050", "051","053","055","057","059","061","063","065","071","081"]
LES_NLT_CASES = ["021","031","036","038","040","042","044","046","048","050", "051","053","055","057","059","061","063","065","071","081"]

Y_POINTS_KM = (30, 0, -40)
RMAX_KM = 50.0

rho0  = 1026.95
alpha = -0.2
g     = 9.81

model_dirs = {
    "hybrid_noLT"           : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/MIX1_noLT/{y}_results",
    "hybrid_LT_rescale_all" : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/MIX1p8_waveLT_all/{y}_results",
    "hybrid_LT_norescale"   : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/MIX1p8_waveLT_norescale/{y}_results",
    "shear"                 : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/shear/{y}_results",
}

model_dirs = {
    "hybrid_noLT"            : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/noLT_10m/{y}_results",
    "hybrid_LT_rescale_all"      : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/waveLT_res_10m/{y}_results",       # rescale_all
    "hybrid_LT_norescale": "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/waveLT_nores_10m/{y}_results", # norescale
    "shear"                           : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/shear_10m/{y}_results",
  }
# plot x-window
XLIM = (1, 2.5)

# metrics x-window (fixed 0-3 as you asked)
METRIC_XLIM = (0.0, 3.0)


SAVE_EPS = "/archive/Qian.Xiao/Qian.Xiao/eps/Me3p"

# ---- inset placement (axes fraction) ----
INSET_RECT_DEFAULT = [0.62, 0.08, 0.38, 0.34]   # [x0, y0, w, h]
INSET_RECT_ROW0    = [0.64, 0.12, 0.30, 0.28]   # row0,col2: more right & smaller

# ---- inset window fixed ----
INSET_XWIN = (1.25, 1.75)
INSET_YTOP = -2.0e-4  # inset y-axis top limit (NOT from 0)

# y-limit padding per column
YLIM_PAD_FRAC = 0.10

# ---- make middle column wider + a bit more room for last-col legend ----
WIDTH_RATIOS = (1.00, 1.35, 1.12)  # col2 wider, col3 slightly wider

# ============================================================
# helpers
# ============================================================
def interp_to(t_src, y_src, t_tgt):
    t_src = np.asarray(t_src); y_src = np.asarray(y_src); t_tgt = np.asarray(t_tgt)
    if t_src.size == 0:
        return np.full_like(t_tgt, np.nan, dtype=float)
    if np.any(np.diff(t_src) < 0):
        order = np.argsort(t_src)
        t_src = t_src[order]; y_src = y_src[order]
    return np.interp(t_tgt, t_src, y_src, left=y_src[0], right=y_src[-1])

def _isel0(da):
    indexers = {}
    for d in da.dims:
        if d in ("Time", "zl", "zi"):
            continue
        indexers[d] = 0
    return da.isel(**indexers) if indexers else da

def _time_days(ds):
    return 300.0/86400.0 * (np.arange(ds.Time.size) + 0.5)

def _find_var(ds, candidates):
    for v in candidates:
        if v in ds.variables:
            return v
    lowmap = {k.lower(): k for k in ds.variables.keys()}
    for v in candidates:
        if v.lower() in lowmap:
            return lowmap[v.lower()]
    return None

def _to_1d_timeseries(da):
    """
    Robust: collapse horizontal dims -> 1D time series.
    If has zl/zi, integrate over that axis (trapz).
    """
    da0 = _isel0(da)
    if "Time" not in da0.dims:
        return np.asarray(da0.squeeze().values)

    arr = np.asarray(da0.values)
    if "zl" in da0.dims:
        z = np.asarray(da0["zl"].values)
        ax = da0.dims.index("zl")
        arr = np.trapezoid(arr, z, axis=ax)
    elif "zi" in da0.dims:
        z = np.asarray(da0["zi"].values)
        ax = da0.dims.index("zi")
        arr = np.trapezoid(arr, z, axis=ax)
    return np.asarray(arr).squeeze()

# =======================
# >>> CHANGED: LES roots + loader (ONLY LES-related change) <<<
# =======================


def load_les_min(case_id, mode="LT"):
    """
    Read LESout.mat from:
      LT:  les_lt_root/<case_id>/LESout.mat
      ST:  les_st_root/<case_id>/LESout.mat
    Expected variables in LESout.mat:
      t      : seconds
      z      : meters (often column vector)
      T      : (Nt, Nz) Kelvin
      tw     : (Nt, Nz_iface) or (Nt, Nz) (as in your earlier convention)
    Convention kept:
      tw = -mat["tw"]  (your convention)
    """
    if mode.upper() == "LT":
        root = les_lt_root
    elif mode.upper() in ["ST", "NLT", "NOLT", "NO_LT", "NO-LT"]:
        root = les_st_root
    else:
        raise ValueError(f"mode must be 'LT' or 'ST', got {mode}")

    f = os.path.join(root, case_id, "LESout.mat")
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing LES file: {f}")

    mat = sio.loadmat(f)
    t_days = np.asarray(mat["t"]).squeeze().astype("f8") / 86400.0
    z = np.asarray(mat["z"]).T.squeeze().astype("f8")
    T = np.asarray(mat["T"]).astype("f8") - 273.15
    tw = -np.asarray(mat["tw"]).astype("f8")  # your convention
    del mat
    return t_days, z, T, tw

def compute_Hbl_maxgrad(T, z):
    dTdz = np.gradient(T, z, axis=1)
    idx = np.argmax(np.abs(dTdz), axis=1)
    return z[idx]

def compute_Me_from_tw(z_iface, tw_iface, Hbl):
    bflux = -(g / rho0) * alpha * tw_iface
    Me = np.full((bflux.shape[0],), np.nan)
    for it in range(bflux.shape[0]):
        H = Hbl[it]
        if not np.isfinite(H) or H <= z_iface[0]:
            continue
        mask = (z_iface <= H)
        zz = z_iface[mask]
        bf = bflux[it, mask]
        neg = bf < 0
        Me[it] = np.trapezoid(bf[neg], zz[neg]) if np.any(neg) else 0.0
    return Me

def load_mom_Me_from_prog_visc(results_dir):
    prog = os.path.join(results_dir, "prog.nc")
    visc = os.path.join(results_dir, "visc.nc")
    if not os.path.exists(prog):
        raise FileNotFoundError(f"Missing file: {prog}")
    if not os.path.exists(visc):
        raise FileNotFoundError(f"Missing file: {visc}")

    with xr.open_dataset(prog) as ds:
        t_prog = _time_days(ds)
        if "temp" in ds.variables:
            temp = _isel0(ds["temp"]).values
        elif "thetao" in ds.variables:
            temp = _isel0(ds["thetao"]).values
        else:
            raise KeyError(f"No temp/thetao in {prog}. Vars: {list(ds.variables)}")
        zl = ds["zl"].values
    Hbl_prog = compute_Hbl_maxgrad(temp, zl)

    with xr.open_dataset(visc) as ds:
        t_visc = _time_days(ds)
        zi = ds["zi"].values
        if "Tflx_dia_diff" not in ds.variables:
            raise KeyError(f"Tflx_dia_diff not found in {visc}. Vars: {list(ds.variables)}")
        tw_zi = -_isel0(ds["Tflx_dia_diff"]).values

    Hbl = interp_to(t_prog, Hbl_prog, t_visc)
    Me = compute_Me_from_tw(zi, tw_zi, Hbl)
    return t_visc, Me

# -------------------------------
# load E_ePBL from hybrid_noLT directory
# -------------------------------
CANDIDATES_EEPBL = ["E_ePBL", "E_epbl", "E_EPBL", "Eepbl", "E_ePBL_dia", "E_ePBL_diag", "E_ePBL_diagnostic"]

def load_mom_EePBL(results_dir):
    """
    Find E_ePBL in visc.nc/prog.nc, return (t_days, E_ePBL_1d).
    """
    for fname in ("visc.nc", "prog.nc"):
        fpath = os.path.join(results_dir, fname)
        if not os.path.exists(fpath):
            continue
        with xr.open_dataset(fpath) as ds:
            v = _find_var(ds, CANDIDATES_EEPBL)
            if v is None:
                continue
            t = _time_days(ds)
            y = _to_1d_timeseries(ds[v])
            return t, y
    raise KeyError(f"Cannot find E_ePBL in visc.nc/prog.nc under {results_dir}. Vars tried: {CANDIDATES_EEPBL}")

def build_me_data_5mps(y_points_km):
    y_to_idx = {y: i for i, y in enumerate(Y_KM_ALL)}
    out = []
    for y in y_points_km:
        if y not in y_to_idx:
            raise ValueError(f"y_km={y} not in {Y_KM_ALL}")
        i = y_to_idx[y]
        y_tag = Y_FOLDER_ALL[i]

        # LES (CHANGED calls only)
        t_lt, z_lt, T_lt, tw_lt = load_les_min(LES_LT_CASES[i], mode="LT")
        t_nl, z_nl, T_nl, tw_nl = load_les_min(LES_NLT_CASES[i], mode="ST")

        H_lt = compute_Hbl_maxgrad(T_lt, z_lt)
        H_nl = compute_Hbl_maxgrad(T_nl, z_nl)

        Me_lt = compute_Me_from_tw(z_lt, tw_lt, H_lt)
        Me_nl = compute_Me_from_tw(z_nl, tw_nl, H_nl)

        # MOM (prog+visc)
        mom = {}
        for key, tpl in model_dirs.items():
            base = tpl.format(y=y_tag)
            mom[key] = load_mom_Me_from_prog_visc(base)

        # EePBL from hybrid_noLT only
        base_h0 = model_dirs["hybrid_noLT"].format(y=y_tag)
        t_eepbl, eepbl = load_mom_EePBL(base_h0)

        out.append({
            "y_km": float(y),
            "LES_noLT": (t_nl, Me_nl),
            "LES_LT": (t_lt, Me_lt),
            "MOM": mom,
            "E_ePBL_noLT": (t_eepbl, -eepbl),   # plot -E_ePBL (your convention)
        })
    return out

def _compute_col_ylims(me_data):
    col_series = {0: [], 1: [], 2: []}
    for rec in me_data:
        t_nl, Me_nl = rec["LES_noLT"]
        t_lt, Me_lt = rec["LES_LT"]
        mom = rec["MOM"]

        col_series[0].append(Me_nl)
        col_series[0].append(mom["hybrid_noLT"][1])
        col_series[0].append(mom["shear"][1])
        col_series[0].append(rec["E_ePBL_noLT"][1])

        col_series[1].append(Me_lt)
        col_series[1].append(mom["hybrid_LT_rescale_all"][1])
        col_series[1].append(mom["hybrid_LT_norescale"][1])

        Me_nl_on_lt = interp_to(t_nl, Me_nl, t_lt)
        d_les = Me_lt - Me_nl_on_lt

        t_h0, Me_h0 = mom["hybrid_noLT"]
        t_ra, Me_ra = mom["hybrid_LT_rescale_all"]
        t_nr, Me_nr = mom["hybrid_LT_norescale"]
        d_ra = interp_to(t_ra, Me_ra, t_h0) - Me_h0
        d_nr = interp_to(t_nr, Me_nr, t_h0) - Me_h0

        col_series[2].append(d_les)
        col_series[2].append(d_ra)
        col_series[2].append(d_nr)

    ylims = {}
    for c in (0, 1, 2):
        yy = np.concatenate([np.asarray(v).ravel() for v in col_series[c]])
        yy = yy[np.isfinite(yy)]
        if yy.size == 0:
            ylims[c] = None
            continue
        ymin, ymax = yy.min(), yy.max()
        dy = (ymax - ymin) if ymax > ymin else (abs(ymax) * 0.1 + 1e-12)
        pad = YLIM_PAD_FRAC * dy
        ylims[c] = (ymin - pad, ymax + 1.2*pad)
    return ylims

def _bias_rmse(pred, ref, mask):
    pred = np.asarray(pred); ref = np.asarray(ref)
    d = pred[mask] - ref[mask]
    d = d[np.isfinite(d)]
    if d.size == 0:
        return np.nan, np.nan
    bias = np.mean(d)
    rmse = np.sqrt(np.mean(d**2))
    return bias, rmse

def _metrics_bias_rmse_percent_A(d_res, d_nores, d_les_ref,
                                t_common, metric_xlim,
                                t_lt, t_nl, t_ra, t_nr):
    """
    Scheme A:
      scale = |min(ΔLES)| over intersection mask within metric_xlim
      Bias% = Bias/scale*100, RMSE% = RMSE/scale*100
    """
    x1, x2 = metric_xlim
    t_common = np.asarray(t_common)

    mask = (
        (t_common >= x1) & (t_common <= x2) &
        (t_common >= np.nanmin(t_lt)) & (t_common <= np.nanmax(t_lt)) &
        (t_common >= np.nanmin(t_nl)) & (t_common <= np.nanmax(t_nl)) &
        (t_common >= np.nanmin(t_ra)) & (t_common <= np.nanmax(t_ra)) &
        (t_common >= np.nanmin(t_nr)) & (t_common <= np.nanmax(t_nr))
    )

    dref_win = np.asarray(d_les_ref)[mask]
    dref_win = dref_win[np.isfinite(dref_win)]
    if dref_win.size == 0:
        return (np.nan, np.nan, np.nan, np.nan)

    scale = abs(np.nanmin(dref_win))
    if (not np.isfinite(scale)) or (scale <= 0):
        return (np.nan, np.nan, np.nan, np.nan)

    bias_res, rmse_res = _bias_rmse(d_res, d_les_ref, mask)
    bias_nr,  rmse_nr  = _bias_rmse(d_nores, d_les_ref, mask)

    return (100.0*bias_res/scale, 100.0*rmse_res/scale,
            100.0*bias_nr/scale,  100.0*rmse_nr/scale)

def _legend_lower_right(ax, fontsize=10):
    ax.legend(
        loc="lower right",
        bbox_to_anchor=(0.98, 0.02),
        bbox_transform=ax.transAxes,
        frameon=False,
        fontsize=fontsize,
        handlelength=2.0,
        borderaxespad=0.0,
        labelspacing=0.30,
    )

def _legend_lower_center(ax, fontsize=10):
    ax.legend(
        loc="lower center",
        bbox_to_anchor=(0.75, 0.4),
        bbox_transform=ax.transAxes,
        frameon=False,
        fontsize=fontsize,
        handlelength=2.0,
        borderaxespad=0.0,
        labelspacing=0.30,
    )

# ============================================================
# Inset
# ============================================================
def add_clean_zoom_inset(ax_main, xwin,
                         t_les, y_les,
                         t_ra,  y_ra,
                         t_nr,  y_nr,
                         rect=None,
                         ytick_right=False,
                         show_y_ticks=True,
                         y_top_fixed=-4e-4):
    t_ref = np.asarray(t_ra)
    y_ra  = np.asarray(y_ra)

    nr_on_ref  = interp_to(t_nr,  y_nr,  t_ref)
    les_on_ref = interp_to(t_les, y_les, t_ref)

    t1, t2 = xwin
    win = (t_ref >= t1) & (t_ref <= t2)

    yy = np.concatenate([y_ra[win], nr_on_ref[win], les_on_ref[win]])
    yy = yy[np.isfinite(yy)]
    if yy.size < 5:
        return

    ymin = np.nanmin(yy)
    pad = 0.06 * max(abs(y_top_fixed - ymin), 1e-12)
    ylow  = ymin - pad
    yhigh = y_top_fixed

    if rect is None:
        rect = INSET_RECT_DEFAULT

    axins = ax_main.inset_axes(rect)
    axins.plot(t_ref, les_on_ref, "b-",  lw=1.15)
    axins.plot(t_ref, y_ra,       "m-",  lw=1.20)
    axins.plot(t_ref, nr_on_ref,  "m--", lw=1.20)

    axins.set_xlim(t1, t2)
    axins.set_ylim(ylow, yhigh)
    axins.grid(True, alpha=0.12)
    axins.tick_params(labelsize=8)

    axins.set_facecolor("white")
    axins.patch.set_alpha(1.0)

    if not show_y_ticks:
        axins.tick_params(labelleft=False, labelright=False)
        axins.set_yticks([])
    else:
        if ytick_right:
            axins.yaxis.tick_right()
            axins.tick_params(labelleft=False, labelright=True)
        else:
            axins.yaxis.tick_left()
            axins.tick_params(labelleft=True, labelright=False)

# ============================================================
# Plot
# ============================================================
def plot_SCM_Fig4_Me_3x3(me_data, xlim=(0,3), metric_xlim=(0,3), save_png=None, save_eps=None):
    fig = plt.figure(figsize=(15.2, 11.4))
    gs = fig.add_gridspec(3, 3, width_ratios=WIDTH_RATIOS, wspace=0.18, hspace=0.30)

    ax = np.empty((3,3), dtype=object)
    for r in range(3):
        for c in range(3):
            sharex = ax[0, c] if (r > 0) else None
            ax[r, c] = fig.add_subplot(gs[r, c], sharex=sharex)

    col_ylims = _compute_col_ylims(me_data)

    for r, rec in enumerate(me_data):
        y_km = rec["y_km"]
        yhat = y_km / RMAX_KM   # = (y - y_c)/RMW  (y_c=0, RMW=RMAX_KM)

        t_nl, Me_nl = rec["LES_noLT"]
        t_lt, Me_lt = rec["LES_LT"]
        mom = rec["MOM"]
        t_eepbl, eepbl = rec["E_ePBL_noLT"]

        # ---- Col 1: baseline (+ E_ePBL)
        a = ax[r, 0]
        a.plot(t_nl, Me_nl, "b--", lw=1.6, label="LES ST")
        t_h0, Me_h0 = mom["hybrid_noLT"]
        a.plot(t_h0, Me_h0, "k-", lw=1.8, label="Hybrid ST")
        t_sh, Me_sh = mom["shear"]
        a.plot(t_sh, Me_sh, ls="--", lw=1.6, label="Shear")  # CHANGED: Shear
        a.plot(t_eepbl, eepbl, color="tab:orange", lw=1.8, label="RH18")  # CHANGED: not italic

        # CHANGED: title text
        ax[r, 1].set_title(rf"$(y-y_c)/RMW$={yhat:+.2f}  (y={y_km:+.0f} km)", pad=14)

        a.set_xlim(*xlim)
        a.grid(True, alpha=0.22)
        if r == 2:
            a.set_xlabel("day")
        if r == 1:
            a.set_ylabel(r"$Me_{i}$  $\rm m^3s^{-3}$")
        if r == 0:
            _legend_lower_right(a, fontsize=10)
        if col_ylims[0] is not None:
            a.set_ylim(*col_ylims[0])

        # ---- Col 2: absolute LT  (NO ylabel, NO main y tick labels)
        b = ax[r, 1]
        b.plot(t_lt, Me_lt, "b-", lw=1.7, label="LES ST+LT")
        t_ra, Me_ra = mom["hybrid_LT_rescale_all"]
        b.plot(t_ra, Me_ra, "m-", lw=1.7, label="Hybrid ST+LT (res)")
        t_nr, Me_nr = mom["hybrid_LT_norescale"]
        b.plot(t_nr, Me_nr, "m--", lw=1.7, label="Hybrid ST+LT (nores)")

        b.set_xlim(*xlim)
        b.grid(True, alpha=0.22)
        if r == 2:
            b.set_xlabel("day")
        b.set_ylabel("")
        b.tick_params(labelleft=False, labelright=False)

        if r == 0:
            _legend_lower_center(b, fontsize=10)

        rect = INSET_RECT_ROW0 if r == 0 else INSET_RECT_DEFAULT
        add_clean_zoom_inset(
            b, INSET_XWIN,
            t_lt, Me_lt,
            t_ra, Me_ra,
            t_nr, Me_nr,
            rect=rect,
            show_y_ticks=(r != 0),
            ytick_right=False,
            y_top_fixed=INSET_YTOP
        )
        if col_ylims[1] is not None:
            b.set_ylim(*col_ylims[1])

        # ---- Col 3: delta LT  (+ Bias/RMSE in legend)
        c = ax[r, 2]

        LW_DELTA = 1.25
        HALO_LW_EXTRA = 2.6
        HALO_ALPHA = 0.98

        # LES Δ on LES time (for plotting)
        Me_nl_on_lt = interp_to(t_nl, Me_nl, t_lt)
        d_les_lt = Me_lt - Me_nl_on_lt
        c.plot(t_lt, d_les_lt, "b-", lw=LW_DELTA, label=r"$\Delta$ LES")  # CHANGED

        # Hybrid Δ on t_common = t_h0 (for plotting & metrics)
        t_common = t_h0
        d_res = interp_to(t_ra, Me_ra, t_common) - Me_h0
        d_nores = interp_to(t_nr, Me_nr, t_common) - Me_h0

        # LES Δ on t_common (for metrics reference)
        les_lt_on_common = interp_to(t_lt, Me_lt, t_common)
        les_nl_on_common = interp_to(t_nl, Me_nl, t_common)
        d_les_common = les_lt_on_common - les_nl_on_common

        bias_res_pct, rmse_res_pct, bias_nr_pct, rmse_nr_pct = _metrics_bias_rmse_percent_A(
            d_res=d_res, d_nores=d_nores, d_les_ref=d_les_common,
            t_common=t_common, metric_xlim=metric_xlim,
            t_lt=t_lt, t_nl=t_nl, t_ra=t_ra, t_nr=t_nr
        )

        # CHANGED: legend names in last column
        lab_res = rf"$\Delta$ Hybrid (res)"   f"\nBias={bias_res_pct:+.1f}%, RMSE={rmse_res_pct:.1f}%"
        lab_nr  = rf"$\Delta$ Hybrid (nores)" f"\nBias={bias_nr_pct:+.1f}%, RMSE={rmse_nr_pct:.1f}%"

        c.plot(t_common, d_res, "m-", lw=LW_DELTA, label=lab_res)

        c.plot(t_common, d_nores, color="w", lw=LW_DELTA + HALO_LW_EXTRA, ls="--",
               alpha=HALO_ALPHA, zorder=9, label="_nolegend_")
        c.plot(t_common, d_nores, "m--", lw=LW_DELTA, zorder=10, label=lab_nr)

        c.axhline(0.0, color="0.35", lw=1.0)
        c.set_xlim(*xlim)
        c.grid(True, alpha=0.22)
        if r == 2:
            c.set_xlabel("day")

        c.legend(
            loc="lower right",
            bbox_to_anchor=(0.98, 0.02),
            frameon=False,
            fontsize=8.5,
            handlelength=2.2,
            borderaxespad=0.0,
            labelspacing=0.35
        )

        if col_ylims[2] is not None:
            c.set_ylim(*col_ylims[2])

    # CHANGED: add row/col tags (only first row has col tags)
    ROW_TAGS = ["(a)", "(b)", "(c)"]
    COL_TAGS = ["(i)", "(ii)", "(iii)"]

    for rr in range(3):
        ax[rr, 0].text(-0.22, 1.03, ROW_TAGS[rr],
                       transform=ax[rr, 0].transAxes,
                       ha="left", va="bottom", fontsize=14)

    for cc in range(3):
        ax[0, cc].text(0.02, 1.03, COL_TAGS[cc],
                       transform=ax[0, cc].transAxes,
                       ha="left", va="bottom", fontsize=14)

    # fig.suptitle("SCM Fig4 (5 mps MLD=10m): $M_e(t)$ baseline, absolute ST+LT (clean zoom), and LT increment (Δ) at 3 points", y=0.99)

    # move middle column left (keep your tweak)
    MID_COL_SHIFT = 0.025
    for r in range(3):
        pos = ax[r, 1].get_position()
        ax[r, 1].set_position([pos.x0 - MID_COL_SHIFT, pos.y0, pos.width, pos.height])

    if save_eps:
        fig.savefig(save_eps, format="eps", bbox_inches="tight")
        print("saved:", save_eps)

    plt.show()


# ============================================================
# RUN
# ============================================================
me_data_5mps = build_me_data_5mps(Y_POINTS_KM)

plot_SCM_Fig4_Me_3x3(
    me_data_5mps,
    xlim=XLIM,                 # plot x-range
    metric_xlim=METRIC_XLIM,   # metrics x-range (0-3)

    save_eps=SAVE_EPS
)








In [ ]:
#this code is for figure 9:The rescaling factor $\mathcal{R}$ and
#LT enhancement factor $m^{\rm LT}_{*} $ as a function of time for 5 $\rm ms^{-1}$ 
#at three different locations. The gray line $y=1$ in the left column indicates 
#the no-rescaling reference $m^*_{LT,nores}$ (i.e. $m^*_{LT,nores}/m^*_{LT,RL19}=1$).

#!/usr/bin/env python3
# ============================================================
# Fig5 (standalone): Rescaling LT verification at 3 points
# Col1: rescale_LES = mstar_LT_LES / MSTAR_LT(nores,read)  vs  LT_top_proxy
# Col2: mstar_LT_LES vs mstar_LT_nores(diagnose) vs mstar_LT_rescale_all(diagnose)
#
# Updates requested:
#  - Add row/col panel tags: row tags (a)(b)(c); col tags (i)(ii) ONLY on first row
#  - y-label Col1: "Rescaling factor \mathcal{R}"
#  - y-label Col2: "m^{LT}_{*}" (with non-italic subscripts)
#  - Titles: replace "y/Rmax=..." with "(y-y_c)/RMW=..."
#  - Legends: rename all m*_LT... to m^{LT}_{*,LES}, m^{LT}_{*,nores}, m^{LT}_{*,res}, etc.
#             AND make ALL subscripts non-italic (use \mathrm{...})
# ============================================================

import os
import numpy as np
import scipy.io as sio
import xarray as xr
import matplotlib.pyplot as plt

# -------------------------------
# SETTINGS
# -------------------------------
les_lt_root = "/archive/bgr/Datasets/LES/MoreHurr/TS05_ML10/LT/"
les_st_root = "/archive/bgr/Datasets/LES/MoreHurr/TS05_ML10/ST/"

Y_KM_ALL      = [300, 200, 150, 130, 110, 90, 70, 50, 30, 10, 0, -20, -40, -60, -80, -100, -120, -140, -200, -300]
Y_FOLDER_ALL  = ["300","200","150","130","110","90","70","50","30","10","0","S20","S40","S60","S80","S100","S120","S140","S200","S300"]

LES_LT_CASES  = ["021","031","036","038","040","042","044","046","048","050","051","053","055","057","059","061","063","065","071","081"]
LES_NLT_CASES = ["021","031","036","038","040","042","044","046","048","050","051","053","055","057","059","061","063","065","071","081"]

Y_POINTS_KM = (30, 0, -40)

# For panel title normalization
RMW_KM = 50.0  # RMW = 50 km

rho0  = 1026.95
alpha = -0.2
g     = 9.81
EPS   = 1e-12

model_dirs = {
    "Hybrid_noLT"     : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/noLT_10m/{y}_results",
    "Hybrid_res_LT"   : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/waveLT_res_10m/{y}_results",       # rescale_all
    "Hybrid_nores_LT" : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/waveLT_nores_10m/{y}_results",     # norescale
    "shear"           : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/shear_10m/{y}_results",
}

XLIM = (0.8, 2.0)

SAVE_EPS = "/archive/Qian.Xiao/Qian.Xiao/eps/RescalingRatio"

# -------------------------------
# HELPERS
# -------------------------------
def interp_to(t_src, y_src, t_tgt):
    t_src = np.asarray(t_src).squeeze()
    y_src = np.asarray(y_src).squeeze()
    t_tgt = np.asarray(t_tgt).squeeze()
    m = np.isfinite(t_src) & np.isfinite(y_src)
    if m.sum() < 2:
        return np.full_like(t_tgt, np.nan, dtype=float)
    order = np.argsort(t_src[m])
    return np.interp(t_tgt, t_src[m][order], y_src[m][order], left=np.nan, right=np.nan)

def isel_scm(ds):
    indexers = {}
    for d in ["xh","yh","xq","yq"]:
        if d in ds.dims:
            indexers[d] = 0
    return ds.isel(**indexers) if indexers else ds

def compute_Hbl_maxgrad(T, z):
    dTdz = np.gradient(T, z, axis=1)
    idx = np.argmax(np.abs(dTdz), axis=1)
    return z[idx]

def compute_Me_from_tw(z_iface, tw_iface, Hbl):
    bflux = -(g / rho0) * alpha * tw_iface
    Me = np.full((bflux.shape[0],), np.nan)
    for it in range(bflux.shape[0]):
        H = Hbl[it]
        if (not np.isfinite(H)) or (H <= z_iface[0]):
            continue
        mask = (z_iface <= H)
        zz = z_iface[mask]
        bf = bflux[it, mask]
        neg = bf < 0
        Me[it] = np.trapezoid(bf[neg], zz[neg]) if np.any(neg) else 0.0
    return Me

def load_les_mstar(case_id, mode="LT"):
    if mode.upper() == "LT":
        root = les_lt_root
    elif mode.upper() in ["ST", "NLT", "NOLT", "NO_LT", "NO-LT"]:
        root = les_st_root
    else:
        raise ValueError(f"mode must be 'LT' or 'ST', got {mode}")

    f = os.path.join(root, case_id, "LESout.mat")
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing LES file: {f}")

    mat = sio.loadmat(f)

    t = np.asarray(mat["t"]).squeeze().astype("f8") / 86400.0
    z = np.asarray(mat["z"]).T.squeeze().astype("f8")
    T = np.asarray(mat["T"]).astype("f8") - 273.15
    tw = -np.asarray(mat["tw"]).astype("f8")

    tau13 = np.asarray(mat["tau13l"]).astype("f8")[:, 0] * rho0
    tau23 = np.asarray(mat["tau23l"]).astype("f8")[:, 0] * rho0
    tau = np.hypot(tau13, tau23)

    ustar = np.sqrt(np.maximum(tau, 0.0) / rho0)

    Hbl = compute_Hbl_maxgrad(T, z)
    Me  = compute_Me_from_tw(z, tw, Hbl)

    mstar = np.abs(Me) / (np.abs(ustar)**3 + EPS)
    return t, mstar

def load_mom_mstar_diag(results_dir):
    prog = os.path.join(results_dir, "prog.nc")
    visc = os.path.join(results_dir, "visc.nc")
    sf   = os.path.join(results_dir, "surffluxes.nc")

    for f in [prog, visc, sf]:
        if not os.path.exists(f):
            raise FileNotFoundError(f"Missing {f}")

    with xr.open_dataset(prog) as ds:
        ds = isel_scm(ds)
        nt = ds.Time.size
        t_prog = 300.0/86400.0 * (np.arange(nt) + 0.5)
        if "temp" in ds.variables:
            T = ds["temp"].values
        elif "thetao" in ds.variables:
            T = ds["thetao"].values
        else:
            raise KeyError(f"No temp/thetao in {prog}.")
        zl = ds["zl"].values
    Hbl_prog = compute_Hbl_maxgrad(T, zl)

    with xr.open_dataset(visc) as ds:
        ds = isel_scm(ds)
        nt = ds.Time.size
        t_visc = 300.0/86400.0 * (np.arange(nt) + 0.5)
        zi = ds["zi"].values
        if "Tflx_dia_diff" not in ds.variables:
            raise KeyError(f"Tflx_dia_diff not found in {visc}.")
        tw_zi = -ds["Tflx_dia_diff"].values

    Hbl = interp_to(t_prog, Hbl_prog, t_visc)
    Me  = compute_Me_from_tw(zi, tw_zi, Hbl)

    with xr.open_dataset(sf) as ds:
        ds = isel_scm(ds)
        nt = ds.Time.size
        t_sf = 300.0/86400.0 * (np.arange(nt) + 0.5)
        tx = ds["taux"].values
        ty = ds["tauy"].values
    tau = np.hypot(tx, ty)
    ustar_sf = np.sqrt(np.maximum(tau, 0.0) / rho0)
    ustar = interp_to(t_sf, ustar_sf, t_visc)

    mstar = np.abs(Me) / (np.abs(ustar)**3 + EPS)
    return t_visc, mstar

def load_mom_MSTAR_LT_read(results_dir):
    visc = os.path.join(results_dir, "visc.nc")
    if not os.path.exists(visc):
        raise FileNotFoundError(f"Missing {visc}")
    with xr.open_dataset(visc) as ds:
        ds = isel_scm(ds)
        nt = ds.Time.size
        t = 300.0/86400.0 * (np.arange(nt) + 0.5)
        if "MSTAR_LT" not in ds.variables:
            raise KeyError(f"MSTAR_LT not found in {visc}")
        mstar_lt = ds["MSTAR_LT"].values
    return t, np.asarray(mstar_lt).squeeze()

# -------------------------------
# PLOT
# -------------------------------
fig, ax = plt.subplots(3, 2, figsize=(12.8, 9.4), sharex=True)

# Panel tags
ROW_TAGS = ["(a)", "(b)", "(c)"]
COL_TAGS = ["(i)", "(ii)"]

y_to_idx = {float(y): i for i, y in enumerate(Y_KM_ALL)}

for r, y_km in enumerate(Y_POINTS_KM):
    y_km = float(y_km)
    if y_km not in y_to_idx:
        raise ValueError(f"Requested y_km={y_km} not in Y_KM_ALL={Y_KM_ALL}")
    iloc = y_to_idx[y_km]
    y_tag = Y_FOLDER_ALL[iloc]

    # normalized coordinate for title
    y_norm_rmw = y_km / RMW_KM

    # --- LES ---
    t_les_lt, mstar_les_lt = load_les_mstar(LES_LT_CASES[iloc], mode="LT")
    t_les_nl, mstar_les_nl = load_les_mstar(LES_NLT_CASES[iloc], mode="ST")

    # use MOM norescale read-time as common axis
    base_nores = model_dirs["Hybrid_nores_LT"].format(y=y_tag)
    t_param, MSTAR_LT_nores_read = load_mom_MSTAR_LT_read(base_nores)

    mstar_les_lt_on = interp_to(t_les_lt, mstar_les_lt, t_param)
    mstar_les_nl_on = interp_to(t_les_nl, mstar_les_nl, t_param)
    mstar_LT_LES = mstar_les_lt_on - mstar_les_nl_on

    # --- MOM diagnosed mstar for 3 runs ---
    base_nlt = model_dirs["Hybrid_noLT"].format(y=y_tag)
    base_res = model_dirs["Hybrid_res_LT"].format(y=y_tag)

    t_nlt,   mstar_nlt   = load_mom_mstar_diag(base_nlt)
    t_nores, mstar_nores = load_mom_mstar_diag(base_nores)
    t_res,   mstar_res   = load_mom_mstar_diag(base_res)

    mstar_nlt_on   = interp_to(t_nlt,   mstar_nlt,   t_param)
    mstar_nores_on = interp_to(t_nores, mstar_nores, t_param)
    mstar_res_on   = interp_to(t_res,   mstar_res,   t_param)

    mstar_LT_nores_diag = mstar_nores_on - mstar_nlt_on
    mstar_LT_res_diag   = mstar_res_on   - mstar_nlt_on

    # --- rescaling factors ---
    ok = np.isfinite(MSTAR_LT_nores_read) & (np.abs(MSTAR_LT_nores_read) > EPS)

    R_LES = np.full_like(MSTAR_LT_nores_read, np.nan, dtype=float)
    R_LES[ok] = mstar_LT_LES[ok] / MSTAR_LT_nores_read[ok]

    R_proxy = np.full_like(MSTAR_LT_nores_read, np.nan, dtype=float)
    R_proxy[ok] = mstar_LT_res_diag[ok] / MSTAR_LT_nores_read[ok]

    # -------- Col 1: factor --------
    a = ax[r, 0]
    a.plot(t_param, R_LES,   "k-", lw=1.9,
           label=r"$\mathcal{R}_{\mathrm{LES}}=m^{\mathrm{LT}}_{*,\mathrm{LES}}/m^{\mathrm{LT}}_{*,\mathrm{RL19}}$")
    a.plot(t_param, R_proxy, "m-", lw=1.9,
           label=r"$\mathcal{R}_{\mathrm{proxy}}=m^{\mathrm{LT}}_{*,\mathrm{res}}/m^{\mathrm{LT}}_{*,\mathrm{RL19}}$")
    a.axhline(1.0, color="0.35", lw=1.0)
    a.grid(True, alpha=0.22)
    a.set_xlim(*XLIM)
    a.set_ylim(-1, 5)
    a.set_title(rf"$(y-y_c)/RMW={y_norm_rmw:+.2f}$  (y={y_km:+.0f} km)", pad=10)
    a.set_ylabel(r"Rescaling factor $\mathcal{R}$")
    if r == 0:
        a.legend(loc="best", frameon=False)

    # -------- Col 2: mstar_LT comparison --------
    b = ax[r, 1]
    b.plot(t_param, mstar_LT_LES,        "k-",  lw=1.9,
           label=r"$m^{\mathrm{LT}}_{*,\mathrm{LES}}$")
    b.plot(t_param, mstar_LT_nores_diag, "m--", lw=1.7,
           label=r"$m^{\mathrm{LT}}_{*,\mathrm{nores}}$")
    b.plot(t_param, mstar_LT_res_diag,   "b-",  lw=1.7,
           label=r"$m^{\mathrm{LT}}_{*,\mathrm{res}}$")
    b.grid(True, alpha=0.22)
    b.set_xlim(*XLIM)
    b.set_ylim(-1, 1)
    b.set_ylabel(r"$m^{\mathrm{LT}}_{*}$")
    if r == 0:
        b.legend(loc="best", frameon=False)

    # ---- row tags on first column axes ----
    ax[r, 0].text(-0.22, 1.03, ROW_TAGS[r],
                  transform=ax[r, 0].transAxes,
                  ha="left", va="bottom", fontsize=14)

# ---- column tags only on first row ----
ax[0, 0].text(0.02, 1.03, COL_TAGS[0],
              transform=ax[0, 0].transAxes,
              ha="left", va="bottom", fontsize=14)
ax[0, 1].text(0.02, 1.03, COL_TAGS[1],
              transform=ax[0, 1].transAxes,
              ha="left", va="bottom", fontsize=14)

ax[2, 0].set_xlabel("day")
ax[2, 1].set_xlabel("day")

# fig.suptitle("Fig5: 5 m s$^{-1}$, MLD=10 m — Rescaling LT verification (LES vs MOM6; diagnosed $m^{\\mathrm{LT}}_{*}$)",
             # y=0.995)

fig.tight_layout()


fig.savefig(SAVE_EPS, format="eps", bbox_inches="tight")

print("saved:", SAVE_EPS)
plt.show()






In [ ]:
#this code is for figure 10:The mechanical energy $Me$ diagnosed by Eq. \ref{eq:Me_diag} for different schemes across the full set of test locations for both 5 $\rm ms^{-1}$ and 10 $\rm ms^{-1}$. Panel (a), (b) and (c) are results under 5 $\rm ms^{-1}$ speed and (d), (e) and (f) are results under 10 $\rm ms^{-1}$ speed. Panel (a) and (d) present the mixing energetics for ST baseline; Panel (b) and (e) are for ST+LT; Panel (c) and (f) are LT-induced energetic increment (ST+LT minus ST).
#!/usr/bin/env python3
# ============================================================
# Combined Fig: Me spatial structure (5 m/s + 10 m/s)
# (your original script) + NEW: caching computed fields to disk
# ============================================================

import os
import json
import hashlib
import numpy as np
import scipy.io as sio
import xarray as xr
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter

# -------------------------------
# COMMON CONSTANTS
# -------------------------------
rho0  = 1026.95
alpha = -0.2
g     = 9.81

# reference time grid (days)
t_ref = np.linspace(0, 3, 120)
t_center = 1.5

# nondim Y and plot range
RMW_M  = 50e3

y_km_list     = [300, 200, 150, 130, 110, 90, 70, 50, 30, 10, 0, -20, -40, -60, -80, -100, -120, -140, -200, -300]
y_folder_list = ["300","200","150","130","110","90","70","50","30","10","0","S20","S40","S60","S80","S100","S120","S140","S200","S300"]

LES_case_list       = ["021","031","036","038","040","042","044","046","048","050","051","053","055","057","059","061","063","065","071","081"]  # LT
LES_noLT_case_list  = ["021","031","036","038","040","042","044","046","048","050","051","053","055","057","059","061","063","065","071","081"]  # ST(noLT)

Y = np.array(y_km_list) * 1e3 / RMW_M
XLIM = (-4, 4)
YLIM = (-4, 4)

# refinement / smoothing controls
NY_FINE = 220
SMOOTH_SIGMA = (0.6, 0.0)  # (sigma_y, sigma_x)

# contour controls
NLEVEL_ABS  = 21
NLEVEL_DIFF = 21
DRAW_CONTOUR_LINES_ABS  = True
DRAW_CONTOUR_LINES_DIFF = True

# plot choice
ME_USE_ABS = False  # signed if False

# -------------------------------
# NEW: CACHE SETTINGS
# -------------------------------
CACHE_DIR = "/archive/Qian.Xiao/Qian.Xiao/eps/"  # <<< change if you want
os.makedirs(CACHE_DIR, exist_ok=True)

# -------------------------------
# HELPERS
# -------------------------------
def interp_to(t_src, y_src, t_tgt):
    t_src = np.asarray(t_src).astype(float)
    y_src = np.asarray(y_src).astype(float)
    t_tgt = np.asarray(t_tgt).astype(float)
    m = np.isfinite(t_src) & np.isfinite(y_src)
    if m.sum() < 2:
        return np.full_like(t_tgt, np.nan, dtype=float)
    t2 = t_src[m]
    y2 = y_src[m]
    if np.any(np.diff(t2) < 0):
        o = np.argsort(t2)
        t2 = t2[o]
        y2 = y2[o]
    return np.interp(t_tgt, t2, y2, left=y2[0], right=y2[-1])

def _isel0(da):
    indexers = {}
    for d in da.dims:
        if d in ("Time", "zl", "zi"):
            continue
        indexers[d] = 0
    return da.isel(**indexers) if indexers else da

def _time_in_days(ds):
    if "Time" in ds.coords or "Time" in ds.variables:
        try:
            tt = np.asarray(ds["Time"].values).astype(float)
            if tt.ndim == 1:
                tmax = np.nanmax(tt)
                if tmax > 100:
                    return tt / 86400.0
                else:
                    return tt
        except Exception:
            pass
    nt = ds.dims["Time"]
    return (300.0 / 86400.0) * (np.arange(nt) + 0.5)

def compute_Hbl_maxgrad(T, z):
    T = np.asarray(T)
    z = np.asarray(z)
    dTdz = np.gradient(T, z, axis=1)
    idx = np.argmax(np.abs(dTdz), axis=1)
    return z[idx]

def compute_Me_from_tw(z_iface, tw_iface, Hbl):
    z = np.asarray(z_iface).astype(float)
    tw = np.asarray(tw_iface).astype(float)
    Hbl = np.asarray(Hbl).astype(float)
    bflux = -(g / rho0) * alpha * tw
    Me = np.full((bflux.shape[0],), np.nan)
    for it in range(bflux.shape[0]):
        H = Hbl[it]
        if not np.isfinite(H) or H <= z[0]:
            continue
        mask = (z <= H)
        zz = z[mask]
        bf = bflux[it, mask]
        neg = (bf < 0)
        Me[it] = np.trapezoid(bf[neg], zz[neg]) if np.any(neg) else 0.0
    return Me

def _ensure_increasing_z(z, A):
    z = np.asarray(z).astype(float)
    if z.size >= 2 and z[1] < z[0]:
        z = z[::-1]
        A = A[..., ::-1]
    return z, A

def load_les_profile(case_id, les_lt_root, les_st_root, mode="LT"):
    modeU = mode.upper()
    root = les_lt_root if modeU == "LT" else les_st_root
    f = os.path.join(root, str(case_id), "LESout.mat")
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing LES file: {f}")

    mat = sio.loadmat(f)
    t = np.asarray(mat["t"]).squeeze().astype("f8") / 86400.0
    z = np.asarray(mat["z"]).T.squeeze().astype("f8")
    T = np.asarray(mat["T"]).astype("f8") - 273.15
    tw = -np.asarray(mat["tw"]).astype("f8")

    if T.ndim == 2 and T.shape[0] != t.size and T.shape[1] == t.size:
        T = T.T
    if tw.ndim == 2 and tw.shape[0] != t.size and tw.shape[1] == t.size:
        tw = tw.T

    z, T  = _ensure_increasing_z(z, T)
    z, tw = _ensure_increasing_z(z, tw)
    return np.asarray(t), np.asarray(z), np.asarray(T), np.asarray(tw)

def load_mom_Me_from_prog_visc(results_dir):
    prog = os.path.join(results_dir, "prog.nc")
    visc = os.path.join(results_dir, "visc.nc")
    if not os.path.exists(prog):
        raise FileNotFoundError(prog)
    if not os.path.exists(visc):
        raise FileNotFoundError(visc)

    with xr.open_dataset(prog) as ds:
        t_prog = _time_in_days(ds)
        if "temp" in ds.variables:
            temp = _isel0(ds["temp"]).values
        elif "thetao" in ds.variables:
            temp = _isel0(ds["thetao"]).values
        else:
            raise KeyError(f"No temp/thetao in {prog}. Vars: {list(ds.variables)}")
        zl = ds["zl"].values
    Hbl_prog = compute_Hbl_maxgrad(temp, zl)

    with xr.open_dataset(visc) as ds:
        t_visc = _time_in_days(ds)
        zi = ds["zi"].values
        if "Tflx_dia_diff" not in ds.variables:
            raise KeyError(f"Tflx_dia_diff not found in {visc}. Vars: {list(ds.variables)}")
        tw_zi = -_isel0(ds["Tflx_dia_diff"]).values

    Hbl = interp_to(t_prog, Hbl_prog, t_visc)
    Me = compute_Me_from_tw(zi, tw_zi, Hbl)
    return np.asarray(t_visc), np.asarray(Me)

def stack_field_over_y(load_fn_list):
    rows = []
    for fn in load_fn_list:
        t, s = fn()
        rows.append(interp_to(t, s, t_ref))
    return np.vstack(rows)

def refine_y(Z, Ycoarse, Yfine):
    f = interp1d(Ycoarse, Z, axis=0, kind="cubic", bounds_error=False, fill_value="extrapolate")
    return f(Yfine)

def robust_limits_sym(fields, pct=98):
    vv = np.concatenate([np.ravel(f[np.isfinite(f)]) for f in fields])
    if vv.size == 0:
        return (-1, 1)
    vmax = np.percentile(np.abs(vv), pct)
    return (-vmax, vmax)

def robust_limits(fields, pct_low=2, pct_high=98):
    vv = np.concatenate([np.ravel(f[np.isfinite(f)]) for f in fields])
    if vv.size == 0:
        return (-1, 1)
    vmin = np.percentile(vv, pct_low)
    vmax = np.percentile(vv, pct_high)
    return (vmin, vmax)

# -------------------------------
# NEW: cache naming / save / load
# -------------------------------
def _stable_obj(obj):
    """
    Convert potentially nested dict/list/path objects into JSON-serializable
    stable structure (sorted keys), so hashing is stable.
    """
    if isinstance(obj, dict):
        return {k: _stable_obj(obj[k]) for k in sorted(obj.keys())}
    if isinstance(obj, (list, tuple)):
        return [_stable_obj(x) for x in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

def make_cache_name(Ustorm, les_lt_root, les_st_root, model_dirs):
    meta = {
        "Ustorm": float(Ustorm),
        "les_lt_root": les_lt_root,
        "les_st_root": les_st_root,
        "model_dirs": model_dirs,
        "y_km_list": y_km_list,
        "y_folder_list": y_folder_list,
        "LES_case_list": LES_case_list,
        "LES_noLT_case_list": LES_noLT_case_list,
        "t_ref": t_ref.tolist(),
        "t_center": float(t_center),
        "RMW_M": float(RMW_M),
        "ME_USE_ABS": bool(ME_USE_ABS),
        # if you ever change physics constants, it should invalidate cache
        "rho0": float(rho0),
        "alpha": float(alpha),
        "g": float(g),
    }
    s = json.dumps(_stable_obj(meta), sort_keys=True, separators=(",", ":"))
    h = hashlib.md5(s.encode("utf-8")).hexdigest()
    return f"Me_fields_U{int(round(Ustorm))}ms_{h}.npz", meta

def save_cache_npz(path, figdata, meta):
    np.savez_compressed(
        path,
        X=np.asarray(figdata["X"]),
        row0_st=np.asarray(figdata["row0_st"]),
        row1_stlt=np.asarray(figdata["row1_stlt"]),
        row2_diff=np.asarray(figdata["row2_diff"]),
        meta_json=json.dumps(_stable_obj(meta), sort_keys=True)
    )

def load_cache_npz(path):
    with np.load(path, allow_pickle=True) as npz:
        figdata = {
            "X": npz["X"],
            "row0_st": npz["row0_st"],
            "row1_stlt": npz["row1_stlt"],
            "row2_diff": npz["row2_diff"],
        }
        meta_json = str(npz["meta_json"])
    return figdata, meta_json

# -------------------------------
# BUILD (one speed) + CACHE
# -------------------------------
def build_all_fields_for_speed(
    Ustorm,
    les_lt_root,
    les_st_root,
    model_dirs,
    cache_dir=CACHE_DIR,
    use_cache=True,
    overwrite_cache=False
):
    cache_file, meta = make_cache_name(Ustorm, les_lt_root, les_st_root, model_dirs)
    cache_path = os.path.join(cache_dir, cache_file)

    if use_cache and (not overwrite_cache) and os.path.exists(cache_path):
        figdata, _meta_json = load_cache_npz(cache_path)
        print(f"[CACHE HIT] {cache_path}")
        return figdata

    print(f"[CACHE MISS] computing fields for U={Ustorm} m/s ...")
    X = (t_ref - t_center) * 86400.0 * Ustorm / RMW_M

    def les_me(case_id, mode):
        t, z, T, tw = load_les_profile(case_id, les_lt_root, les_st_root, mode=mode)
        H = compute_Hbl_maxgrad(T, z)
        Me = compute_Me_from_tw(z, tw, H)
        return t, Me

    les_lt  = stack_field_over_y([(lambda cid=cid: les_me(cid, "LT")) for cid in LES_case_list])
    les_nlt = stack_field_over_y([(lambda cid=cid: les_me(cid, "ST")) for cid in LES_noLT_case_list])

    def mom_me(key, ytag):
        base = model_dirs[key].format(y=ytag)
        return load_mom_Me_from_prog_visc(base)

    mom = {}
    for key in model_dirs.keys():
        mom[key] = stack_field_over_y([
            (lambda key=key, ytag=ytag: mom_me(key, ytag)) for ytag in y_folder_list
        ])

    def proc(Z):
        return np.abs(Z) if ME_USE_ABS else Z

    les_lt_p  = proc(les_lt)
    les_nlt_p = proc(les_nlt)
    mom_p     = {k: proc(v) for k, v in mom.items()}

    figdata = {
        "X": X,
        "row0_st":   np.stack([les_nlt_p, mom_p["hybrid_noLT"], mom_p["shear"]], axis=0),  # (3, ny, nt)
        "row1_stlt": np.stack([les_lt_p,  mom_p["waveLT_rescale_all"], mom_p["waveLT_norescale"]], axis=0),
        "row2_diff": np.stack([
            les_lt_p - les_nlt_p,
            mom_p["waveLT_rescale_all"] - mom_p["hybrid_noLT"],
            mom_p["waveLT_norescale"]   - mom_p["hybrid_noLT"]
        ], axis=0),
    }

    # save cache
    if use_cache:
        save_cache_npz(cache_path, figdata, meta)
        print(f"[CACHE SAVE] {cache_path}")

    return figdata

# -------------------------------
# PLOT (combined)  (your original, unchanged)
# -------------------------------
def plot_combined(figdata_5, figdata_10):
    Yfine = np.linspace(Y.min(), Y.max(), NY_FINE)

    def prep(Z):
        Zf = refine_y(Z, Y, Yfine)
        if SMOOTH_SIGMA is not None:
            Zf = gaussian_filter(Zf, sigma=SMOOTH_SIGMA)
        return Zf

    # figdata row arrays are (3, ny, nt)
    rows_5 = [
        [prep(figdata_5["row0_st"][i, :, :]) for i in range(3)],
        [prep(figdata_5["row1_stlt"][i, :, :]) for i in range(3)],
        [prep(figdata_5["row2_diff"][i, :, :]) for i in range(3)],
    ]
    rows_10 = [
        [prep(figdata_10["row0_st"][i, :, :]) for i in range(3)],
        [prep(figdata_10["row1_stlt"][i, :, :]) for i in range(3)],
        [prep(figdata_10["row2_diff"][i, :, :]) for i in range(3)],
    ]

    abs_panels_all  = rows_5[0] + rows_5[1] + rows_10[0] + rows_10[1]
    diff_panels_all = rows_5[2] + rows_10[2]
    _ = robust_limits(abs_panels_all, 2, 98)
    _ = robust_limits_sym(diff_panels_all, 98)

    abs_levels  = np.linspace(-0.0005,  0.0005,  NLEVEL_ABS)
    diff_levels = np.linspace(-0.0002,  0.0002,  NLEVEL_DIFF)

    fig = plt.figure(figsize=(12.8, 16.2))
    gs = fig.add_gridspec(
        nrows=6, ncols=4,
        width_ratios=[1, 1, 1, 0.045],
        height_ratios=[1, 1, 1, 1, 1, 1],
        wspace=0.10, hspace=0.28
    )

    axs = np.empty((6,3), dtype=object)
    caxs = []
    for r in range(6):
        for c in range(3):
            axs[r,c] = fig.add_subplot(gs[r,c])
        caxs.append(fig.add_subplot(gs[r,3]))

    titles_r0 = ["LES_ST", "Hybrid_ST", "Shear"]
    titles_r1 = ["LES_ST+LT", "Hybrid_ST+LT (res)", "Hybrid_ST+LT (nores)"]
    titles_r2 = ["LES_ST+LT − LES_ST", "Hybrid_ST+LT (res) − Hybrid_ST", "Hybrid_ST+LT (nores) − Hybrid_ST"]

    cmap_abs  = "RdBu_r"
    cmap_diff = "RdBu_r"

    def draw_row(r_ax_base, X, row_panels, titles, levels, cmap, draw_lines, cax, cbar_label):
        m = None
        for c in range(3):
            ax = axs[r_ax_base, c]
            Z  = row_panels[c]
            m = ax.contourf(X, Yfine, Z, levels=levels, cmap=cmap, extend="both")
            if draw_lines:
                ax.contour(X, Yfine, Z, levels=levels, colors="k", linewidths=0.45, alpha=0.45)
            ax.set_title(titles[c], fontsize=11)
            ax.set_xlim(*XLIM); ax.set_ylim(*YLIM)
        cb = fig.colorbar(m, cax=cax)
        cb.set_label(cbar_label, rotation=90)
        return m

    X5  = figdata_5["X"]
    X10 = figdata_10["X"]

    draw_row(0, X5,  rows_5[0],  titles_r0, abs_levels,  cmap_abs,  DRAW_CONTOUR_LINES_ABS,  caxs[0],
             r"$Me_i\ \mathrm{(m^3\,s^{-3})}$")
    draw_row(1, X5,  rows_5[1],  titles_r1, abs_levels,  cmap_abs,  DRAW_CONTOUR_LINES_ABS,  caxs[1],
             r"$Me_i\ \mathrm{(m^3\,s^{-3})}$")
    draw_row(2, X5,  rows_5[2],  titles_r2, diff_levels, cmap_diff, DRAW_CONTOUR_LINES_DIFF, caxs[2],
             r"$\Delta Me_i\ \mathrm{(m^3\,s^{-3})}$")

    draw_row(3, X10, rows_10[0], titles_r0, abs_levels,  cmap_abs,  DRAW_CONTOUR_LINES_ABS,  caxs[3],
             r"$Me_i\ \mathrm{(m^3\,s^{-3})}$")
    draw_row(4, X10, rows_10[1], titles_r1, abs_levels,  cmap_abs,  DRAW_CONTOUR_LINES_ABS,  caxs[4],
             r"$Me_i\ \mathrm{(m^3\,s^{-3})}$")
    draw_row(5, X10, rows_10[2], titles_r2, diff_levels, cmap_diff, DRAW_CONTOUR_LINES_DIFF, caxs[5],
             r"$\Delta Me_i\ \mathrm{(m^3\,s^{-3})}$")

    for r in range(6):
        axs[r,0].set_ylabel(r"$(y-y_c)/RMW$")
        for c in (1,2):
            axs[r,c].set_ylabel("")
            axs[r,c].tick_params(labelleft=False)

    for c in range(3):
        axs[5,c].set_xlabel(r"$(x-x_c)/RMW$")

    for r in range(6):
        for c in range(3):
            axs[r,c].set_xticks([-4, -2, 0, 2, 4])
            axs[r,c].set_yticks([-4, -2, 0, 2, 4])

    ROW_TAGS = ["(a)","(b)","(c)","(d)","(e)","(f)"]
    COL_TAGS = ["(i)","(ii)","(iii)"]

    for r in range(6):
        axs[r,0].text(-0.22, 1.03, ROW_TAGS[r],
                      transform=axs[r,0].transAxes,
                      ha="left", va="bottom", fontsize=14)

    for c in range(3):
        axs[0,c].text(0.02, 1.03, COL_TAGS[c],
                      transform=axs[0,c].transAxes,
                      ha="left", va="bottom", fontsize=14)

    SAVE_EPS = "/archive/Qian.Xiao/Qian.Xiao/eps/Me_SCM"
    fig.savefig(SAVE_EPS, format="eps", bbox_inches="tight")
    print("saved:", SAVE_EPS)
    plt.show()

# ============================================================
# RUN (unchanged)
# ============================================================
les_lt_root_5 = "/archive/bgr/Datasets/LES/MoreHurr/TS05_ML10/LT/"
les_st_root_5 = "/archive/bgr/Datasets/LES/MoreHurr/TS05_ML10/ST/"
model_dirs_5 = {
    "hybrid_noLT"          : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/noLT_10m/{y}_results",
    "waveLT_rescale_all"   : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/waveLT_res_10m/{y}_results",
    "waveLT_norescale"     : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/waveLT_nores_10m/{y}_results",
    "shear"                : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/shear_10m/{y}_results",
}

figdata_5 = build_all_fields_for_speed(
    Ustorm=5.0,
    les_lt_root=les_lt_root_5,
    les_st_root=les_st_root_5,
    model_dirs=model_dirs_5,
    use_cache=True,
    overwrite_cache=False
)

les_lt_root_10 = "/archive/bgr/Datasets/LES/MoreHurr/TS10_ML10/LT/"
les_st_root_10 = "/archive/bgr/Datasets/LES/MoreHurr/TS10_ML10/ST/"
model_dirs_10 = {
    "hybrid_noLT"          : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/noLT_10m_10mps/{y}_results",
    "waveLT_rescale_all"   : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/waveLT_res_10m_10mps/{y}_results",
    "waveLT_norescale"     : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/waveLT_nores_10m_10mps/{y}_results",
    "shear"                : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/shear_10m_10mps/{y}_results",
}

figdata_10 = build_all_fields_for_speed(
    Ustorm=10.0,
    les_lt_root=les_lt_root_10,
    les_st_root=les_st_root_10,
    model_dirs=model_dirs_10,
    use_cache=True,
    overwrite_cache=False
)

plot_combined(figdata_5, figdata_10)








In [ ]:
##this code is for figure 11:Multi-location SST diagnostics for one-dimensional idealized hurricanes translating at 5 $\rm ms^{-1}$ and 10 $\rm ms^{-1}$. The SST values in panel (a), (b), (d) and (e) are maps with $\theta(t)-\theta(t=0)$ where the initial SST value $\theta(t=0)=29.25 \rm ^\circ C$. Panel (a), (b) and (c) are results under 5 $\rm ms^{-1}$ speed and (d), (e) and (f) are results under 10 $\rm ms^{-1}$ speed. Panel (a) and (d) present the SST in the case of ST baseline; Panel (b) and (e) are for ST+LT; Panel (c) and (f) are the LT-induced SST cooling increment.
#!/usr/bin/env python3
# ============================================================
# Combined Fig: SST spatial structure (TS05 5 m/s + TS10 10 m/s)
# 6 rows x 3 cols + row-wise colorbars
#
# Rows 0-2: TS05 (5 m/s)
# Rows 3-5: TS10 (10 m/s)
#
# Row 0/3 (ABS, ST):     [LES_ST, Hybrid_ST, Shear]
# Row 1/4 (ABS, ST+LT):  [LES_ST+LT, Hybrid_ST+LT (res), Hybrid_ST+LT (nores)]
# Row 2/5 (DIFF):        [LES_ST+LT - LES_ST,
#                         Hybrid_ST+LT (res) - Hybrid_ST,
#                         Hybrid_ST+LT (nores) - Hybrid_ST]
#
# Formatting (your requirements):
#  - row tags (a)-(f) on first column only
#  - col tags (i)-(iii) on first row only
#  - y label: (y-y_c)/RMW     x label: (x-x_c)/RMW
#  - titles EXACT, no "(SST-29.5)" etc
#  - caching: save computed coarse matrices so later plotting is fast
# ============================================================

import os
import json
import hashlib
import numpy as np
import scipy.io as sio
import xarray as xr
import matplotlib.pyplot as plt
import cmocean
import matplotlib.colors as mcolors
from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter

# -------------------------------
# COMMON SETTINGS
# -------------------------------
SST0   = 29.5
RMW_M  = 50e3

# use ONE common time grid for both speeds (keep as you like)
t_ref  = np.linspace(0, 3, 140)
t_c    = 1.5  # storm center time (days)

XLIM = (-4, 4)
YLIM = (-4, 4)

# Y list (shared)
y_km_list     = [300, 200, 150, 130, 110, 90, 70, 50, 30, 10, 0, -20, -40, -60, -80, -100, -120, -140, -200, -300]
y_folder_list = ["300","200","150","130","110","90","70","50","30","10","0","S20","S40","S60","S80","S100","S120","S140","S200","S300"]

LES_case_list      = ["021","031","036","038","040","042","044","046","048","050","051","053","055","057","059","061","063","065","071","081"]  # LT
LES_noLT_case_list = ["021","031","036","038","040","042","044","046","048","050","051","053","055","057","059","061","063","065","071","081"]  # ST

assert len(y_km_list)==len(y_folder_list)==len(LES_case_list)==len(LES_noLT_case_list)

# nondim Y
Yc = np.array(y_km_list) * 1e3 / RMW_M

# refine / smooth (plot-only)
NY_FINE      = 260
SMOOTH_SIGMA = (0.55, 0.0)

# contour controls
NLEVEL_ABS  = 22
NLEVEL_DIFF = 22
abs_levels  = np.linspace(-2.0, 0.5, NLEVEL_ABS)
diff_levels = np.linspace(-0.3, 0.3, NLEVEL_DIFF)

CMAP_ABS  = cmocean.cm.balance
CMAP_DIFF = cmocean.cm.balance
abs_norm  = mcolors.TwoSlopeNorm(vmin=-2.0, vcenter=0.0, vmax=0.5)
diff_norm = mcolors.TwoSlopeNorm(vmin=-0.3, vcenter=0.0, vmax=0.3)

DRAW_CONTOUR_LINES_ABS  = True
DRAW_CONTOUR_LINES_DIFF = True
LINE_LW    = 0.45
LINE_ALPHA = 0.45

# output
DPI     = 250
SAVEFIG = True
OUTEPS  = "/archive/Qian.Xiao/Qian.Xiao/eps/SSTSCM20.eps"

# cache
CACHE_DIR = "/archive/Qian.Xiao/Qian.Xiao/eps/SCM_SST/"
os.makedirs(CACHE_DIR, exist_ok=True)

# -------------------------------
# helpers
# -------------------------------
def interp_to(t_src, y_src, t_tgt):
    t_src = np.asarray(t_src, float).squeeze()
    y_src = np.asarray(y_src, float).squeeze()
    t_tgt = np.asarray(t_tgt, float).squeeze()

    m = np.isfinite(t_src) & np.isfinite(y_src)
    if m.sum() < 2:
        return np.full_like(t_tgt, np.nan, dtype=float)

    t2 = t_src[m]; y2 = y_src[m]
    if np.any(np.diff(t2) < 0):
        o = np.argsort(t2); t2 = t2[o]; y2 = y2[o]

    return np.interp(t_tgt, t2, y2, left=y2[0], right=y2[-1])

def isel_horiz0_ds(ds):
    idx = {}
    for d in ds.dims:
        if d in ("Time","zl","zi"):
            continue
        idx[d] = 0
    return ds.isel(**idx) if idx else ds

def mom_time_in_days(ds):
    # keep your original "300s timestep" fallback
    if "Time" in ds.coords or "Time" in ds.variables:
        try:
            tt = np.asarray(ds["Time"].values).astype(float)
            if tt.ndim == 1:
                tmax = np.nanmax(tt)
                return tt/86400.0 if tmax > 100 else tt
        except Exception:
            pass
    nt = ds.dims["Time"]
    return (300.0/86400.0) * (np.arange(nt) + 0.5)

def load_les_sst_from_LESout(case_id, root):
    f = os.path.join(root, str(case_id), "LESout.mat")
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing LES file: {f}")
    mat = sio.loadmat(f)
    t = np.asarray(mat["t"]).squeeze().astype(float)/86400.0
    T = np.asarray(mat["T"]).astype(float)
    # ensure (Nt,Nz)
    if T.ndim != 2:
        raise ValueError(f"T has shape {T.shape} in {f}")
    if T.shape[0] != t.size and T.shape[1] == t.size:
        T = T.T
    sst = (T[:,0] - 273.15).squeeze()
    return t, sst

def load_mom_sst_from_prog(results_dir):
    prog = os.path.join(results_dir, "prog.nc")
    if not os.path.exists(prog):
        raise FileNotFoundError(prog)

    with xr.open_dataset(prog, decode_times=False) as ds:
        ds0 = isel_horiz0_ds(ds)

        # ✅ 关键：永远用 300s 的合成时间轴（跟你原来脚本完全一致）
        nt = ds0.dims["Time"]
        t = (300.0 / 86400.0) * (np.arange(nt) + 0.5)

        if "temp" in ds0.variables:
            sst = ds0["temp"].isel(zl=0).values
        elif "thetao" in ds0.variables:
            sst = ds0["thetao"].isel(zl=0).values
        else:
            raise KeyError(f"No temp/thetao in {prog}. Vars={list(ds0.variables)}")

        # ✅ 保证是 (Time,) 一维
        sst = np.asarray(sst).reshape(-1)

    return np.asarray(t), np.asarray(sst)


def refine_y(Z_coarse, y_coarse, y_fine):
    f = interp1d(y_coarse, Z_coarse, axis=0, kind="cubic",
                 bounds_error=False, fill_value="extrapolate")
    return f(y_fine)

def stable_hash(cfg: dict):
    s = json.dumps(cfg, sort_keys=True, separators=(",",":"))
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def assert_not_all_nan(A, name, thr=0.99):
    frac = np.isnan(A).mean()
    if frac >= thr:
        raise RuntimeError(f"[BAD FIELD] {name} is almost all NaN (nan_frac={frac:.3f}). "
                           f"Most likely a read/interp issue for that dataset.")

# -------------------------------
# build one speed (compute + cache)
# -------------------------------
def build_speed(tag, Ustorm, les_lt_root, les_st_root, model_dirs,
                use_cache=True, overwrite_cache=False):

    cfg = dict(
        tag=tag,
        Ustorm=float(Ustorm),
        les_lt_root=les_lt_root,
        les_st_root=les_st_root,
        model_dirs=model_dirs,
        y_km_list=y_km_list,
        y_folder_list=y_folder_list,
        LES_case_list=LES_case_list,
        LES_noLT_case_list=LES_noLT_case_list,
        t_ref=t_ref.tolist(),
        SST0=float(SST0),
        RMW_M=float(RMW_M),
    )
    cache_file = os.path.join(CACHE_DIR, f"SST_{tag}_{stable_hash(cfg)}.npz")

    if use_cache and (not overwrite_cache) and os.path.exists(cache_file):
        with np.load(cache_file, allow_pickle=True) as npz:
            data = {k: npz[k] for k in npz.files if k != "cfg_json"}
        print(f"[CACHE HIT] {cache_file}")
        return data

    print(f"[CACHE MISS] computing {tag} ...")

    ny = len(y_folder_list)
    nt = len(t_ref)

    # -------- LES matrices (ny, nt) --------
    LES_ST  = np.full((ny, nt), np.nan)
    LES_LT  = np.full((ny, nt), np.nan)

    for j, cid in enumerate(LES_noLT_case_list):
        t, sst = load_les_sst_from_LESout(cid, les_st_root)
        LES_ST[j,:] = interp_to(t, sst, t_ref)

    for j, cid in enumerate(LES_case_list):
        t, sst = load_les_sst_from_LESout(cid, les_lt_root)
        LES_LT[j,:] = interp_to(t, sst, t_ref)

    # -------- MOM matrices (ny, nt) --------
    MOM = {}
    for key, tpl in model_dirs.items():
        A = np.full((ny, nt), np.nan)
        for j, ytag in enumerate(y_folder_list):
            base = tpl.format(y=ytag)
            t, sst = load_mom_sst_from_prog(base)
            A[j,:] = interp_to(t, sst, t_ref)
        MOM[key] = A

    # sanity checks (catch your "only first col has data" issue early)
    assert_not_all_nan(LES_ST, f"{tag}:LES_ST")
    assert_not_all_nan(LES_LT, f"{tag}:LES_LT")
    for key in ("hybrid_noLT","waveLT_rescale_all","waveLT_norescale","shear"):
        assert_not_all_nan(MOM[key], f"{tag}:MOM:{key}")

    # anomalies
    LES_ST_a = LES_ST - SST0
    LES_LT_a = LES_LT - SST0
    H0_a     = MOM["hybrid_noLT"]        - SST0
    RES_a    = MOM["waveLT_rescale_all"] - SST0
    NORES_a  = MOM["waveLT_norescale"]   - SST0
    SHEAR_a  = MOM["shear"]             - SST0

    # rows stored as (3, ny, nt)
    row0 = np.stack([LES_ST_a, H0_a,    SHEAR_a], axis=0)
    row1 = np.stack([LES_LT_a, RES_a,   NORES_a], axis=0)
    row2 = np.stack([LES_LT_a-LES_ST_a,
                     RES_a-H0_a,
                     NORES_a-H0_a], axis=0)

    # coords
    X = (t_ref - t_c) * 86400.0 * Ustorm / RMW_M

    data = dict(
        X=X,
        Yc=Yc,
        row0_st=row0,
        row1_stlt=row1,
        row2_diff=row2,
    )

    if use_cache:
        np.savez_compressed(cache_file, cfg_json=json.dumps(cfg, sort_keys=True), **data)
        print(f"[CACHE SAVE] {cache_file}")

    return data

# -------------------------------
# plot combined 6x3
# -------------------------------
def plot_combined(data5, data10):
    Yfine = np.linspace(Yc.min(), Yc.max(), NY_FINE)

    def prep(Z):  # (ny, nt)
        Zf = refine_y(Z, Yc, Yfine)
        if SMOOTH_SIGMA is not None:
            Zf = gaussian_filter(Zf, sigma=SMOOTH_SIGMA)
        return Zf

    # build panels: list of 6 rows, each row is list of 3 panels (Yfine, X)
    blocks = []
    for dat in (data5, data10):
        blocks.append([prep(dat["row0_st"][i,:,:])   for i in range(3)])
        blocks.append([prep(dat["row1_stlt"][i,:,:]) for i in range(3)])
        blocks.append([prep(dat["row2_diff"][i,:,:]) for i in range(3)])

    titles_r0 = ["LES_ST", "Hybrid_ST", "Shear"]
    titles_r1 = ["LES_ST+LT", "Hybrid_ST+LT (res)", "Hybrid_ST+LT (nores)"]
    titles_r2 = ["LES_ST+LT - LES_ST",
                 "Hybrid_ST+LT (res) - Hybrid_ST",
                 "Hybrid_ST+LT (nores) - Hybrid_ST"]
    row_titles = [titles_r0, titles_r1, titles_r2, titles_r0, titles_r1, titles_r2]

    fig = plt.figure(figsize=(12.8, 16.6))
    gs = fig.add_gridspec(
        nrows=6, ncols=4,
        width_ratios=[1, 1, 1, 0.065],
        wspace=0.10, hspace=0.30
    )

    axs = np.empty((6,3), dtype=object)
    caxs = []
    for r in range(6):
        for c in range(3):
            axs[r,c] = fig.add_subplot(gs[r,c])
        caxs.append(fig.add_subplot(gs[r,3]))

    def draw_row(r, X, panels, titles, is_diff=False):
        levels = diff_levels if is_diff else abs_levels
        cmap   = CMAP_DIFF if is_diff else CMAP_ABS
        norm   = diff_norm if is_diff else abs_norm
        draw_lines = DRAW_CONTOUR_LINES_DIFF if is_diff else DRAW_CONTOUR_LINES_ABS

        m = None
        for c in range(3):
            ax = axs[r,c]
            Z  = panels[c]
            m = ax.contourf(X, Yfine, Z, levels=levels, cmap=cmap, norm=norm, extend="both")
            if draw_lines:
                ax.contour(X, Yfine, Z, levels=levels, colors="k",
                           linewidths=LINE_LW, alpha=LINE_ALPHA)
            ax.set_title(titles[c], fontsize=11)
            ax.set_xlim(*XLIM); ax.set_ylim(*YLIM)

        cb = fig.colorbar(m, cax=caxs[r])
        cb.set_label(r"$^\circ$C", rotation=90)

    # X coords
    X5  = data5["X"]
    X10 = data10["X"]

    # rows 0-2 use X5; rows 3-5 use X10
    for r in range(6):
        X = X5 if r < 3 else X10
        is_diff = (r % 3 == 2)
        draw_row(r, X, blocks[r], row_titles[r], is_diff=is_diff)

    # axis labels: y only on first col; x only on last row
    for r in range(6):
        axs[r,0].set_ylabel(r"$(y-y_c)/RMW$")
        for c in (1,2):
            axs[r,c].set_ylabel("")
            axs[r,c].tick_params(labelleft=False)

    for c in range(3):
        axs[5,c].set_xlabel(r"$(x-x_c)/RMW$")

    # ticks
    for r in range(6):
        for c in range(3):
            axs[r,c].set_xticks([-4,-2,0,2,4])
            axs[r,c].set_yticks([-4,-2,0,2,4])

    # panel tags
    ROW_TAGS = ["(a)","(b)","(c)","(d)","(e)","(f)"]
    COL_TAGS = ["(i)","(ii)","(iii)"]

    for r in range(6):
        axs[r,0].text(-0.22, 1.03, ROW_TAGS[r],
                      transform=axs[r,0].transAxes,
                      ha="left", va="bottom", fontsize=14)

    for c in range(3):
        axs[0,c].text(0.02, 1.03, COL_TAGS[c],
                      transform=axs[0,c].transAxes,
                      ha="left", va="bottom", fontsize=14)

    if SAVEFIG:
        # EPS 不支持透明度：确保 contour 线/填色都不使用 alpha<1（我们 alpha 只用在 contour 线上）
        # 这不会影响绘制，但如果你不想看到 warning，可以把 LINE_ALPHA=1.0
        fig.savefig(OUTEPS, format="eps", bbox_inches="tight")
        print("saved:", OUTEPS)

    plt.show()

# ============================================================
# RUN
# ============================================================

# ---- TS05 (5 m/s) ----
les_lt_root_5 = "/archive/bgr/Datasets/LES/MoreHurr/TS05_ML10/LT/"
les_st_root_5 = "/archive/bgr/Datasets/LES/MoreHurr/TS05_ML10/ST/"
model_dirs_5 = {
    "hybrid_noLT"         : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/noLT_10m/{y}_results",
    "waveLT_rescale_all"  : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/waveLT_res_10m/{y}_results",
    "waveLT_norescale"    : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/waveLT_nores_10m/{y}_results",
    "shear"               : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/shear_10m/{y}_results",
}

data5 = build_speed(
    tag="TS05_ML10",
    Ustorm=5.0,
    les_lt_root=les_lt_root_5,
    les_st_root=les_st_root_5,
    model_dirs=model_dirs_5,
    use_cache=True,
    overwrite_cache=False
)

# ---- TS10 (10 m/s) ----
les_lt_root_10 = "/archive/bgr/Datasets/LES/MoreHurr/TS10_ML10/LT/"
les_st_root_10 = "/archive/bgr/Datasets/LES/MoreHurr/TS10_ML10/ST/"
model_dirs_10 = {
    "hybrid_noLT"         : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/noLT_10m_10mps/{y}_results",
    "waveLT_rescale_all"  : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/waveLT_res_10m_10mps/{y}_results",
    "waveLT_norescale"    : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/waveLT_nores_10m_10mps/{y}_results",
    "shear"               : "/archive/Qian.Xiao/Qian.Xiao/MOM6_kappa_ePBL/ice/shear_10m_10mps/{y}_results",
}

data10 = build_speed(
    tag="TS10_ML10",
    Ustorm=10.0,
    les_lt_root=les_lt_root_10,
    les_st_root=les_st_root_10,
    model_dirs=model_dirs_10,
    use_cache=True,
    overwrite_cache=False
)

plot_combined(data5, data10)









In [ ]:
##This code is for figure 12 part(a):Sea surface temperature (SST) anomaly relative to the initial temperature $\theta(t,0)-\theta (t=0,0)$ of (left) hybrid ST, (middle) hybrid ST+LT (res) and (right) LT-induced SST increment $\theta_{LT+ST}-\theta_{ST}$. The first three rows correspond to the 5$\rm ms^{-1}$ simulations and the last three rows to the 2$\rm ms^{-1}$. For each translation speed: the top row shows three dimensional simulations, the second row shows the matched 1D column experiments; the third row shows the 3D-1D residuals obtained by the top row minus the second row. The black dashed circle marks 1 RMW. The red dashed line with dots indicates the sampling transect for the vertical temperature profiles analysis in Fig. \ref{fig:tempo3D}. All of the results presented are at day 3 ($t$=72h).

import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import copy
import cmocean

# ============================================================
# Fig2A (5 m/s): 3 rows × 3 cols
#   Row0: 3D snapshot ΔSST (storm-relative)  [noLT, LT, ΔΔ]
#   Row1: 1D mapped to (Y,X) then INTERP to fine grid (visual like 3D)
#   Row2: (3D on same fine grid) - (1D fine grid)
#
# Added:
#   - Panel labels: (a)(b)(c) rows + (i)(ii)(iii) cols, shown as "(a) (i)" per panel (no bold)
#   - Print min/max for each panel (left->right, top->bottom)
# ============================================================

# -------------------------
# 0) USER SETTINGS
# -------------------------
RMW = 50e3
Lx_m, Ly_m = 3.0e6, 1.8e6
lon_min, lon_max = -13.5, 13.5
lat_min, lat_max = -8.1,  8.1

# storm params
U = 5.0
X0_m = 3.432e6
Y0_m = 9.0e5
t_final_day = 3.0
t_final_sec = t_final_day * 86400.0

TI0 = 0
TI_END = -1

# 1D mapping x_ref (非边界位置)
x_ref_m = 2947.5e3

# 1D station design (360 pts)
NUM_TOT = 360
Y0_KM = 900.0
DYPRIME_START = -897.5
DYPRIME_STEP  = 5.0
sid_list   = np.arange(1, NUM_TOT + 1)
yprime_km  = DYPRIME_START + DYPRIME_STEP * (sid_list - 1)
Y_1D       = (yprime_km * 1e3) / RMW   # y/RMW

# plot window
XLIM_COMMON = (-4.0, 16.0)
YLIM_COMMON = (-6.0, 6.0)

# ===== 规则细网格（让 1D / diff 看起来不糊）=====
DX_PLOT = 0.1
DY_PLOT = 0.1
X_plot = np.arange(XLIM_COMMON[0], XLIM_COMMON[1] + 0.5*DX_PLOT, DX_PLOT)
Y_plot = np.arange(YLIM_COMMON[0], YLIM_COMMON[1] + 0.5*DY_PLOT, DY_PLOT)

# 1D 输出时间轴（保持你原来的中心时刻写法）
DT_OUT_SEC = 600.0

# boundary mask (3D plane edges)
STRIP_X = 0.06
STRIP_Y = 0.06

# profile markers
Y_PROFILES = [+1.05, -0.95]
X_PROFILES = [-1.97, +0.03, +2.03, +4.03, +6.03]
PROFILE_LINE_COLOR = "red"
PROFILE_LW = 2.2
PROFILE_MS = 18

# ===== 缓存（强烈建议开）=====
USE_CACHE_1D = True
CACHE_1D_FILE = "cache_1D_sst_360_noLT_LT_dt600.npz"

USE_CACHE_3D = True
CACHE_3D_FILE = "cache_3D_dSST_onXY_5mps_day3_TI0TIEnd_dx0p1_dy0p1.npz"

# -------------------------
# 1) FILE PATHS
# -------------------------
PROG_3D_noLT = "/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp/3D_hurricane/3D_kappa_ePBL_5km/5_ustar0p01_results/20110401.prog.nc"
PROG_3D_LT   = "/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp/3D_hurricane/3D_kappa_ePBL_oneway_5km/5_rescale_ustar0p01_results/20110401.prog.nc"

DIR_1D_BASE = {
    "noLT": "/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp/3D_hurricane/1DVS3D/1D_kappa_ePBL_5mps/noLT_360",
    "LT"  : "/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp/3D_hurricane/1DVS3D/1D_kappa_ePBL_oneway_5mps/oneway_all_360",
}

# -------------------------
# 2) COLORBAR RANGE (全部手动)
# -------------------------
DSST_CLIM = (-5.0,  1.0)
DDSST_CLIM = (-0.6, 0.6)
DSST_DIFF_CLIM = (-1.0, 1.0)
DDSST_DIFF_CLIM = (-0.6, 0.6)

NLEV = 41

# -------------------------
# 3) colormap / norms
# -------------------------
cmap = copy.copy(cmocean.cm.balance)
cmap.set_bad("0.85")

def mk_levels_norm(vmin, vmax, nlev=NLEV):
    levels = np.linspace(vmin, vmax, nlev)
    norm   = mpl.colors.TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
    return levels, norm

lev_ds,  norm_ds  = mk_levels_norm(*DSST_CLIM)
lev_dd,  norm_dd  = mk_levels_norm(*DDSST_CLIM)
lev_dif, norm_dif = mk_levels_norm(*DSST_DIFF_CLIM)
lev_ddf, norm_ddf = mk_levels_norm(*DDSST_DIFF_CLIM)

# -------------------------
# 4) HELPERS
# -------------------------
def annotate_profiles(ax):
    for yy in Y_PROFILES:
        ax.axhline(yy, color=PROFILE_LINE_COLOR, lw=PROFILE_LW, ls="--", zorder=30)
        ax.scatter(X_PROFILES, [yy]*len(X_PROFILES),
                   s=PROFILE_MS, c=PROFILE_LINE_COLOR, zorder=35)

def lonlat_to_meters_1d(lon_1d, lat_1d):
    x_m = (lon_1d - lon_min) * (Lx_m / (lon_max - lon_min))
    y_m = (lat_1d - lat_min) * (Ly_m / (lat_max - lat_min))
    return x_m, y_m

def storm_center_xy(tsec):
    xc = X0_m - U * tsec   # transdir=180
    yc = Y0_m
    return float(xc), float(yc)

def draw_rmw_circle(ax, r=1.0):
    th = np.linspace(0, 2*np.pi, 361)
    ax.plot(r*np.cos(th), r*np.sin(th), "k--", lw=1.2, zorder=10)

def apply_masks_3d(Xn, Yn, field, x_left, x_right, y_bot, y_top):
    mask_edge = (
        (Xn <= (x_left  + STRIP_X)) |
        (Xn >= (x_right - STRIP_X)) |
        (Yn <= (y_bot   + STRIP_Y)) |
        (Yn >= (y_top   - STRIP_Y))
    )
    return np.where(mask_edge, np.nan, field)

def add_xy_m(ds, xname="xh", yname="yh"):
    x_m = (ds[xname] - lon_min) / (lon_max - lon_min) * Lx_m
    y_m = (ds[yname] - lat_min) / (lat_max - lat_min) * Ly_m
    ds2 = ds.assign_coords(
        x_m=(ds[xname].dims, x_m.data),
        y_m=(ds[yname].dims, y_m.data),
    )
    return ds2.swap_dims({xname: "x_m", yname: "y_m"})

def interp_field_to_stormXY(field_yx_m, X, Y, t_snap_day):
    tsec = t_snap_day * 86400.0
    xc, yc = storm_center_xy(tsec)
    X_da = xr.DataArray(np.asarray(X), dims=("X",))
    Y_da = xr.DataArray(np.asarray(Y), dims=("Y",))
    x_tgt = xc + X_da * RMW
    y_tgt = yc + Y_da * RMW
    return field_yx_m.interp(x_m=x_tgt, y_m=y_tgt)

def dyprime_tag(dyprime_km):
    return f"m{abs(dyprime_km):.1f}km" if dyprime_km < 0 else f"p{abs(dyprime_km):.1f}km"

def prog_path(base_dir, sid, dyprime_km):
    sid3 = f"{sid:03d}"
    tag  = dyprime_tag(dyprime_km)
    d = os.path.join(base_dir, f"{sid3}_{tag}_results")
    p1 = os.path.join(d, "prog.nc")
    if os.path.exists(p1):
        return p1
    p2 = os.path.join(d, "prog.nc.gcp")
    if os.path.exists(p2):
        return p2
    raise FileNotFoundError(f"Missing prog file under: {d} (need prog.nc or prog.nc.gcp)")

def load_1d_surface_temp(path_prog, dt_out_sec=DT_OUT_SEC):
    with xr.open_dataset(path_prog, decode_timedelta=False, engine="netcdf4") as ds:
        Nt = ds.sizes["Time"]
        mom_times = (np.arange(Nt) + 0.5) * dt_out_sec / 86400.0
        Tsfc = ds["temp"].isel(zl=0, xh=0, yh=0).values.astype("f4")
    return mom_times, Tsfc

def nanminmax(a):
    a = np.asarray(a)
    return float(np.nanmin(a)), float(np.nanmax(a))

# -------------------------
# 5) LOAD 3D snapshot plane + 3D ΔSST fields for interpolation
# -------------------------
with xr.open_dataset(PROG_3D_noLT, decode_timedelta=False) as ds0, \
     xr.open_dataset(PROG_3D_LT,   decode_timedelta=False) as ds1:

    # storm-relative plane coordinates (day-3)
    xh = ds0["xh"].values
    yh = ds0["yh"].values
    x_m_1d, y_m_1d = lonlat_to_meters_1d(xh, yh)
    X_m, Y_m = np.meshgrid(x_m_1d, y_m_1d, indexing="xy")

    xc3, yc3 = storm_center_xy(t_final_sec)
    Xn = (X_m - xc3) / RMW
    Yn = (Y_m - yc3) / RMW

    x_left  = (0.0  - xc3) / RMW
    x_right = (Lx_m - xc3) / RMW
    y_bot   = (0.0  - yc3) / RMW
    y_top   = (Ly_m - yc3) / RMW

    SST0_noLT = ds0["SST"].isel(Time=TI0).values
    SSTt_noLT = ds0["SST"].isel(Time=TI_END).values
    SST0_LT   = ds1["SST"].isel(Time=TI0).values
    SSTt_LT   = ds1["SST"].isel(Time=TI_END).values

    dSST3D_noLT_plane = apply_masks_3d(Xn, Yn, SSTt_noLT - SST0_noLT, x_left, x_right, y_bot, y_top)
    dSST3D_LT_plane   = apply_masks_3d(Xn, Yn, SSTt_LT   - SST0_LT,   x_left, x_right, y_bot, y_top)
    ddSST3D_plane     = apply_masks_3d(Xn, Yn, (SSTt_LT - SST0_LT) - (SSTt_noLT - SST0_noLT),
                                      x_left, x_right, y_bot, y_top)

    # ΔSST on (y_m, x_m)
    ds0m = add_xy_m(ds0, "xh", "yh")
    ds1m = add_xy_m(ds1, "xh", "yh")
    dSST3D_noLT_m = (ds0m["SST"].isel(Time=TI_END) - ds0m["SST"].isel(Time=TI0))
    dSST3D_LT_m   = (ds1m["SST"].isel(Time=TI_END) - ds1m["SST"].isel(Time=TI0))
    ddSST3D_m     = dSST3D_LT_m - dSST3D_noLT_m

# -------------------------
# 6) LOAD 1D (360) + map to X_1D + interp to fine grid
# -------------------------
if USE_CACHE_1D and os.path.exists(CACHE_1D_FILE):
    z = np.load(CACHE_1D_FILE, allow_pickle=False)
    mom_times_1d = z["mom_times_1d"]
    sst_noLT = z["sst_noLT"]
    sst_LT   = z["sst_LT"]
else:
    p0 = prog_path(DIR_1D_BASE["noLT"], sid_list[0], yprime_km[0])
    mom_times_1d, _ = load_1d_surface_temp(p0, dt_out_sec=DT_OUT_SEC)

    Ny1 = len(sid_list)
    Nx1 = len(mom_times_1d)
    sst_noLT = np.empty((Ny1, Nx1), dtype="f4")
    sst_LT   = np.empty((Ny1, Nx1), dtype="f4")

    for iy, (sid, dy) in enumerate(zip(sid_list, yprime_km)):
        p_no = prog_path(DIR_1D_BASE["noLT"], sid, dy)
        _, Ts = load_1d_surface_temp(p_no, dt_out_sec=DT_OUT_SEC)
        sst_noLT[iy, :] = Ts

        p_lt = prog_path(DIR_1D_BASE["LT"], sid, dy)
        _, Ts = load_1d_surface_temp(p_lt, dt_out_sec=DT_OUT_SEC)
        sst_LT[iy, :] = Ts

        if (iy % 30) == 0:
            print(f"[1D read] {iy+1}/{Ny1}")

    if USE_CACHE_1D:
        np.savez_compressed(CACHE_1D_FILE,
                            mom_times_1d=mom_times_1d,
                            sst_noLT=sst_noLT,
                            sst_LT=sst_LT)

# time->x mapping
t_entry_days = (X0_m - x_ref_m) / U / 86400.0
X_1D = (U * (mom_times_1d - t_entry_days) * 86400.0) / RMW

# ΔSST on native grid
dSST1D_noLT = sst_noLT - sst_noLT[:, [0]]
dSST1D_LT   = sst_LT   - sst_LT[:,   [0]]
ddSST1D     = dSST1D_LT - dSST1D_noLT

dSST1D_noLT_da = xr.DataArray(dSST1D_noLT, dims=("Y","X"), coords={"Y":Y_1D, "X":X_1D})
dSST1D_LT_da   = xr.DataArray(dSST1D_LT,   dims=("Y","X"), coords={"Y":Y_1D, "X":X_1D})
ddSST1D_da     = xr.DataArray(ddSST1D,     dims=("Y","X"), coords={"Y":Y_1D, "X":X_1D})

# interp to plotting grid (visual sharpening)
dSST1D_noLT_f = dSST1D_noLT_da.interp(X=X_plot, Y=Y_plot)
dSST1D_LT_f   = dSST1D_LT_da.interp(X=X_plot, Y=Y_plot)
ddSST1D_f     = ddSST1D_da.interp(X=X_plot, Y=Y_plot)

# -------------------------
# 7) 3D interp to same plotting grid (WITH CACHE)
# -------------------------
def _cache3d_ok(cachefile, X_plot, Y_plot, meta):
    try:
        z = np.load(cachefile, allow_pickle=True)
        if "X_plot" not in z or "Y_plot" not in z:
            return False
        if z["X_plot"].shape != X_plot.shape or z["Y_plot"].shape != Y_plot.shape:
            return False
        if not np.allclose(z["X_plot"], X_plot) or not np.allclose(z["Y_plot"], Y_plot):
            return False
        if "meta" not in z:
            return False
        meta0 = z["meta"].item()
        for k, v in meta.items():
            if k not in meta0:
                return False
            if isinstance(v, float):
                if abs(meta0[k] - v) > 1e-12:
                    return False
            else:
                if meta0[k] != v:
                    return False
        return True
    except Exception:
        return False

meta3d = dict(
    U=float(U), X0_m=float(X0_m), Y0_m=float(Y0_m),
    t_final_day=float(t_final_day),
    TI0=int(TI0), TI_END=int(TI_END),
    RMW=float(RMW), Lx_m=float(Lx_m), Ly_m=float(Ly_m),
    lon_min=float(lon_min), lon_max=float(lon_max),
    lat_min=float(lat_min), lat_max=float(lat_max)
)

if USE_CACHE_3D and os.path.exists(CACHE_3D_FILE) and _cache3d_ok(CACHE_3D_FILE, X_plot, Y_plot, meta3d):
    z = np.load(CACHE_3D_FILE, allow_pickle=True)
    dSST3D_noLT_on = xr.DataArray(z["dSST3D_noLT_on"], dims=("Y","X"), coords={"Y":Y_plot, "X":X_plot})
    dSST3D_LT_on   = xr.DataArray(z["dSST3D_LT_on"],   dims=("Y","X"), coords={"Y":Y_plot, "X":X_plot})
    ddSST3D_on     = xr.DataArray(z["ddSST3D_on"],     dims=("Y","X"), coords={"Y":Y_plot, "X":X_plot})
    print("[3D cache] loaded:", CACHE_3D_FILE)
else:
    print("[3D cache] computing 3D interp onto (X_plot,Y_plot)...")
    dSST3D_noLT_on = interp_field_to_stormXY(dSST3D_noLT_m, X_plot, Y_plot, t_snap_day=t_final_day)
    dSST3D_LT_on   = interp_field_to_stormXY(dSST3D_LT_m,   X_plot, Y_plot, t_snap_day=t_final_day)
    ddSST3D_on     = interp_field_to_stormXY(ddSST3D_m,     X_plot, Y_plot, t_snap_day=t_final_day)

    if USE_CACHE_3D:
        np.savez_compressed(
            CACHE_3D_FILE,
            X_plot=X_plot, Y_plot=Y_plot,
            dSST3D_noLT_on=dSST3D_noLT_on.values.astype("f4"),
            dSST3D_LT_on=dSST3D_LT_on.values.astype("f4"),
            ddSST3D_on=ddSST3D_on.values.astype("f4"),
            meta=meta3d
        )
        print("[3D cache] saved:", CACHE_3D_FILE)

# 3D-1D on fine grid
dSST_3Dm1D_noLT = dSST3D_noLT_on - dSST1D_noLT_f
dSST_3Dm1D_LT   = dSST3D_LT_on   - dSST1D_LT_f
ddSST_3Dm1D     = ddSST3D_on     - ddSST1D_f

# -------------------------
# 7.5) PRINT min/max (left->right, top->bottom)
# -------------------------
panel_data = [
    ("(a) (i)    3D ΔSST_noLT", dSST3D_noLT_plane),
    ("(a) (ii)   3D ΔSST_LT",   dSST3D_LT_plane),
    ("(a) (iii)  3D ΔΔSST",     ddSST3D_plane),

    ("(b) (i)    1D ΔSST_noLT", dSST1D_noLT_f.values),
    ("(b) (ii)   1D ΔSST_LT",   dSST1D_LT_f.values),
    ("(b) (iii)  1D ΔΔSST",     ddSST1D_f.values),

    ("(c) (i)    (3D-1D) ΔSST_noLT", dSST_3Dm1D_noLT.values),
    ("(c) (ii)   (3D-1D) ΔSST_LT",   dSST_3Dm1D_LT.values),
    ("(c) (iii)  (3D-1D) ΔΔSST",     ddSST_3Dm1D.values),
]

print("\n===== Panel min/max (left→right, top→bottom) =====")
for name, arr in panel_data:
    mn, mx = nanminmax(arr)
    print(f"{name:30s}  min = {mn:10.4f}   max = {mx:10.4f}")
print("=================================================\n")

# -------------------------
# 8) PLOT
# -------------------------
fig = plt.figure(figsize=(15.0, 9.2))
# gs = GridSpec(
#     nrows=3, ncols=6, figure=fig,
#     width_ratios=[1.0, 0.045, 1.0, 0.045, 1.0, 0.045],
#     wspace=0.08, hspace=0.30
# )

gs = GridSpec(
    nrows=3, ncols=6, figure=fig,
    width_ratios=[0.6, 0.028, 0.6, 0.028, 0.6, 0.028],
    wspace=0.35, hspace=0.25
)
axs  = np.empty((3,3), dtype=object)
caxs = np.empty((3,3), dtype=object)
for i in range(3):
    axs[i,0]  = fig.add_subplot(gs[i,0]); caxs[i,0] = fig.add_subplot(gs[i,1])
    axs[i,1]  = fig.add_subplot(gs[i,2]); caxs[i,1] = fig.add_subplot(gs[i,3])
    axs[i,2]  = fig.add_subplot(gs[i,4]); caxs[i,2] = fig.add_subplot(gs[i,5])

row_titles = ["3D (snapshot ΔSST)", "1D (interp to fine grid)", "3D − 1D (fine grid)"]

# Row0: 3D plane
m00 = axs[0,0].contourf(Xn, Yn, np.ma.masked_invalid(dSST3D_noLT_plane),
                        levels=lev_ds, cmap=cmap, norm=norm_ds, extend="both")
m01 = axs[0,1].contourf(Xn, Yn, np.ma.masked_invalid(dSST3D_LT_plane),
                        levels=lev_ds, cmap=cmap, norm=norm_ds, extend="both")
m02 = axs[0,2].contourf(Xn, Yn, np.ma.masked_invalid(ddSST3D_plane),
                        levels=lev_dd, cmap=cmap, norm=norm_dd, extend="both")

for j in range(3):
    draw_rmw_circle(axs[0,j], 1.0)
    annotate_profiles(axs[0,j])

cb = fig.colorbar(m00, cax=caxs[0,0], orientation="vertical"); cb.set_label(r"$\Delta SST_{\mathrm{noLT}}$ (°C)", fontsize=12)
cb = fig.colorbar(m01, cax=caxs[0,1], orientation="vertical"); cb.set_label(r"$\Delta SST_{\mathrm{LT}}$ (°C)", fontsize=12)
cb = fig.colorbar(m02, cax=caxs[0,2], orientation="vertical"); cb.set_label(r"$\Delta\Delta SST$ (°C)", fontsize=12)

# Row1: 1D fine grid (contourf like 3D)
m10 = axs[1,0].contourf(X_plot, Y_plot, dSST1D_noLT_f.values, levels=lev_ds, cmap=cmap, norm=norm_ds, extend="both")
m11 = axs[1,1].contourf(X_plot, Y_plot, dSST1D_LT_f.values,   levels=lev_ds, cmap=cmap, norm=norm_ds, extend="both")
m12 = axs[1,2].contourf(X_plot, Y_plot, ddSST1D_f.values,     levels=lev_dd, cmap=cmap, norm=norm_dd, extend="both")

cb = fig.colorbar(m10, cax=caxs[1,0], orientation="vertical"); cb.set_label(r"$\Delta SST_{\mathrm{noLT}}$ (°C)", fontsize=12)
cb = fig.colorbar(m11, cax=caxs[1,1], orientation="vertical"); cb.set_label(r"$\Delta SST_{\mathrm{LT}}$ (°C)", fontsize=12)
cb = fig.colorbar(m12, cax=caxs[1,2], orientation="vertical"); cb.set_label(r"$\Delta\Delta SST$ (°C)", fontsize=12)

# Row2: 3D-1D fine grid
m20 = axs[2,0].contourf(X_plot, Y_plot, dSST_3Dm1D_noLT.values, levels=lev_dif, cmap=cmap, norm=norm_dif, extend="both")
m21 = axs[2,1].contourf(X_plot, Y_plot, dSST_3Dm1D_LT.values,   levels=lev_dif, cmap=cmap, norm=norm_dif, extend="both")
m22 = axs[2,2].contourf(X_plot, Y_plot, ddSST_3Dm1D.values,     levels=lev_ddf, cmap=cmap, norm=norm_ddf, extend="both")

cb = fig.colorbar(m20, cax=caxs[2,0], orientation="vertical"); cb.set_label(r"$(\Delta SST)_{3D-1D}$ (°C)", fontsize=12)
cb = fig.colorbar(m21, cax=caxs[2,1], orientation="vertical"); cb.set_label(r"$(\Delta SST)_{3D-1D}$ (°C)", fontsize=12)
cb = fig.colorbar(m22, cax=caxs[2,2], orientation="vertical"); cb.set_label(r"$(\Delta\Delta SST)_{3D-1D}$ (°C)", fontsize=12)

# Cosmetics
for i in range(3):
    # axs[i,0].text(-0.45, 0.5, row_titles[i], transform=axs[i,0].transAxes,
    #               rotation=90, va="center", ha="center", fontsize=12)
    for j in range(3):
        ax = axs[i,j]
        ax.set_xlim(*XLIM_COMMON)
        ax.set_ylim(*YLIM_COMMON)
        ax.set_aspect("equal", adjustable="box")
        ax.tick_params(labelsize=10, length=3)
        ax.set_xlabel(r"$(x-x_c)/RMW$", fontsize=12)
        if j == 0:
            ax.set_ylabel(r"$(y-y_c)/RMW$", fontsize=12)
        else:
            ax.set_yticklabels([])

            
for i in range(3):
    for j in range(3):
        cbax = caxs[i,j]
        cbax.yaxis.set_ticks_position("right")
        cbax.yaxis.set_label_position("right")
        cbax.tick_params(length=2, width=0.8, labelsize=10, pad=1)
        cbax.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter("%.1f"))

# -------------------------
# 8.5) Panel labels: (a)(b)(c) rows + (i)(ii)(iii) cols
#     Show as "(a) (i)" in each panel (no bold)
# -------------------------
row_tags = ["(a)", "(b)", "(c)"]
col_tags = ["(i)", "(ii)", "(iii)"]

for i in range(3):
    axs[i,0].text(-0.22, 1.03, row_tags[i],
                  transform=axs[i,0].transAxes,
                  ha="left", va="bottom",
                  fontsize=14)

# 列标：第一行每列标一次
for j in range(3):
    axs[0,j].text(0.02, 1.03, col_tags[j],
                  transform=axs[0,j].transAxes,
                  ha="left", va="bottom",
                  fontsize=14)

plt.show()
# fig.savefig("Fig2A_5mps_3D_1D_3Dm1D_interp_cached_labeled.png", dpi=300, bbox_inches="tight")





In [ ]:
###This code is for figure 12 part(b):Sea surface temperature (SST) anomaly relative to the initial temperature $\theta(t,0)-\theta (t=0,0)$ of (left) hybrid ST, (middle) hybrid ST+LT (res) and (right) LT-induced SST increment $\theta_{LT+ST}-\theta_{ST}$. The first three rows correspond to the 5$\rm ms^{-1}$ simulations and the last three rows to the 2$\rm ms^{-1}$. For each translation speed: the top row shows three dimensional simulations, the second row shows the matched 1D column experiments; the third row shows the 3D-1D residuals obtained by the top row minus the second row. The black dashed circle marks 1 RMW. The red dashed line with dots indicates the sampling transect for the vertical temperature profiles analysis in Fig. \ref{fig:tempo3D}. All of the results presented are at day 3 ($t$=72h).

import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import copy
import cmocean

# ============================================================
# Fig2A (2 m/s): 3 rows × 3 cols
#   Row0: 3D snapshot ΔSST (storm-relative)  [noLT, LT, ΔΔ]
#   Row1: 1D (360 stations) mapped to (Y,X) then INTERP to REGULAR grid
#   Row2: (3D on same regular grid, masked for X>6.3) - (1D regular grid)
#
# Requirements:
#   - Use regular grid (X_plot, Y_plot) + contourf
#   - Tighten layout (same figsize & axis limits as your 2mps script)
#   - Print min/max for each panel (L->R, top->bottom)
#   - Label rows: (a)(b)(c); columns: (i)(ii)(iii)
#   - 1D: directly load your existing cache if present
# ============================================================

# -------------------------
# 0) USER SETTINGS (2 m/s)  (保持你原来的范围/参数)
# -------------------------
RMW = 50e3
Lx_m, Ly_m = 3.0e6, 1.8e6
lon_min, lon_max = -13.5, 13.5
lat_min, lat_max = -8.1,  8.1

U = 2.0
X0_m = 3.1728e6
Y0_m = 9.0e5
transdir = np.deg2rad(180.0)
t0_sec = 0.0
t_final_day = 3.0
t_final_sec = t_final_day * 86400.0

STRIP_X = 0.06
STRIP_Y = 0.06
XCUT_3D = 6.3

XLIM_COMMON = (-4.0, 6.0)   # 不改
YLIM_COMMON = (-6.0, 6.0)   # 不改

# ===== 规则网格（你要求的）=====
DX_PLOT = 0.1
DY_PLOT = 0.1
X_plot = np.arange(XLIM_COMMON[0], XLIM_COMMON[1] + 0.5*DX_PLOT, DX_PLOT)
Y_plot = np.arange(YLIM_COMMON[0], YLIM_COMMON[1] + 0.5*DY_PLOT, DY_PLOT)

# Fig2B markers (storm-relative)
Y_PROFILES = [+1.05, -0.95]
X_PROFILES = [-1.97, +0.03, +2.03, +4.03]
PROFILE_LINE_COLOR = "red"
PROFILE_LW = 2.6
PROFILE_DOT_S = 26

def annotate_profiles(ax):
    for yy in Y_PROFILES:
        ax.axhline(yy, color=PROFILE_LINE_COLOR, lw=PROFILE_LW, ls="--", zorder=40)
        ax.scatter(X_PROFILES, [yy]*len(X_PROFILES),
                   s=PROFILE_DOT_S, c=PROFILE_LINE_COLOR, zorder=45)

# -------------------------
# 1) FILE PATHS (2 m/s)
# -------------------------
PROG_3D_noLT = "/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp/3D_hurricane/3D_kappa_ePBL_5km_2mps/5_results/20110401.prog.nc"
PROG_3D_LT   = "/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp/3D_hurricane/3D_kappa_ePBL_oneway_5km_2mps/5_results/20110401.prog.nc"

DIR_1D_BASE = {
    "noLT": "/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp/3D_hurricane/1DVS3D/1D_kappa_ePBL_2mps/noLT_360",
    "LT"  : "/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp/3D_hurricane/1DVS3D/1D_kappa_ePBL_oneway_2mps/oneway_all_360",
}

# -------------------------
# 2) 1D station design (360 pts) —— 按 5mps 那种命名
# -------------------------
NUM_TOT = 360
DYPRIME_START = -897.5
DYPRIME_STEP  = 5.0
sid_list   = np.arange(1, NUM_TOT + 1)
yprime_km  = DYPRIME_START + DYPRIME_STEP * (sid_list - 1)
Y_1D       = (yprime_km * 1e3) / RMW   # y/RMW

# 2mps 的 1D mapping x_ref（保持你的设置）
x_ref_m = 2947.5e3

# -------------------------
# 3) colormap
# -------------------------
cmap = copy.copy(cmocean.cm.balance)
cmap.set_bad("0.85")

# -------------------------
# 4) helpers
# -------------------------
def lonlat_to_meters_1d(lon_1d, lat_1d):
    x_m = (lon_1d - lon_min) * (Lx_m / (lon_max - lon_min))
    y_m = (lat_1d - lat_min) * (Ly_m / (lat_max - lat_min))
    return x_m, y_m

def storm_center_xy(tsec, X0, Y0, V, transdir, t0_sec):
    xc = X0 + (tsec - t0_sec) * V * np.cos(transdir)
    yc = Y0 + (tsec - t0_sec) * V * np.sin(transdir)
    return float(xc), float(yc)

def draw_rmw_circle(ax, r=1.0):
    th = np.linspace(0, 2*np.pi, 361)
    ax.plot(r*np.cos(th), r*np.sin(th), "k--", lw=1.2, zorder=20)

def mom_time_days_from_size(Nt, dt_out_sec):
    return (np.arange(Nt) + 0.5) * dt_out_sec / 86400.0

def pick_time_index_near_day(ds, t_day, dt_out_sec):
    tdays = mom_time_days_from_size(ds.Time.size, dt_out_sec)
    return int(np.argmin(np.abs(tdays - t_day)))

def apply_masks_3d(Xn, Yn, field, x_left, x_right, y_bot, y_top, xcut=None):
    mask_edge = (
        (Xn <= (x_left  + STRIP_X)) |
        (Xn >= (x_right - STRIP_X)) |
        (Yn <= (y_bot   + STRIP_Y)) |
        (Yn >= (y_top   - STRIP_Y))
    )
    if xcut is not None:
        mask_edge = mask_edge | (Xn > xcut)
    return np.where(mask_edge, np.nan, field)

def add_xy_m(ds, xname="xh", yname="yh"):
    x_m = (ds[xname] - lon_min) / (lon_max - lon_min) * Lx_m
    y_m = (ds[yname] - lat_min) / (lat_max - lat_min) * Ly_m
    ds2 = ds.assign_coords(
        x_m=(ds[xname].dims, x_m.data),
        y_m=(ds[yname].dims, y_m.data),
    )
    return ds2.swap_dims({xname: "x_m", yname: "y_m"})

def interp_field_to_stormXY(field_yx_m, X, Y, t_snap_day):
    """
    field_yx_m: DataArray dims (y_m, x_m)
    """
    tsec = t_snap_day * 86400.0
    # transdir=180deg => xc = X0 - U*t ; yc = Y0
    xc = X0_m - U * tsec
    yc = Y0_m

    X_da = xr.DataArray(np.asarray(X), dims=("X",))
    Y_da = xr.DataArray(np.asarray(Y), dims=("Y",))
    x_tgt = xc + X_da * RMW
    y_tgt = yc + Y_da * RMW
    return field_yx_m.interp(x_m=x_tgt, y_m=y_tgt)

# ===== 1D file naming like 5mps =====
def dyprime_tag(dyprime_km):
    return f"m{abs(dyprime_km):.1f}km" if dyprime_km < 0 else f"p{abs(dyprime_km):.1f}km"

def prog_path(base_dir, sid, dyprime_km):
    sid3 = f"{sid:03d}"
    tag  = dyprime_tag(dyprime_km)
    d = os.path.join(base_dir, f"{sid3}_{tag}_results")
    p1 = os.path.join(d, "prog.nc")
    if os.path.exists(p1):
        return p1
    p2 = os.path.join(d, "prog.nc.gcp")
    if os.path.exists(p2):
        return p2
    raise FileNotFoundError(f"Missing prog file under: {d} (need prog.nc or prog.nc.gcp)")

def load_1d_surface_temp(path_prog, dt_out_sec, engine="netcdf4"):
    with xr.open_dataset(path_prog, decode_timedelta=False, engine=engine) as ds:
        Nt = ds.sizes["Time"]
        mom_times = mom_time_days_from_size(Nt, dt_out_sec)
        Tsfc = ds["temp"].isel(zl=0, xh=0, yh=0).values.astype("f4")
    return mom_times, Tsfc

def ensure_monotonic_x(X, arr_yx):
    """
    xarray.interp 要求坐标单调递增；如果 X 不是递增，排序一下并重排数据。
    arr_yx dims: (Y, X)
    """
    X = np.asarray(X)
    if np.all(np.diff(X) > 0):
        return X, arr_yx
    idx = np.argsort(X)
    return X[idx], arr_yx[:, idx]

def nanminmax(a):
    a = np.asarray(a)
    return float(np.nanmin(a)), float(np.nanmax(a))

# -------------------------
# 5) LOAD 3D (plane) and compute ΔSST snapshot @ day=3
# -------------------------
DT_OUT_3D_SEC = 1800.0  # 3D 维持你原脚本的 1800s

with xr.open_dataset(PROG_3D_noLT, decode_timedelta=False) as ds0, \
     xr.open_dataset(PROG_3D_LT,   decode_timedelta=False) as ds1:

    TI0 = 0
    TI_END0 = pick_time_index_near_day(ds0, t_final_day, DT_OUT_3D_SEC)
    TI_END1 = pick_time_index_near_day(ds1, t_final_day, DT_OUT_3D_SEC)

    xh = ds0["xh"].values
    yh = ds0["yh"].values
    x_m_1d, y_m_1d = lonlat_to_meters_1d(xh, yh)
    X_m, Y_m = np.meshgrid(x_m_1d, y_m_1d, indexing="xy")

    xc3, yc3 = storm_center_xy(t_final_sec, X0_m, Y0_m, U, transdir, t0_sec)
    Xn = (X_m - xc3) / RMW
    Yn = (Y_m - yc3) / RMW

    x_left  = (0.0  - xc3) / RMW
    x_right = (Lx_m - xc3) / RMW
    y_bot   = (0.0  - yc3) / RMW
    y_top   = (Ly_m - yc3) / RMW

    SST0_noLT = ds0["SST"].isel(Time=TI0).values
    SSTt_noLT = ds0["SST"].isel(Time=TI_END0).values
    SST0_LT   = ds1["SST"].isel(Time=TI0).values
    SSTt_LT   = ds1["SST"].isel(Time=TI_END1).values

    dSST3D_noLT_plane = apply_masks_3d(Xn, Yn, SSTt_noLT - SST0_noLT, x_left, x_right, y_bot, y_top, xcut=XCUT_3D)
    dSST3D_LT_plane   = apply_masks_3d(Xn, Yn, SSTt_LT   - SST0_LT,   x_left, x_right, y_bot, y_top, xcut=XCUT_3D)
    ddSST3D_plane     = apply_masks_3d(Xn, Yn, (SSTt_LT - SST0_LT) - (SSTt_noLT - SST0_noLT),
                                      x_left, x_right, y_bot, y_top, xcut=XCUT_3D)

# -------------------------
# 6) LOAD 1D (360) from your existing cache (preferred)
# -------------------------
DT_OUT_1D_SEC = 600.0  # 你说现在都是 600s
USE_CACHE_1D  = True
CACHE_1D_FILE = "cache_1D_sst_360_2mps_noLT_LT_dt1800.npz"  # 你之前保存的文件名（按你要求不改）

if USE_CACHE_1D and os.path.exists(CACHE_1D_FILE):
    z = np.load(CACHE_1D_FILE, allow_pickle=False)
    mom_times_1d = z["mom_times_1d"]   # (Time,) days
    sst_noLT     = z["sst_noLT"]       # (360, Time)
    sst_LT       = z["sst_LT"]         # (360, Time)
    print("[1D cache] loaded:", CACHE_1D_FILE, "  shape:", sst_noLT.shape, sst_LT.shape)
else:
    # 如果 cache 不在，就按 600s 重建（但仍保存为你给的文件名）
    p0 = prog_path(DIR_1D_BASE["noLT"], sid_list[0], yprime_km[0])
    mom_times_1d, _ = load_1d_surface_temp(p0, dt_out_sec=DT_OUT_1D_SEC)

    Ny1 = len(sid_list)
    Nx1 = len(mom_times_1d)
    sst_noLT = np.empty((Ny1, Nx1), dtype="f4")
    sst_LT   = np.empty((Ny1, Nx1), dtype="f4")

    for iy, (sid, dy) in enumerate(zip(sid_list, yprime_km)):
        p_no = prog_path(DIR_1D_BASE["noLT"], sid, dy)
        _, Ts = load_1d_surface_temp(p_no, dt_out_sec=DT_OUT_1D_SEC)
        sst_noLT[iy, :] = Ts

        p_lt = prog_path(DIR_1D_BASE["LT"], sid, dy)
        _, Ts = load_1d_surface_temp(p_lt, dt_out_sec=DT_OUT_1D_SEC)
        sst_LT[iy, :] = Ts

        if (iy % 40) == 0:
            print(f"[1D read] {iy+1}/{Ny1}")

    np.savez_compressed(CACHE_1D_FILE,
                        mom_times_1d=mom_times_1d.astype("f8"),
                        sst_noLT=sst_noLT,
                        sst_LT=sst_LT)
    print("[1D cache] saved:", CACHE_1D_FILE)

# time->x mapping (保持你 2mps 的 x_ref_m / X0_m / U 逻辑)
t_entry_days = (X0_m - x_ref_m) / U / 86400.0
X_1D = (U * (mom_times_1d - t_entry_days) * 86400.0) / RMW

# ΔSST relative to initial
dSST1D_noLT = sst_noLT - sst_noLT[:, [0]]
dSST1D_LT   = sst_LT   - sst_LT[:,   [0]]
ddSST1D     = dSST1D_LT - dSST1D_noLT

# ensure X_1D monotonic for interp
X_1D_sorted, dSST1D_noLT = ensure_monotonic_x(X_1D, dSST1D_noLT)
_,          dSST1D_LT    = ensure_monotonic_x(X_1D, dSST1D_LT)
_,          ddSST1D      = ensure_monotonic_x(X_1D, ddSST1D)

# map to DataArray then interp to RULE GRID
dSST1D_noLT_da = xr.DataArray(dSST1D_noLT, dims=("Y","X"), coords={"Y":Y_1D, "X":X_1D_sorted})
dSST1D_LT_da   = xr.DataArray(dSST1D_LT,   dims=("Y","X"), coords={"Y":Y_1D, "X":X_1D_sorted})
ddSST1D_da     = xr.DataArray(ddSST1D,     dims=("Y","X"), coords={"Y":Y_1D, "X":X_1D_sorted})

dSST1D_noLT_f = dSST1D_noLT_da.interp(Y=Y_plot, X=X_plot)
dSST1D_LT_f   = dSST1D_LT_da.interp(Y=Y_plot, X=X_plot)
ddSST1D_f     = ddSST1D_da.interp(Y=Y_plot, X=X_plot)

# -------------------------
# 7) 3D interp to same RULE GRID + mask for X>XCUT_3D, then 3D-1D
# -------------------------
with xr.open_dataset(PROG_3D_noLT, decode_timedelta=False) as ds0, \
     xr.open_dataset(PROG_3D_LT,   decode_timedelta=False) as ds1:

    TI0 = 0
    TI_END0 = pick_time_index_near_day(ds0, t_final_day, DT_OUT_3D_SEC)
    TI_END1 = pick_time_index_near_day(ds1, t_final_day, DT_OUT_3D_SEC)

    ds0m = add_xy_m(ds0, "xh", "yh")
    ds1m = add_xy_m(ds1, "xh", "yh")

    dSST3D_noLT_m = (ds0m["SST"].isel(Time=TI_END0) - ds0m["SST"].isel(Time=TI0))
    dSST3D_LT_m   = (ds1m["SST"].isel(Time=TI_END1) - ds1m["SST"].isel(Time=TI0))
    ddSST3D_m     = dSST3D_LT_m - dSST3D_noLT_m

    dSST3D_noLT_onYX = interp_field_to_stormXY(dSST3D_noLT_m, X_plot, Y_plot, t_snap_day=t_final_day)
    dSST3D_LT_onYX   = interp_field_to_stormXY(dSST3D_LT_m,   X_plot, Y_plot, t_snap_day=t_final_day)
    ddSST3D_onYX     = interp_field_to_stormXY(ddSST3D_m,     X_plot, Y_plot, t_snap_day=t_final_day)

# mask 3D on rule grid for X > XCUT_3D (only affects Row2)
maskX = xr.DataArray(np.asarray(X_plot) > XCUT_3D, dims=("X",), coords={"X":X_plot})
dSST3D_noLT_onYX = dSST3D_noLT_onYX.where(~maskX)
dSST3D_LT_onYX   = dSST3D_LT_onYX.where(~maskX)
ddSST3D_onYX     = ddSST3D_onYX.where(~maskX)

dSST_3Dm1D_noLT = dSST3D_noLT_onYX - dSST1D_noLT_f
dSST_3Dm1D_LT   = dSST3D_LT_onYX   - dSST1D_LT_f
ddSST_3Dm1D     = ddSST3D_onYX     - ddSST1D_f

# -------------------------
# 7.5) PRINT min/max (左到右，上到下)
# -------------------------
panel_data = [
    ("(a)(i)   3D ΔSST_noLT", dSST3D_noLT_plane),
    ("(a)(ii)  3D ΔSST_LT",   dSST3D_LT_plane),
    ("(a)(iii) 3D ΔΔSST",     ddSST3D_plane),

    ("(b)(i)   1D ΔSST_noLT", dSST1D_noLT_f.values),
    ("(b)(ii)  1D ΔSST_LT",   dSST1D_LT_f.values),
    ("(b)(iii) 1D ΔΔSST",     ddSST1D_f.values),

    ("(c)(i)   (3D-1D) ΔSST_noLT", dSST_3Dm1D_noLT.values),
    ("(c)(ii)  (3D-1D) ΔSST_LT",   dSST_3Dm1D_LT.values),
    ("(c)(iii) (3D-1D) ΔΔSST",     ddSST_3Dm1D.values),
]

print("\n===== Panel min/max (left→right, top→bottom) =====")
for name, arr in panel_data:
    mn, mx = nanminmax(arr)
    print(f"{name:28s}  min = {mn:9.4f}   max = {mx:9.4f}")
print("=================================================\n")

# -------------------------
# 8) PLOT (保持你 2mps 原来的 figsize/layout + 更紧一些)
# -------------------------
DSST_CLIM_3D   = (-9.0,  1.0)
DSST_CLIM_1D   = (-6.0,  1.0)
DSST_CLIM_DIFF = (-5.0,  5.0)

DDSST_CLIM_3D   = (-0.9, 0.9)
DDSST_CLIM_1D   = (-0.9, 0.9)
DDSST_CLIM_DIFF = (-0.6, 0.6)

NLEV = 21
def mk_levels_norm(vmin, vmax):
    levels = np.linspace(vmin, vmax, NLEV)
    norm   = mpl.colors.TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
    return levels, norm

levels_r0_ds, norm_r0_ds = mk_levels_norm(*DSST_CLIM_3D)
levels_r1_ds, norm_r1_ds = mk_levels_norm(*DSST_CLIM_1D)
levels_r2_ds, norm_r2_ds = mk_levels_norm(*DSST_CLIM_DIFF)

levels_r0_dd, norm_r0_dd = mk_levels_norm(*DDSST_CLIM_3D)
levels_r1_dd, norm_r1_dd = mk_levels_norm(*DDSST_CLIM_1D)
levels_r2_dd, norm_r2_dd = mk_levels_norm(*DDSST_CLIM_DIFF)

fig = plt.figure(figsize=(15.0, 9.2))  # 不改

# 更紧：减小 colorbar 宽度 + wspace/hspace

gs = GridSpec(
    nrows=3, ncols=6, figure=fig,
    width_ratios=[0.6, 0.028, 0.6, 0.028, 0.6, 0.028],
    wspace=0.35, hspace=0.25
)

axs  = np.empty((3,3), dtype=object)
caxs = np.empty((3,3), dtype=object)
for i in range(3):
    axs[i,0]  = fig.add_subplot(gs[i,0]); caxs[i,0] = fig.add_subplot(gs[i,1])
    axs[i,1]  = fig.add_subplot(gs[i,2]); caxs[i,1] = fig.add_subplot(gs[i,3])
    axs[i,2]  = fig.add_subplot(gs[i,4]); caxs[i,2] = fig.add_subplot(gs[i,5])

# Row0: 3D plane
m00 = axs[0,0].contourf(Xn, Yn, np.ma.masked_invalid(dSST3D_noLT_plane),
                        levels=levels_r0_ds, cmap=cmap, norm=norm_r0_ds, extend="both")
m01 = axs[0,1].contourf(Xn, Yn, np.ma.masked_invalid(dSST3D_LT_plane),
                        levels=levels_r0_ds, cmap=cmap, norm=norm_r0_ds, extend="both")
m02 = axs[0,2].contourf(Xn, Yn, np.ma.masked_invalid(ddSST3D_plane),
                        levels=levels_r0_dd, cmap=cmap, norm=norm_r0_dd, extend="both")

for j in range(3):
    draw_rmw_circle(axs[0,j], 1.0)
    annotate_profiles(axs[0,j])

cb = fig.colorbar(m00, cax=caxs[0,0], orientation="vertical", extendrect=True)
cb.set_label(r"$\Delta SST_{\mathrm{noLT}}$ (°C)", fontsize=12, labelpad=4)
cb = fig.colorbar(m01, cax=caxs[0,1], orientation="vertical", extendrect=True)
cb.set_label(r"$\Delta SST_{\mathrm{LT}}$ (°C)", fontsize=12, labelpad=4)
cb = fig.colorbar(m02, cax=caxs[0,2], orientation="vertical", extendrect=True)
cb.set_label(r"$\Delta\Delta SST$ (°C)", fontsize=12, labelpad=4)

# Row1: 1D rule grid (contourf)
m10 = axs[1,0].contourf(X_plot, Y_plot, np.ma.masked_invalid(dSST1D_noLT_f.values),
                        levels=levels_r1_ds, cmap=cmap, norm=norm_r1_ds, extend="both")
m11 = axs[1,1].contourf(X_plot, Y_plot, np.ma.masked_invalid(dSST1D_LT_f.values),
                        levels=levels_r1_ds, cmap=cmap, norm=norm_r1_ds, extend="both")
m12 = axs[1,2].contourf(X_plot, Y_plot, np.ma.masked_invalid(ddSST1D_f.values),
                        levels=levels_r1_dd, cmap=cmap, norm=norm_r1_dd, extend="both")

cb = fig.colorbar(m10, cax=caxs[1,0], orientation="vertical", extendrect=True)
cb.set_label(r"$\Delta SST_{\mathrm{noLT}}$ (°C)", fontsize=12, labelpad=4)
cb = fig.colorbar(m11, cax=caxs[1,1], orientation="vertical", extendrect=True)
cb.set_label(r"$\Delta SST_{\mathrm{LT}}$ (°C)", fontsize=12, labelpad=4)
cb = fig.colorbar(m12, cax=caxs[1,2], orientation="vertical", extendrect=True)
cb.set_label(r"$\Delta\Delta SST$ (°C)", fontsize=12, labelpad=4)

# Row2: 3D-1D rule grid
m20 = axs[2,0].contourf(X_plot, Y_plot, np.ma.masked_invalid(dSST_3Dm1D_noLT.values),
                        levels=levels_r2_ds, cmap=cmap, norm=norm_r2_ds, extend="both")
m21 = axs[2,1].contourf(X_plot, Y_plot, np.ma.masked_invalid(dSST_3Dm1D_LT.values),
                        levels=levels_r2_ds, cmap=cmap, norm=norm_r2_ds, extend="both")
m22 = axs[2,2].contourf(X_plot, Y_plot, np.ma.masked_invalid(ddSST_3Dm1D.values),
                        levels=levels_r2_dd, cmap=cmap, norm=norm_r2_dd, extend="both")

cb = fig.colorbar(m20, cax=caxs[2,0], orientation="vertical", extendrect=True)
cb.set_label(r"$(\Delta SST)_{3D-1D}$ (°C)", fontsize=12, labelpad=4)
cb = fig.colorbar(m21, cax=caxs[2,1], orientation="vertical", extendrect=True)
cb.set_label(r"$(\Delta SST)_{3D-1D}$ (°C)", fontsize=12, labelpad=4)
cb = fig.colorbar(m22, cax=caxs[2,2], orientation="vertical", extendrect=True)
cb.set_label(r"$(\Delta\Delta SST)_{3D-1D}$ (°C)", fontsize=12, labelpad=4)

# axis cosmetics (保持你原来的)
for i in range(3):
    for j in range(3):
        ax = axs[i,j]
        ax.set_xlim(*XLIM_COMMON)
        ax.set_ylim(*YLIM_COMMON)
        ax.set_aspect("equal", adjustable="box")
        ax.set_anchor("E") 
        ax.tick_params(labelsize=10, length=3)
        ax.set_xlabel(r"$(x-x_c)/RMW$", fontsize=12, labelpad=4)
        if j == 0:
            ax.set_ylabel(r"$(y-y_c)/RMW$", fontsize=12, labelpad=6)
        else:
            ax.set_yticklabels([])

# colorbar styling (更紧)
for i in range(3):
    for j in range(3):
        cbax = caxs[i,j]
        cbax.yaxis.set_ticks_position("right")
        cbax.yaxis.set_label_position("right")
        cbax.tick_params(length=2, width=0.8, labelsize=10, pad=1)
        cbax.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter("%.1f"))

# ===== 行列标号：行(a)(b)(c)，列(i)(ii)(iii) =====
row_tags = ["(d)", "(e)", "(f)"]
col_tags = ["(i)", "(ii)", "(iii)"]

# 行标：每行左边第一个 panel 标一次
for i in range(3):
    axs[i,0].text(-0.22, 1.03, row_tags[i],
                  transform=axs[i,0].transAxes,
                  ha="left", va="bottom",
                  fontsize=14)

# 列标：第一行每列标一次
for j in range(3):
    axs[0,j].text(0.02, 1.03, col_tags[j],
                  transform=axs[0,j].transAxes,
                  ha="left", va="bottom",
                  fontsize=14)

# 收紧整个画布边距（关键）
fig.subplots_adjust(left=0.06, right=0.985, bottom=0.06, top=0.985)

plt.show()
# fig.savefig("Fig2A_2mps_rulegrid_360stations_tight_labeled.png", dpi=300, bbox_inches="tight")


In [ ]:
##this code is for figure 13 par(a): (Vertical temperature anomaly profiles relative to initial temperature $\theta(t,z)-\theta_{0,z}$ along the sampling transect indicated Fig. \ref{fig:3DVS1D} first row, located at fixed $y/RMW= \pm1.05$ and $x/RMW=$-1.97,0.03,2.03,4.03,6.03 on the right side of the storm. The first two rows show the 5$\rm ms^{-1}$ simulations and the last two rows shows 2$\rm ms^{-1}$ cases. For each translation speed: the first row shows right side transect points at $y/RMW=+1.05$ and the second row shows left ones $y/RMW=-1.05$. The blue line is the Hybrid ST result, orange dashed line is the Hybrid ST+LT (res) results and the green line indicates LT-induced difference (Hybrid ST+LT - Hybrid ST).


import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ============================================================
# Fig2B (Plan C) profiles for 5 m/s:
# ΔT_noLT(z), ΔT_LT(z), ΔΔT(z) at multiple x/RMW,
# for two y-slices (Right/Left) in storm-relative coords.
# ============================================================

# -------------------------
# 0) USER SETTINGS
# -------------------------
RMW = 50e3

# mapping used in your idealized grid (same as before)
Lx_m, Ly_m = 3.0e6, 1.8e6
lon_min, lon_max = -13.5, 13.5
lat_min, lat_max = -8.1,  8.1

# storm center at day-3
t_final_sec = 3.0 * 86400.0

# 5 m/s storm params (from you)
V = 5.0
X0 = 3.432e6
Y0 = 9.0e5
transdir = np.deg2rad(180.0)
t0_sec = 0.0

# data paths (5 m/s)
PROG_noLT = "/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp/3D_hurricane/3D_kappa_ePBL_5km/5_ustar0p01_results/20110401.prog.nc"
PROG_LT   = "/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp/3D_hurricane/3D_kappa_ePBL_waveLT_5km/5_all_ustar0p01_results/20110401.prog.nc"

# variable names
TEMP_NAME = "temp"   # <-- if your prog uses different name, change here

# time indices
TI0    = 0
TI_END = -1

# which x/RMW profiles to plot (keep 6.03, drop 8.03/10.03)
x_list = [-1.97, 0.03, 2.03, 4.03, 6.03]

# two y-slices (storm-relative)
y_right =  1.05
y_left  = -0.95

# depth range (positive-down)
DEPTH_MAX = 200.0

# x-axis range + ticks (denser on positive side)
X_LIM   = (-5.0, 3.0)
X_TICKS = [-5, -4, -3, -2, -1, 0.0, 1.5,3.0]

# plotting
FIGSIZE = (18.0, 6.8)
LW = 2.2

# -------------------------
# 1) HELPERS
# -------------------------
def lonlat_to_meters_1d(lon_1d, lat_1d):
    x_m = (lon_1d - lon_min) * (Lx_m / (lon_max - lon_min))
    y_m = (lat_1d - lat_min) * (Ly_m / (lat_max - lat_min))
    return x_m, y_m

def storm_center_xy(tsec):
    xc = X0 + (tsec - t0_sec) * V * np.cos(transdir)
    yc = Y0 + (tsec - t0_sec) * V * np.sin(transdir)
    return float(xc), float(yc)

def nearest_idx(arr, val):
    arr = np.asarray(arr)
    return int(np.nanargmin(np.abs(arr - val)))

def infer_depth(ds):
    """Return (depth_pos, zname). depth_pos is positive-down in meters."""
    for zname in ["zl", "z_l", "z", "depth", "st_ocean"]:
        if zname in ds.coords or zname in ds.dims:
            z = ds[zname].values
            z = np.asarray(z).squeeze()
            # make positive-down
            if np.nanmax(z) <= 0.0:   # typical negative-down
                depth = -z
            else:
                depth = z
            return depth, zname
    raise KeyError("Cannot find vertical coordinate (tried zl, z_l, z, depth, st_ocean).")

def get_profile(ds, var, ti, yidx, xidx, zdim):
    """Return 1D profile(var) at (ti, yidx, xidx) over vertical dimension zdim."""
    da = ds[var]
    sel = {}
    if "Time" in da.dims: sel["Time"] = ti
    if "yh"   in da.dims: sel["yh"]   = yidx
    if "xh"   in da.dims: sel["xh"]   = xidx
    # keep only vertical dim
    prof = da.isel(**sel).transpose(zdim).values
    return np.asarray(prof).squeeze()

# -------------------------
# 2) LOAD + COMPUTE PROFILES
# -------------------------
xc, yc = storm_center_xy(t_final_sec)

with xr.open_dataset(PROG_noLT, decode_timedelta=False) as ds0, \
     xr.open_dataset(PROG_LT,   decode_timedelta=False) as ds1:

    # coords (degrees) -> meters
    xh = ds0["xh"].values
    yh = ds0["yh"].values
    x_m_1d, y_m_1d = lonlat_to_meters_1d(xh, yh)

    # storm-relative 1D coords (nice: independent)
    xN_1d = (x_m_1d - xc) / RMW
    yN_1d = (y_m_1d - yc) / RMW

    # pick y indices
    iyR = nearest_idx(yN_1d, y_right)
    iyL = nearest_idx(yN_1d, y_left)

    # depth coordinate
    depth_pos, zname = infer_depth(ds0)

    # depth mask (0..DEPTH_MAX)
    mdep = (depth_pos >= 0.0) & (depth_pos <= DEPTH_MAX)
    depth_plot = depth_pos[mdep]

    # choose x indices
    ix_list = [nearest_idx(xN_1d, xx) for xx in x_list]
    x_pick  = [xN_1d[ix] for ix in ix_list]  # actual achieved x/RMW

    def make_row_profiles(yidx):
        dT0 = []
        dT1 = []
        ddT = []
        for ix in ix_list:
            T0_0 = get_profile(ds0, TEMP_NAME, TI0,    yidx, ix, zname)
            T0_t = get_profile(ds0, TEMP_NAME, TI_END, yidx, ix, zname)
            T1_0 = get_profile(ds1, TEMP_NAME, TI0,    yidx, ix, zname)
            T1_t = get_profile(ds1, TEMP_NAME, TI_END, yidx, ix, zname)

            dT_noLT = (T0_t - T0_0)[mdep]
            dT_LT   = (T1_t - T1_0)[mdep]
            dT_diff = dT_LT - dT_noLT

            dT0.append(dT_noLT)
            dT1.append(dT_LT)
            ddT.append(dT_diff)
        return dT0, dT1, ddT

    dT0_R, dT1_R, ddT_R = make_row_profiles(iyR)
    dT0_L, dT1_L, ddT_L = make_row_profiles(iyL)

# -------------------------
# 3) PLOT (2 rows × len(x_list) cols)
# -------------------------
ncol = len(x_list)
fig, axs = plt.subplots(
    2, ncol, figsize=FIGSIZE, sharey=True,
    gridspec_kw=dict(wspace=0.28, hspace=0.28)
)

# ensure axs is 2D even if ncol=1
axs = np.atleast_2d(axs)

def format_ax(ax):
    ax.set_xlim(*X_LIM)
    ax.set_xticks(X_TICKS)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f"))
    ax.grid(True, alpha=0.22)
    ax.tick_params(labelsize=10, length=3)

    # depth positive-down, surface on top
    ax.set_ylim(DEPTH_MAX, 0.0)

for j in range(ncol):
    # ----- Right row -----
    ax = axs[0, j]
    ax.plot(dT0_R[j], depth_plot, lw=LW, label=r"$\Delta T_{\mathrm{noLT}}$")
    ax.plot(dT1_R[j], depth_plot, lw=LW, ls="--", label=r"$\Delta T_{\mathrm{LT}}$")
    ax.plot(ddT_R[j], depth_plot, lw=LW, label=r"$\Delta\Delta T$")
    ax.set_title(f"x/RMW={x_pick[j]:.2f}", fontsize=12)
    format_ax(ax)
    if j == 0:
        ax.set_ylabel("depth (m)", fontsize=12)
        ax.text(-0.40, 0.50, f"Right\n(y/RMW≈{yN_1d[iyR]:.2f})",
                transform=ax.transAxes, rotation=90,
                va="center", ha="center", fontsize=12)

        # legend only once
        ax.legend(loc="lower left", fontsize=10, frameon=False)

    ax.set_xlabel(r"$\Delta T\ /\ \Delta\Delta T\ (^\circ C)$", fontsize=12)

    # ----- Left row -----
    ax = axs[1, j]
    ax.plot(dT0_L[j], depth_plot, lw=LW)
    ax.plot(dT1_L[j], depth_plot, lw=LW, ls="--")
    ax.plot(ddT_L[j], depth_plot, lw=LW)
    ax.set_title(f"x/RMW={x_pick[j]:.2f}", fontsize=12)
    format_ax(ax)
    if j == 0:
        ax.set_ylabel("depth (m)", fontsize=12)
        ax.text(-0.40, 0.50, f"Left\n(y/RMW≈{yN_1d[iyL]:.2f})",
                transform=ax.transAxes, rotation=90,
                va="center", ha="center", fontsize=12)

    ax.set_xlabel(r"$\Delta T\ /\ \Delta\Delta T\ (^\circ C)$", fontsize=12)

fig.suptitle("Plan-C profiles: ΔT(noLT), ΔT(LT), ΔΔT at multiple x/RMW (5 m/s)", fontsize=13, y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# fig.savefig("Fig2B_profiles_planC_5mps.png", dpi=300, bbox_inches="tight")




In [ ]:
##this code is for figure 13 part(b): (Vertical temperature anomaly profiles relative to initial temperature $\theta(t,z)-\theta_{0,z}$ along the sampling transect indicated Fig. \ref{fig:3DVS1D} first row, located at fixed $y/RMW= \pm1.05$ and $x/RMW=$-1.97,0.03,2.03,4.03,6.03 on the right side of the storm. The first two rows show the 5$\rm ms^{-1}$ simulations and the last two rows shows 2$\rm ms^{-1}$ cases. For each translation speed: the first row shows right side transect points at $y/RMW=+1.05$ and the second row shows left ones $y/RMW=-1.05$. The blue line is the Hybrid ST result, orange dashed line is the Hybrid ST+LT (res) results and the green line indicates LT-induced difference (Hybrid ST+LT - Hybrid ST).
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# ============================================================
# 2 m/s only — Method-B (x-only interpolation)
# Plan-C profiles: ΔT(noLT), ΔT(LT), ΔΔT at exact x/RMW
#   - x: linear interpolation to hit EXACT x/RMW targets
#   - y: nearest grid row to target y/RMW
# depth plotted: 0–200 m (positive downward)
# ============================================================

# -------------------------
# USER SETTINGS
# -------------------------
RMW = 50e3
Lx_m, Ly_m = 3.0e6, 1.8e6
lon_min, lon_max = -13.5, 13.5
lat_min, lat_max = -8.1,  8.1

t_final_sec = 3.0 * 86400.0
TI0, TIEND = 0, -1

VAR  = "temp"      # <--- 改成你 prog.nc 里的变量名（比如 temp / thetao / T 等）
ZMAX = 200.0

# Plan-C targets (exact columns)
x_targets = [-1.97, 0.03, 2.03, 3.03]

# y targets (nearest)
y_targets = {"Right": +1.05, "Left": -0.95}

# x-axis range/ticks (more positive ticks)
XLIM   = (-7.0, 3.0)
XTICKS = [-6,-4,-2,0,1,2,3]

OUTDIR = "./fig2B_planC_2mps_methodB"
os.makedirs(OUTDIR, exist_ok=True)

# -------------------------
# 2 m/s CASE (ONLY)
# -------------------------
case = dict(
    label="2 m/s", V=2.0,
    X0=3.1728e6, Y0=9.0e5,
    transdir=np.deg2rad(180.0), t0_sec=0.0,
    prog_noLT="/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp/3D_hurricane/3D_kappa_ePBL_5km_2mps/5_results/20110401.prog.nc",
    prog_LT  ="/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp/3D_hurricane/3D_kappa_ePBL_waveLT_5km_2mps/5_all_results/20110401.prog.nc",
)

# -------------------------
# HELPERS
# -------------------------
def lonlat_to_meters_1d(lon_1d, lat_1d):
    x_m = (lon_1d - lon_min) * (Lx_m / (lon_max - lon_min))
    y_m = (lat_1d - lat_min) * (Ly_m / (lat_max - lat_min))
    return x_m, y_m

def storm_center_xy(tsec, X0, Y0, V, transdir, t0_sec):
    xc = X0 + (tsec - t0_sec) * V * np.cos(transdir)
    yc = Y0 + (tsec - t0_sec) * V * np.sin(transdir)
    return float(xc), float(yc)

def get_z_coord(ds):
    for zn in ["zl", "z_l", "z", "depth"]:
        if zn in ds.coords or zn in ds.variables:
            return zn
    raise ValueError("Cannot find vertical coordinate (zl/z_l/z/depth) in dataset.")

def nearest_index(arr, val):
    arr = np.asarray(arr)
    return int(np.argmin(np.abs(arr - val)))

def interp_profile_x(field_zx, Xn_1d, x_target):
    """
    Linear interpolate (nz,nx) -> (nz,) at exact x_target along Xn_1d.
    """
    Xn_1d = np.asarray(Xn_1d)

    if (x_target < Xn_1d[0]) or (x_target > Xn_1d[-1]):
        return np.full(field_zx.shape[0], np.nan, dtype=np.float64)

    j1 = int(np.searchsorted(Xn_1d, x_target, side="left"))
    if j1 <= 0:
        return field_zx[:, 0].astype(np.float64)
    if j1 >= len(Xn_1d):
        return field_zx[:, -1].astype(np.float64)

    j0 = j1 - 1
    x0, x1 = Xn_1d[j0], Xn_1d[j1]
    w = (x_target - x0) / (x1 - x0 + 1e-30)
    return ((1.0 - w) * field_zx[:, j0] + w * field_zx[:, j1]).astype(np.float64)

# -------------------------
# LOAD + COMPUTE (2 m/s)
# -------------------------
ds0 = xr.open_dataset(case["prog_noLT"], decode_timedelta=False)
ds1 = xr.open_dataset(case["prog_LT"],   decode_timedelta=False)

xh = ds0["xh"].values
yh = ds0["yh"].values
zname = get_z_coord(ds0)
z = ds0[zname].values

# make z positive-down
z_pos = z.copy()
if np.nanmedian(z_pos) < 0:
    z_pos = -z_pos

# top ZMAX
iz = np.where(z_pos <= ZMAX)[0]
iz = np.sort(iz)

# storm center (day-3)
xc, yc = storm_center_xy(t_final_sec, case["X0"], case["Y0"], case["V"], case["transdir"], case["t0_sec"])

# storm-relative 1D axes
x_m_1d, y_m_1d = lonlat_to_meters_1d(xh, yh)
Xn_1d = (x_m_1d - xc) / RMW
Yn_1d = (y_m_1d - yc) / RMW

# ΔT fields (Time,z,y,x) -> numpy (nz,Ny,Nx)
T0_noLT = ds0[VAR].isel(Time=TI0,   **{zname: iz}).values
Tt_noLT = ds0[VAR].isel(Time=TIEND, **{zname: iz}).values
T0_LT   = ds1[VAR].isel(Time=TI0,   **{zname: iz}).values
Tt_LT   = ds1[VAR].isel(Time=TIEND, **{zname: iz}).values

dT_noLT_zyx = (Tt_noLT - T0_noLT).astype(np.float64)
dT_LT_zyx   = (Tt_LT   - T0_LT  ).astype(np.float64)
ddT_zyx     = (dT_LT_zyx - dT_noLT_zyx).astype(np.float64)

# pick y rows (nearest), but x will be interpolated exactly
profiles = {}
print("\n=== 2 m/s Method-B (x-interp) picked y rows ===")
for side, ytar in y_targets.items():
    iy = nearest_index(Yn_1d, ytar)
    y_picked = Yn_1d[iy]
    print(f"{side}: target y/RMW={ytar:+.2f}, picked y/RMW={y_picked:+.3f} (iy={iy})")

    dT_noLT_zx = dT_noLT_zyx[:, iy, :]  # (nz,nx)
    dT_LT_zx   = dT_LT_zyx[:,   iy, :]
    ddT_zx     = ddT_zyx[:,     iy, :]

    P0, P1, P2 = [], [], []
    for xt in x_targets:
        P0.append(interp_profile_x(dT_noLT_zx, Xn_1d, xt))
        P1.append(interp_profile_x(dT_LT_zx,   Xn_1d, xt))
        P2.append(interp_profile_x(ddT_zx,     Xn_1d, xt))

    profiles[side] = dict(
        y_picked=y_picked,
        dT_noLT=np.stack(P0, axis=0),  # (ncol,nz)
        dT_LT  =np.stack(P1, axis=0),
        ddT    =np.stack(P2, axis=0),
    )

ds0.close(); ds1.close()

z_plot = z_pos[iz]   # positive downward

# -------------------------
# PLOT (2 rows x Ncols)
# -------------------------
ncol = len(x_targets)
fig, axs = plt.subplots(2, ncol, figsize=(3.2*ncol, 6.0), sharey=True)
fig.suptitle("Plan-C profiles: ΔT(noLT), ΔT(LT), ΔΔT at exact x/RMW (2 m/s)", y=0.98, fontsize=14)

row_order = ["Right", "Left"]
for ir, side in enumerate(row_order):
    P = profiles[side]
    for ic, xt in enumerate(x_targets):
        ax = axs[ir, ic]

        ax.plot(P["dT_noLT"][ic, :], z_plot, lw=2.2, label=r"$\Delta T_{\mathrm{noLT}}$")
        ax.plot(P["dT_LT"][ic, :],   z_plot, lw=2.2, ls="--", label=r"$\Delta T_{\mathrm{LT}}$")
        ax.plot(P["ddT"][ic, :],     z_plot, lw=2.2, label=r"$\Delta\Delta T$")

        ax.axvline(0.0, color="0.7", lw=1.0)
        ax.grid(True, alpha=0.25)

        ax.set_xlim(*XLIM)
        ax.set_xticks(XTICKS)

        # depth axis: 0 at top, 200 at bottom
        ax.set_ylim(ZMAX, 0)

        ax.set_title(f"x/RMW={xt:+.2f}", fontsize=12)
        ax.set_xlabel(r"$\Delta T\ /\ \Delta\Delta T\ (\degree C)$", fontsize=12)

        if ic == 0:
            ax.set_ylabel(f"depth (m)\n{side} (y/RMW≈{P['y_picked']:+.2f})", fontsize=12)

        if (ir == 0) and (ic == 0):
            ax.legend(loc="lower left", fontsize=10, frameon=False)

plt.subplots_adjust(left=0.06, right=0.99, bottom=0.12, top=0.90, wspace=0.35, hspace=0.50)

outpng = os.path.join(OUTDIR, "Fig2B_PlanC_2mps_methodB.png")
fig.savefig(outpng, dpi=300)
plt.show()
print(f"[OK] saved: {outpng}")


In [ ]:
#this code is for figure 14:SST response to Hurricane Ivan (2004) in the Northwest Atlantic. Panel (a) Hybrid ST; Panel(b) Hybrid ST+LT ; Panel (c) Satellite SST (GOES SST) reproduced from \citet{zambon2014investigation} where the SST data is obtained from the JPL Physical Oceanography Distributed Active Archive Center (PODAAC); Panel (d) LT-induced SST cooling increment obtained by (b) - (a).  The first columns at each row represent modeled pre-storm SST defined as the mean over 9 September 12:00 UTC to 12 September 12:00 UTC; The second columns at each row are modeled post-storm SST averaged over 17 September 00:00 UTC to 18 September 00:00 UTC. The same averaging windows and color scale are used in satellite panel (c) to facilitate comparison. The track overlaid on panels (a), (b) and (d) (red) and on panel (c) (black) track is from the best-track data, NHC, NOAA. The text label besides the line indicates the dates when Ivan hits the location. The red lines in model panel indicate the location where TC hits at 12:00 pm UTC while the black lines in the satellite panel use 00:00 pm UTC.


#!/usr/bin/env python3
# ============================================================
# Regional SST: Pre-storm mean, Post-storm mean, and ΔSST (Post–Pre)
# 4 rows x 3 cols, each panel has its OWN colorbar
#
# Row (a): Hybrid_ST            (was: kappa noLT)
# Row (b): Hybrid_ST+LT (res)   (was: kappa wave_all)
# Row (c): Satellite PNG (IvanSatellite.png)  (spans 3 columns, NOT split)
# Row (d): Hybrid_ST+LT (res) - Hybrid_ST     (pairwise diff)
#
# Column tags (i)(ii)(iii) shown ONLY on first row.
# Row tags (a)(b)(c)(d) shown on first column / satellite row.
#
# Track overlay:
#   - plot storm track line
#   - markers + date labels at EVERY DAY 00:00 (midnight), not 12:00
#   - time range same as your original (end_exclusive unchanged)
#
# Satellite row:
#   - load IvanSatellite.png
#   - auto-crop near-white margins so it visually matches other rows
#   - imshow(aspect="auto") to fill the axis (bigger / consistent)
#   - add 3 titles above the satellite (for its 3 internal columns)
#
# ============================================================

import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import ListedColormap
import matplotlib.image as mpimg

# ============================================================
# 0) PATHS (your original)
# ============================================================
pp_root_kappa_noLT = "/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp/NWA12_Ivan_kappa/"
pp_root_kappa_LT   = "/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp/NWA12_LT_Ivan_kappa/"
pp_root_shear      = "/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp/NWA12_shear_Ivan/"
pp_root_ePBLShear  = "/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp/NWA12_ePBLShear_Ivan/"
pp_root_kappa_wave = "/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_GOTM_kapp_C6/NWA12_wave_Ivan/"

# models used in this figure
sst_kappa_noLT      = xr.open_dataset(pp_root_kappa_noLT + "/20040907_results/20040907.ocean_hourly.nc")["SST_hm"]
sst_kappa_wave_all  = xr.open_dataset(pp_root_kappa_wave + "/20040907_results_all/20040907.ocean_hourly.nc")["SST_hm"]

# ============================================================
# 1) TRACK overlay (UPDATED: label at 00:00 daily)
# ============================================================
MODEL_TRACK_CSV = "/archive/Qian.Xiao/Qian.Xiao/FMS_Wave_Coupling_ZC/c6/postProcessing/buoydata/2004/ivan_Model_track_hourly.csv"
TRACK_END_EXCLUSIVE = "2004-09-17T00:00"     # remove 09/17-09/18

TRACK_COLOR = "red"
TRACK_LW = 1.8
TRACK_MARKER = "o"
TRACK_MARKER_SIZE = 5

# change here: label at midnight (00:00)
LABEL_AT_HOUR = 0
LABEL_TEXT_FMT = "%m/%d"
LABEL_TEXT_COLOR = "k"
LABEL_DX = -1.0
LABEL_DY = 0.15

LAND_COLOR = "lightgray"

def lon360_to_180(lon):
    lon = np.asarray(lon)
    return ((lon + 180.0) % 360.0) - 180.0

def prep_model_track(csv_path, end_exclusive):
    df = pd.read_csv(csv_path, parse_dates=["time"]).copy()
    df["time"] = pd.to_datetime(df["time"])
    if df["lon_smooth"].max() > 180:
        df["lon_smooth"] = lon360_to_180(df["lon_smooth"].values)
    df = df.sort_values("time").reset_index(drop=True)
    df = df[df["time"] < pd.to_datetime(end_exclusive)].copy().reset_index(drop=True)
    return df

def pick_track_labels(df, label_hour=0, fmt="%m/%d"):
    out = df.copy()
    out = out[(out["time"].dt.hour == label_hour) & (out["time"].dt.minute == 0)]
    out["day"] = out["time"].dt.floor("D")
    out = out.drop_duplicates(subset=["day"])
    out["label"] = out["time"].dt.strftime(fmt)
    return out

def overlay_track(ax, track_df, label_df):
    ax.plot(track_df["lon_smooth"], track_df["lat_smooth"],
            color=TRACK_COLOR, lw=TRACK_LW, zorder=6)
    ax.plot(label_df["lon_smooth"], label_df["lat_smooth"],
            linestyle="none", marker=TRACK_MARKER, markersize=TRACK_MARKER_SIZE,
            markerfacecolor=TRACK_COLOR, markeredgecolor=TRACK_COLOR, zorder=7)
    for _, r in label_df.iterrows():
        ax.text(r["lon_smooth"] + LABEL_DX, r["lat_smooth"] + LABEL_DY, r["label"],
                color=LABEL_TEXT_COLOR, fontsize=9, zorder=8, ha="right", va="bottom")

track_df = prep_model_track(MODEL_TRACK_CSV, TRACK_END_EXCLUSIVE)
label_df = pick_track_labels(track_df, LABEL_AT_HOUR, LABEL_TEXT_FMT)

# ============================================================
# 2) LAND MASK helper (same idea as you had)
# ============================================================
def plot_land_mask_from_ref(ax, ref_2d_da, lon_name="xh", lat_name="yh", land_color="lightgray"):
    lon = ref_2d_da[lon_name].values
    lat = ref_2d_da[lat_name].values
    land_mask = np.isnan(ref_2d_da.values).astype(int)  # 1=land, 0=ocean
    Lon2d, Lat2d = np.meshgrid(lon, lat)
    cmap_land = ListedColormap(["none", land_color])
    ax.pcolormesh(
        Lon2d, Lat2d, land_mask,
        cmap=cmap_land, vmin=0, vmax=1,
        shading="auto", zorder=0
    )

# ============================================================
# 3) Pre/Post extraction (your original logic)
# ============================================================
def extract_sst_pre_post(
    model_sst_dict: dict,
    pre_period: tuple[str, str],
    post_period: tuple[str, str],
    lon_bounds: tuple[float, float],
    lat_bounds: tuple[float, float]
) -> dict:
    out = {}
    pre_start, pre_end   = pre_period
    post_start, post_end = post_period

    for name, da in model_sst_dict.items():
        sub = da.sel(
            xh=slice(lon_bounds[0], lon_bounds[1]),
            yh=slice(lat_bounds[0], lat_bounds[1])
        )
        pre_mean  = sub.sel(time=slice(pre_start,  pre_end)).mean(dim="time", skipna=True)
        post_mean = sub.sel(time=slice(post_start, post_end)).mean(dim="time", skipna=True)
        diff      = post_mean - pre_mean
        out[name] = {"pre": pre_mean, "post": post_mean, "diff": diff}
    return out

def compute_pairwise_sst_diffs(sst_prepost: dict, pairs: dict) -> dict:
    out = {}
    for label, (m1, m2) in pairs.items():
        pre_diff  = sst_prepost[m1]["pre"]  - sst_prepost[m2]["pre"]
        post_diff = sst_prepost[m1]["post"] - sst_prepost[m2]["post"]
        d_diff    = sst_prepost[m1]["diff"] - sst_prepost[m2]["diff"]
        out[label] = {"pre": pre_diff, "post": post_diff, "diff": d_diff}
    return out

# ============================================================
# 4) Small utilities: colorbar per axis + crop satellite margins
# ============================================================
def add_cbar(ax, mappable, label="°C", size="3.5%", pad=0.05):
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size=size, pad=pad)
    cb = plt.colorbar(mappable, cax=cax)
    cb.set_label(label)
    return cb

def autocrop_white_margin(img, thr=0.985, pad=2):
    """
    Crop near-white margins around the PNG so it visually matches other rows.
    thr close to 1 => treat very bright pixels as white.
    """
    if img.ndim == 2:
        arr = img
        mask = arr < thr
    else:
        arr = img[..., :3].astype(float)
        if arr.max() > 1.0:
            arr = arr / 255.0
        gray = arr.mean(axis=2)
        mask = gray < thr

    if not np.any(mask):
        return img

    ys, xs = np.where(mask)
    y0, y1 = ys.min(), ys.max()
    x0, x1 = xs.min(), xs.max()
    y0 = max(0, y0 - pad); y1 = min(img.shape[0]-1, y1 + pad)
    x0 = max(0, x0 - pad); x1 = min(img.shape[1]-1, x1 + pad)

    if img.ndim == 3:
        return img[y0:y1+1, x0:x1+1, :]
    return img[y0:y1+1, x0:x1+1]

# ============================================================
# 5) ONE combined plot: 4 rows x 3 cols, satellite spans 3 cols
# ============================================================
def plot_combined_with_satellite(
    sst_prepost: dict,
    diffs: dict,
    sat_png_path: str,
    lon_name="xh",
    lat_name="yh",
):
    # ---- rename labels exactly as you asked
    rowA_name = "Hybrid_ST"                 # was: kappa noLT
    rowB_name = "Hybrid_ST+LT (res)"        # was: kappa wave_all
    rowD_name = "Hybrid_ST+LT (res) - Hybrid_ST"

    col_titles = ["Pre-storm SST", "Post-storm SST", "ΔSST (Post–Pre)"]
    row_tags = ["(a)", "(b)", "(c)", "(d)"]
    col_tags = ["(i)", "(ii)", "(iii)"]

    # color settings (your original)
    cmap_sst  = "jet"
    cmap_diff = "bwr"
    vmin_sst, vmax_sst = 26, 31
    vmin_dd,  vmax_dd  = -3, 3
    vmin_pair, vmax_pair = -0.4, 0.4

    # wider aspect like you wanted
    fig = plt.figure(figsize=(17.5, 23.0))
    gs = GridSpec(
        nrows=4, ncols=3, figure=fig,
        height_ratios=[1.0, 1.0, 1.5, 1.0],  # satellite row taller
        wspace=0.18, hspace=0.45
    )

    axs = np.empty((4,3), dtype=object)
    for r in range(4):
        for c in range(3):
            axs[r,c] = fig.add_subplot(gs[r,c])

    def draw_data_panel(ax, da, title, vmin, vmax, cmap, add_track=True):
        plot_land_mask_from_ref(ax, da, lon_name=lon_name, lat_name=lat_name, land_color=LAND_COLOR)
        lon = da[lon_name]
        lat = da[lat_name]
        im = ax.pcolormesh(lon, lat, da, vmin=vmin, vmax=vmax, cmap=cmap, shading="auto")
        if add_track:
            overlay_track(ax, track_df, label_df)
        ax.set_title(title, fontsize=12)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        add_cbar(ax, im, label="°C", size="3.5%", pad=0.05)

    # ---- Row (a): Hybrid_ST  (kappa noLT)
    fieldsA = sst_prepost["kappa noLT"]
    draw_data_panel(axs[0,0], fieldsA["pre"],  f"{rowA_name}\n{col_titles[0]}", vmin_sst, vmax_sst, cmap_sst)
    draw_data_panel(axs[0,1], fieldsA["post"], f"{rowA_name}\n{col_titles[1]}", vmin_sst, vmax_sst, cmap_sst)
    draw_data_panel(axs[0,2], fieldsA["diff"], f"{rowA_name}\n{col_titles[2]}", vmin_dd,  vmax_dd,  cmap_diff)

    # ---- Row (b): Hybrid_ST+LT (res)  (kappa wave_all)
    fieldsB = sst_prepost["kappa wave_all"]
    draw_data_panel(axs[1,0], fieldsB["pre"],  f"{rowB_name}\n{col_titles[0]}", vmin_sst, vmax_sst, cmap_sst)
    draw_data_panel(axs[1,1], fieldsB["post"], f"{rowB_name}\n{col_titles[1]}", vmin_sst, vmax_sst, cmap_sst)
    draw_data_panel(axs[1,2], fieldsB["diff"], f"{rowB_name}\n{col_titles[2]}", vmin_dd,  vmax_dd,  cmap_diff)

    # ---- Row (c): Satellite PNG spans 3 columns (no split, no copy)
    # Turn off the three axes and create a spanning axis using their positions
    for c in range(3):
        axs[2,c].axis("off")

    p0 = axs[2,0].get_position()
    p2 = axs[2,2].get_position()
    ax_sat = fig.add_axes([p0.x0, p0.y0, p2.x1 - p0.x0, p0.height])

    img = mpimg.imread(sat_png_path)
    img = autocrop_white_margin(img, thr=0.985, pad=2)  # enlarge visually by removing margins
    ax_sat.imshow(img, aspect="auto")                   # fill the axis
    ax_sat.axis("off")

    # row tag (c)
    ax_sat.text(-0.02, 1.02, row_tags[2], transform=ax_sat.transAxes,
                ha="left", va="bottom", fontsize=14)

    # satellite titles above its 3 internal columns
    sat_titles = [
        "Satellite Pre-storm SST",
        "Satellite Post-storm SST",
        "Satellite Δ SST (Post–Pre)",
    ]
    for x, tt in zip([1/6, 3/6, 5/6], sat_titles):
        ax_sat.text(x, 1.02, tt, transform=ax_sat.transAxes,
                    ha="center", va="bottom", fontsize=12)

    # ---- Row (d): pairwise diff
    fieldsD = diffs["kappa wave_all - kappa noLT"]
    draw_data_panel(axs[3,0], fieldsD["pre"],  f"{rowD_name}\n{col_titles[0]}", vmin_pair, vmax_pair, cmap_diff)
    draw_data_panel(axs[3,1], fieldsD["post"], f"{rowD_name}\n{col_titles[1]}", vmin_pair, vmax_pair, cmap_diff)
    draw_data_panel(axs[3,2], fieldsD["diff"], f"{rowD_name}\n{col_titles[2]}", vmin_pair, vmax_pair, cmap_diff)

    # ---- row tags (a)(b)(d) on first column axes
    axs[0,0].text(-0.18, 1.02, row_tags[0], transform=axs[0,0].transAxes,
                  ha="left", va="bottom", fontsize=14)
    axs[1,0].text(-0.18, 1.02, row_tags[1], transform=axs[1,0].transAxes,
                  ha="left", va="bottom", fontsize=14)
    axs[3,0].text(-0.18, 1.02, row_tags[3], transform=axs[3,0].transAxes,
                  ha="left", va="bottom", fontsize=14)

    # ---- col tags only on first row
    for c in range(3):
        axs[0,c].text(0.02, 1.02, col_tags[c], transform=axs[0,c].transAxes,
                      ha="left", va="bottom", fontsize=14)

    plt.show()

# ============================================================
# 6) RUN
# ============================================================

model_sst_prepost = {
    "kappa noLT":      sst_kappa_noLT,
    "kappa wave_all":  sst_kappa_wave_all,
}

sst_prepost = extract_sst_pre_post(
    model_sst_prepost,
    pre_period = ("2004-09-09", "2004-09-12"),
    post_period= ("2004-09-17", "2004-09-18"),
    lon_bounds = (-100, -75),
    lat_bounds = (5, 33)
)

pairs = {
    "kappa wave_all - kappa noLT": ("kappa wave_all", "kappa noLT"),
}
diffs = compute_pairwise_sst_diffs(sst_prepost, pairs)

SAT_PNG = "/home/Qian.Xiao/IvanSatellite.png"  # <<< change to your real path if needed

plot_combined_with_satellite(
    sst_prepost=sst_prepost,
    diffs=diffs,
    sat_png_path=SAT_PNG,
    lon_name="xh",
    lat_name="yh",
)


